# Pandas, Clearly: Solutions Workbook

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 00. A short Python bridge

Read the small amount of Python needed to begin the pandas lessons.

Before you start: No programming experience is required; work through one line at a time.

Official reference: https://docs.python.org/3/tutorial/introduction.html

### M00C01. Names hold values

A variable is a name associated with a value. The equals sign assigns a value; two equals signs compare values. Quotation marks make text, while an unquoted number can take part in arithmetic. print() displays a value so you can inspect it.

**Read:** Which line assigns a value, and which line compares two values?

Answer: age = 20 assigns 20 to age. The expression age == 20 compares age with 20 and produces True.

**Predict:** What will next_age print? Will age itself become 21?

Answer: next_age prints 21. age stays 20 because the calculated value was assigned to a different name.

**Explain:** Why are "20" and 20 different?

Answer: "20" is text; 20 is a whole number. Their types determine which operations make sense.



In [ ]:
# Predict before running: What will next_age print? Will age itself become 21?
name = "Amira"
age = 20
next_age = age + 1
is_twenty = age == 20
print(name)
print(next_age)
print(is_twenty)
print(type(age).__name__)

**Your turn:** Describe Bilal, age 21, using name and age. Calculate his age in three years in next_age. Store the comparison of his current age with 21 in is_twenty_one, then print the results.

For the check, keep these result names: `age`, `is_twenty_one`, `name`, `next_age`.

**Hint:** Change the name and starting age, then use age + 3. Keep = for assignment and == for comparison.

**Watch for this:** Python names are case-sensitive: age and Age are different names.

In [ ]:
# Change this code to complete the task.
# Question: Describe Bilal, age 21, using name and age. Calculate his age in three years in next_age. Store the comparison of his current age with 21 in is_twenty_one, then print the results.
name = "Bilal"
age = 21
next_age = age + 3
is_twenty_one = age == 21
print(name)
print(next_age)
print(is_twenty_one)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert name == 'Bilal' and age == 21
assert next_age == 24 and is_twenty_one is True
print("Code checks passed. Review the written explanation as well.")

The results are Bilal, 24, and True. The original age remains 21.

**Quick check:** What does age == 20 do?

1. Assigns 20 to age
2. Compares age with 20
3. Converts age to text

Answer: 2. The double equals sign compares two values. A single equals sign assigns a value.

**Explain what you learned:** I can distinguish a value, a variable, an assignment, and a comparison.

### M00C02. A list keeps values in order

Square brackets create a list. Each value has a position, beginning at 0. len() counts the values, and a slice selects a range of positions. A slice includes its start position and excludes its stop position.

**Read:** What is the difference between scores and scores[0]?

Answer: scores is the whole list. scores[0] retrieves the first value, 88.

**Predict:** Does scores[0:2] contain two values or three?

Answer: Two values: 88 and 72. Position 2 is excluded from the slice.

**Explain:** Why would scores[3] fail for this list?

Answer: The list has three values at positions 0, 1, and 2. Position 3 does not exist, so Python raises IndexError.



In [ ]:
# Predict before running: Does scores[0:2] contain two values or three?
scores = [88, 72, 91]
first_score = scores[0]
first_two = scores[0:2]
print(first_score)
print(first_two)
print(len(scores))

**Your turn:** Select the second and third scores into result. Then add 65 to the end of scores using append(), and print the new list length.

For the check, keep these result names: `result`, `scores`.

**Hint:** Use the slice [1:3], then scores.append(65).

**Watch for this:** A list position is an integer. The text "0" is not the same as position 0.

In [ ]:
# Change this code to complete the task.
# Question: Select the second and third scores into result. Then add 65 to the end of scores using append(), and print the new list length.
scores = [88, 72, 91]
result = scores[1:3]
scores.append(65)
print(result)
print(scores)
print(len(scores))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == [72, 91]
assert scores == [88, 72, 91, 65]
print("Code checks passed. Review the written explanation as well.")

result contains 72 and 91. The original list now has four values, ending in 65.

**Quick check:** Which position refers to the first list value?

1. 0
2. 1
3. The list length

Answer: 1. Python list positions begin at zero.

**Explain what you learned:** I can count, select, and append list values.

### M00C03. Give a calculation a name

A function is a named set of instructions you can reuse. def starts a function definition, and the indented body states what it does. A parameter receives an input value when the function is called. return sends the result back so it can be saved or used in another calculation.

**Read:** In add_bonus(72, 3), what values do score and bonus receive?

Answer: score receives 72 and bonus receives 3 for that call.

**Predict:** What do first and second contain?

Answer: first contains 75; second contains 90.

**Explain:** Why return the answer instead of only printing it inside the function?

Answer: A returned answer can be saved in a variable and used later. Printing alone displays it but does not return that answer to the caller.



In [ ]:
# Predict before running: What do first and second contain?
def add_bonus(score, bonus):
    return score + bonus

first = add_bonus(72, 3)
second = add_bonus(88, 2)
print(first)
print(second)

**Your turn:** Create a function called to_percentage(points, maximum) that returns points divided by maximum, multiplied by 100. Calculate 16 out of 20 into result and print it.

For the check, keep these result names: `result`.

**Hint:** Keep the return line indented. For this exercise, maximum is positive.

**Watch for this:** Indentation is part of Python syntax. Division also requires a nonzero denominator.

In [ ]:
# Change this code to complete the task.
# Question: Create a function called to_percentage(points, maximum) that returns points divided by maximum, multiplied by 100. Calculate 16 out of 20 into result and print it.
def to_percentage(points, maximum):
    return (points / maximum) * 100

result = to_percentage(16, 20)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == 80.0
assert to_percentage(35, 50) == 70.0
print("Code checks passed. Review the written explanation as well.")

The returned percentage is 80.0. The same function can be called with a different number of points and a different positive maximum.

**Quick check:** Which keyword sends the calculated answer back to the caller?

1. def
2. print
3. return

Answer: 3. return supplies the function's result. def defines a function; print displays information.

**Explain what you learned:** I can read a small function, call it, and use its returned value.

### M00C04. Import a tool and read an error

import makes a module's tools available; as gives the module a shorter name. This example uses Python's built-in math module. The try/except block catches an expected conversion error so we can read its type and message, then continue. Read the error name and the failing value before changing code.

**Read:** What does m refer to after import math as m?

Answer: m is a short name for the math module. m.sqrt accesses its square-root function.

**Predict:** Why does float("nine") raise ValueError, but float("9") works?

Answer: The word nine cannot be parsed as a numeric value by float. The digit string 9 can, producing 9.0.

**Explain:** What should you inspect first when a cell reports an error?

Answer: Read the error type, its message, and the line that failed. Check the actual input and variable names rather than changing unrelated lines.



In [ ]:
# Predict before running: Why does float("nine") raise ValueError, but float("9") works?
import math as m

try:
    number = float("nine")
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))

number = float("9")
result = m.sqrt(number)
print("Corrected result:", result)

**Your turn:** Import math as m, set text to "25", convert text to a number, and calculate its square root in result. Print the result.

For the check, keep these result names: `number`, `result`, `text`.

**Hint:** Use float(text) for conversion and m.sqrt(number) for the calculation.

**Watch for this:** Catching an error does not repair its cause. This example prints the expected error and then supplies a valid value explicitly.

In [ ]:
# Change this code to complete the task.
# Question: Import math as m, set text to "25", convert text to a number, and calculate its square root in result. Print the result.
import math as m

text = "25"
number = float(text)
result = m.sqrt(number)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert text == '25' and number == 25.0 and result == 5.0
print("Code checks passed. Review the written explanation as well.")

The result is 5.0. Later, import pandas as pd follows the same alias pattern; pandas supplies the data tools used in the course.

**Quick check:** What does as pd mean in import pandas as pd?

1. Create a DataFrame called pd
2. Use pd as a short name for the pandas module
3. Convert every value to a number

Answer: 2. The alias pd refers to the imported pandas module, which is why pd.Series and pd.DataFrame work.

**Explain what you learned:** I can recognize an import alias and use an error message to guide a correction.

## Module 01. Start with a dictionary

Read, access, and change named values in a Python dictionary.

Before you start: You only need to recognize text, numbers, and the print function.

Official reference: https://docs.python.org/3/tutorial/datastructures.html#dictionaries

### M01C01. What are the key and the value?

A dictionary stores pairs: a key and its value. The key is a name you use to find a value, such as "Name" or "Age". Curly brackets create the dictionary; a colon separates each key from its value. Here, one dictionary describes one fictional student.

**Read:** In "Age": 20, which part is the key and which part is the value?

Answer: "Age" is the key. The integer 20 is its value.

**Predict:** What will len(student) print: the number of letters, pairs, or students?

Answer: It prints 3, the number of key-value pairs.

**Explain:** Why is "20" different from 20?

Answer: "20" is text; 20 is an integer. Numbers stored as text need conversion before arithmetic.



In [ ]:
# Predict before running: What will len(student) print: the number of letters, pairs, or students?
# Text uses quotes. A whole number does not need quotes.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(student)
print(type(student).__name__)
print(len(student))

**Your turn:** Create a dictionary for Bilal, age 21, at Dubai. Add a fourth key called Program with value IT. Print the dictionary and its number of pairs.

For the check, keep these result names: `student`.

**Hint:** Use four key: value pairs inside one pair of curly brackets.

**Watch for this:** Do not mix up braces { } for dictionaries with brackets [ ] used for lists and access.

In [ ]:
# Change this code to complete the task.
# Question: Create a dictionary for Bilal, age 21, at Dubai. Add a fourth key called Program with value IT. Print the dictionary and its number of pairs.
student = {"Name": "Bilal", "Age": 21, "Campus": "Dubai", "Program": "IT"}
print(student)
print(len(student))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert student['Name'] == 'Bilal' and student['Age'] == 21
assert student['Program'] == 'IT' and len(student) == 4
print("Code checks passed. Review the written explanation as well.")

The result has four pairs. Program is a new key, so the dictionary length increases to 4.

**Quick check:** What does the colon do in "Age": 20?

1. It ends the dictionary.
2. It connects a key to its value.
3. It converts text to a number.

Answer: 2. A colon separates a dictionary key from its associated value.

**Explain what you learned:** I can identify the keys, values, and number of pairs in a dictionary.

### M01C02. Find a value by its key

Square brackets after a dictionary look up a key. The key must match exactly, including upper and lower case. get() is useful when a key might be missing: you can supply a default value instead of raising KeyError.

**Read:** What does student["Name"] ask Python to find?

Answer: It asks for the value associated with the exact key "Name".

**Predict:** What will the three print statements display?

Answer: Amira, then 20, then Not provided, each on a separate line.

**Explain:** Would student["name"] work here? Why?

Answer: No. The key is "Name", with an uppercase N. The different key "name" is absent, so direct access raises KeyError.



In [ ]:
# Predict before running: What will the three print statements display?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(student["Name"])
print(student["Age"])
print(student.get("Email", "Not provided"))

**Your turn:** Store the campus in campus. Use get() to request Program with the default "Undeclared" and save the value in program. Store whether "Age" is a key in has_age. Print all three values.

For the check, keep these result names: `campus`, `has_age`, `program`.

**Hint:** Use student["Campus"], student.get(...), and "Age" in student.

**Watch for this:** A default from get() does not insert the missing key into the dictionary.

In [ ]:
# Change this code to complete the task.
# Question: Store the campus in campus. Use get() to request Program with the default "Undeclared" and save the value in program. Store whether "Age" is a key in has_age. Print all three values.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
campus = student["Campus"]
program = student.get("Program", "Undeclared")
has_age = "Age" in student
print(campus)
print(program)
print(has_age)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert campus == 'RAK' and program == 'Undeclared' and has_age is True
print("Code checks passed. Review the written explanation as well.")

Campus is RAK. Program is missing, so get() returns the chosen default. Membership with in checks keys, so has_age is True.

**Quick check:** Which expression safely returns "Unknown" when Phone is missing?

1. student["Phone", "Unknown"]
2. student["Phone"]
3. student.get("Phone", "Unknown")

Answer: 3. get(key, default) supplies the default only when the requested key is absent.

**Explain what you learned:** I can choose between direct key access and a safe lookup with a default.

### M01C03. Change, add, and remove a pair

Assigning a value to an existing key replaces its old value. Assigning to a new key adds a pair. pop() removes a key and returns its value. A dictionary cannot hold two separate entries with the same key.

**Read:** Which line changes an existing pair, and which line adds a new pair?

Answer: student["Age"] = 21 changes Age. student["Program"] = "AI" adds Program.

**Predict:** How many pairs remain after Campus is removed?

Answer: Three: Name, Age, and Program.

**Explain:** Why does changing Age not increase the dictionary length?

Answer: Age already exists. Assignment replaces its value instead of adding a second Age key.



In [ ]:
# Predict before running: How many pairs remain after Campus is removed?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
student["Age"] = 21
student["Program"] = "AI"
removed = student.pop("Campus")
print(student)
print("Removed:", removed)

**Your turn:** Start with the same original dictionary. Increase Age by 2, add Program as IT, and remove Campus. Print the final dictionary.

For the check, keep these result names: `student`.

**Hint:** Read the existing age on the right side of the assignment: student["Age"] + 2.

**Watch for this:** pop() raises KeyError for an absent key unless you also provide a default.

In [ ]:
# Change this code to complete the task.
# Question: Start with the same original dictionary. Increase Age by 2, add Program as IT, and remove Campus. Print the final dictionary.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
student["Age"] = student["Age"] + 2
student["Program"] = "IT"
student.pop("Campus")
print(student)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert student == {'Name': 'Amira', 'Age': 22, 'Program': 'IT'}
print("Code checks passed. Review the written explanation as well.")

Age is 22 and Program is IT. Campus is no longer a key. Reading the old age lets the same calculation work for other starting ages.

**Quick check:** What happens when you assign another value to an existing key?

1. Its value is replaced.
2. A second identical key is added.
3. The dictionary is sorted.

Answer: 1. Each dictionary key identifies one entry; assigning to it replaces its associated value.

**Explain what you learned:** I can update a dictionary without confusing replacement with insertion.

### M01C04. Read all keys and all pairs

keys() gives the keys, values() gives their values, and items() gives key-value pairs. list() turns these views into lists that are easy to display. A for loop visits each pair in turn; the indented line runs once per pair.

**Read:** What two things are assigned to key and value on each loop turn?

Answer: One dictionary key and the value associated with that key.

**Predict:** How many times does the indented print line run?

Answer: Three times, once for each pair.

**Explain:** What is the difference between "RAK" in student and "RAK" in student.values()?

Answer: The first checks keys and is False. The second checks values and is True.



In [ ]:
# Predict before running: How many times does the indented print line run?
student = {"Name": "Amira", "Age": 20, "Campus": "RAK"}
print(list(student.keys()))
print(list(student.values()))
for key, value in student.items():
    print(key, "=", value)

**Your turn:** Add Program as AI, print only the keys, and build a list named pairs using list(student.items()). Print pairs.

For the check, keep these result names: `pairs`.

**Hint:** items() exposes both parts of each dictionary entry together.

**Watch for this:** A Python dictionary preserves insertion order, but this does not mean the keys are alphabetically sorted.

In [ ]:
# Change this code to complete the task.
# Question: Add Program as AI, print only the keys, and build a list named pairs using list(student.items()). Print pairs.
student = {"Name": "Amira", "Age": 20, "Campus": "RAK", "Program": "AI"}
print(list(student.keys()))
pairs = list(student.items())
print(pairs)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert pairs == [('Name', 'Amira'), ('Age', 20), ('Campus', 'RAK'), ('Program', 'AI')]
print("Code checks passed. Review the written explanation as well.")

There are four keys and four pairs. Each pair is displayed as a tuple, such as ("Program", "AI").

**Quick check:** Which method gives key-value pairs?

1. values()
2. items()
3. keys()

Answer: 2. items() provides each key together with its value.

**Explain what you learned:** I can inspect a dictionary and explain what a key-value loop does.

## Module 02. From one student to many

Represent several records with lists and dictionaries before building a table.

Before you start: Complete the dictionary examples in Module 1.

Official reference: https://docs.python.org/3/tutorial/datastructures.html

### M02C01. A list has positions

A list stores values in order. Positions start at 0, so names[1] is the second value. A slice such as [0:2] includes positions 0 and 1 but excludes position 2. A negative position counts back from the end.

**Read:** What do the square brackets mean when names is first created?

Answer: They create a list containing four text values in order.

**Predict:** What is printed by names[-1] and names[0:2]?

Answer: Dana, then ["Amira", "Bilal"].

**Explain:** Why does [0:2] return two names rather than three?

Answer: The start position is included, but the stop position 2 is excluded.



In [ ]:
# Predict before running: What is printed by names[-1] and names[0:2]?
names = ["Amira", "Bilal", "Chloe", "Dana"]
print(names[0])
print(names[-1])
result = names[0:2]
print(result)

**Your turn:** Select only Bilal and Chloe into result using one slice. Then append Ethan and print the new list length.

For the check, keep these result names: `names`, `result`.

**Hint:** Start the slice at position 1 and stop before position 3.

**Watch for this:** A single position returns one item; a slice returns a list, even if it contains one item.

In [ ]:
# Change this code to complete the task.
# Question: Select only Bilal and Chloe into result using one slice. Then append Ethan and print the new list length.
names = ["Amira", "Bilal", "Chloe", "Dana"]
result = names[1:3]
names.append("Ethan")
print(result)
print(len(names))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == ['Bilal', 'Chloe']
assert names[-1] == 'Ethan' and len(names) == 5
print("Code checks passed. Review the written explanation as well.")

The slice returns Bilal and Chloe. append() adds a value at the end, so the full list now has five names.

**Quick check:** Which expression selects the second value?

1. names[2]
2. names[1]
3. names[0]

Answer: 2. Positions start at 0, so the second value is at position 1.

**Explain what you learned:** I can predict list positions and slices.

### M02C02. A dictionary of lists describes columns

Now each dictionary value is a list. The keys describe future column names, and values at the same position belong to the same student. data["Age"] chooses a list; the second pair of brackets chooses an item inside that list.

**Read:** What is the type of data, and what is the type of data["Age"]?

Answer: data is a dictionary. data["Age"] is a list.

**Predict:** Which student is described by position 1 in all three lists?

Answer: Bilal, age 21, at Dubai.

**Explain:** What can go wrong if you reorder only the names list?

Answer: Names could become paired with another student's age and campus. The lists must stay aligned by position.



In [ ]:
# Predict before running: Which student is described by position 1 in all three lists?
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19],
    "Campus": ["RAK", "Dubai", "RAK"]
}
print(data["Name"])
print(data["Age"][1])
print(data["Name"][1], data["Campus"][1])

**Your turn:** Change Chloe's age from 19 to 20 without changing the other ages. Print Chloe's name and updated age.

For the check, keep these result names: `data`.

**Hint:** Chloe is at position 2 inside each list.

**Watch for this:** Positions in lists are numbers; dictionary keys can be text. Do not swap their roles.

In [ ]:
# Change this code to complete the task.
# Question: Change Chloe's age from 19 to 20 without changing the other ages. Print Chloe's name and updated age.
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19],
    "Campus": ["RAK", "Dubai", "RAK"]
}
data["Age"][2] = 20
print(data["Name"][2], data["Age"][2])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert data['Age'] == [20, 21, 20]
assert data['Name'][2] == 'Chloe'
print("Code checks passed. Review the written explanation as well.")

The age list becomes [20, 21, 20]. Updating one nested list position does not change the other positions.

**Quick check:** What does data["Age"][1] select?

1. The first key.
2. Every age.
3. The second age value.

Answer: 3. The first brackets retrieve the Age list; the second retrieve its position 1.

**Explain what you learned:** I can read a value from a dictionary containing lists.

### M02C03. A list of dictionaries describes rows

A second arrangement stores one dictionary per student inside a list. Each dictionary is a record, or row. rows[1] selects the second record; ["Name"] then selects a field within it. Both arrangements can later become a pandas DataFrame.

**Read:** What is stored in each element of rows?

Answer: One dictionary describing one student record.

**Predict:** What will rows[1]["Name"] print?

Answer: Bilal.

**Explain:** Why is the order of the brackets different from data["Name"][1] in the previous cell?

Answer: rows is organized as a list of records, so select the record first. data was organized as named columns, so select the column first.



In [ ]:
# Predict before running: What will rows[1]["Name"] print?
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21, "Campus": "Dubai"},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
print(rows[1])
print(rows[1]["Name"])
print(len(rows))

**Your turn:** Add Dana, age 22, at Dubai as a fourth dictionary. Print the last student's name and the total record count.

For the check, keep these result names: `rows`.

**Hint:** Use rows.append({...}) and then rows[-1]["Name"].

**Watch for this:** Count records with len(rows); len(rows[0]) counts fields in the first record.

In [ ]:
# Change this code to complete the task.
# Question: Add Dana, age 22, at Dubai as a fourth dictionary. Print the last student's name and the total record count.
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21, "Campus": "Dubai"},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
rows.append({"Name": "Dana", "Age": 22, "Campus": "Dubai"})
print(rows[-1]["Name"])
print(len(rows))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(rows) == 4 and rows[-1] == {'Name': 'Dana', 'Age': 22, 'Campus': 'Dubai'}
print("Code checks passed. Review the written explanation as well.")

Dana is the last record and there are four records. This arrangement keeps each student's fields together.

**Quick check:** Which arrangement stores one dictionary per row?

1. A list of dictionaries.
2. A dictionary of numbers.
3. A single text string.

Answer: 1. A list of dictionaries groups each record's fields inside its own dictionary.

**Explain what you learned:** I can tell column-oriented data from row-oriented records.

### M02C04. Check lengths before making a table

A rectangular table needs the same number of values in each plain list column. len() counts the values in a list. You can inspect all column lengths before constructing the DataFrame. A missing measurement still needs a position, often filled with None.

**Read:** What does == do here?

Answer: It compares two lengths and returns True or False. It does not assign a value.

**Predict:** What happens to same_length if you remove the last age only?

Answer: It becomes False: there would be three names but only two ages.

**Explain:** Why is adding a zero not always a good way to represent an unknown age?

Answer: Zero is a real numerical value. Using it could incorrectly suggest the student is zero years old and distort calculations.



In [ ]:
# Predict before running: What happens to same_length if you remove the last age only?
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19]
}
lengths = {"Name": len(data["Name"]), "Age": len(data["Age"])}
same_length = lengths["Name"] == lengths["Age"]
print(lengths)
print("Ready for a rectangular table:", same_length)

**Your turn:** Add Dana to the Name list. Her age is unknown, so append None to Age. Confirm that the two lengths still match.

For the check, keep these result names: `data`, `same_length`.

**Hint:** Append one value to each list; use None without quotes.

**Watch for this:** None is a missing-value marker here; "None" in quotes is ordinary text.

In [ ]:
# Change this code to complete the task.
# Question: Add Dana to the Name list. Her age is unknown, so append None to Age. Confirm that the two lengths still match.
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19]
}
data["Name"].append("Dana")
data["Age"].append(None)
same_length = len(data["Name"]) == len(data["Age"])
print(data)
print("Matching lengths:", same_length)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert same_length is True and len(data['Name']) == 4
assert data['Age'][-1] is None
print("Code checks passed. Review the written explanation as well.")

Both lists now contain four positions. None marks the missing age while preserving the row alignment.

**Quick check:** What is needed for a DataFrame built from plain list columns?

1. Every value must be text.
2. All column lists must have equal length.
3. Each column must have a different length.

Answer: 2. Plain list columns must have equal lengths. Indexed Series have separate alignment rules, introduced next.

**Explain what you learned:** I can check whether list-based columns form a consistent table.

## Module 03. Meet the pandas Series

Create a labeled column and explain positions, labels, and arithmetic.

Before you start: Know dictionary keys, list positions, and the difference between = and ==.

Official reference: https://pandas.pydata.org/docs/user_guide/dsintro.html#series

### M03C01. One column, with labels

The setup imports pandas using the short name pd. A Series is a one-dimensional collection of values with an index: labels attached to those values. Without an explicit index, pandas supplies labels starting at 0. A Series also has a dtype describing how its values are stored.

**Read:** What does pd refer to in pd.Series(...) ?

Answer: pd is the alias for the pandas package created by import pandas as pd in the setup.

**Predict:** Which index labels will this Series have?

Answer: 0, 1, and 2.

**Explain:** Is the index printed on the left an extra score?

Answer: No. It labels each score; it is not an additional measurement in the Series.



In [ ]:
# Predict before running: Which index labels will this Series have?
scores = pd.Series([88, 72, 91], name="Score")
print(scores)
print("Number of values:", len(scores))
print("Index:", scores.index.tolist())

**Your turn:** Create a Series called hours containing 8, 5, and 9. Give it the name Study_Hours and print its length.

For the check, keep these result names: `hours`.

**Hint:** Change both the list values and the name= argument.

**Watch for this:** Do not call the dtype text a data row; it describes the Series.

In [ ]:
# Change this code to complete the task.
# Question: Create a Series called hours containing 8, 5, and 9. Give it the name Study_Hours and print its length.
hours = pd.Series([8, 5, 9], name="Study_Hours")
print(hours)
print("Number of values:", len(hours))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert hours.tolist() == [8, 5, 9] and hours.name == 'Study_Hours'
print("Code checks passed. Review the written explanation as well.")

The Series contains three study-hour values and is named Study_Hours. Its default index is still 0, 1, 2.

**Quick check:** A Series is best described as:

1. A complete folder.
2. A Python keyword.
3. One labeled sequence of values.

Answer: 3. A Series is a one-dimensional labeled pandas data structure.

**Explain what you learned:** I can distinguish Series values, index labels, and the Series name.

### M03C02. Choose labels or positions explicitly

You can supply meaningful labels, such as student IDs, as the Series index. loc selects an index label; iloc selects an integer position. These can identify the same value, but they ask different questions. Using them explicitly avoids confusing a number used as a label with a position.

**Read:** Which expression asks for student S002, and which asks for the second value?

Answer: scores.loc["S002"] asks for the label S002. scores.iloc[1] asks for position 1, the second value.

**Predict:** What two values are printed on the last line?

Answer: 72 and 72.

**Explain:** After sorting scores, will position 1 always belong to S002?

Answer: No. Sorting can change positions. The S002 label still identifies that student's score.



In [ ]:
# Predict before running: What two values are printed on the last line?
scores = pd.Series([88, 72, 91], index=["S001", "S002", "S003"], name="Score")
by_label = scores.loc["S002"]
by_position = scores.iloc[1]
print(scores)
print(by_label, by_position)

**Your turn:** Select S003 by label and the first score by position. Store them as by_label and by_position, then print both.

For the check, keep these result names: `by_label`, `by_position`.

**Hint:** Use loc with the ID text, and iloc with 0 for the first position.

**Watch for this:** Prefer .iloc[0] when you mean first position; do not rely on scores[0] with a text index.

In [ ]:
# Change this code to complete the task.
# Question: Select S003 by label and the first score by position. Store them as by_label and by_position, then print both.
scores = pd.Series([88, 72, 91], index=["S001", "S002", "S003"], name="Score")
by_label = scores.loc["S003"]
by_position = scores.iloc[0]
print(by_label, by_position)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert by_label == 91 and by_position == 88
print("Code checks passed. Review the written explanation as well.")

S003 has score 91. Position 0 contains S001's score, 88.

**Quick check:** Which selector is explicitly positional?

1. iloc
2. loc
3. name

Answer: 1. iloc always selects by integer position rather than index label.

**Explain what you learned:** I can select a Series value by label or by position.

### M03C03. Do arithmetic on a whole Series

Pandas can apply an arithmetic operation to all values at once. This is called a vectorized operation. scores + 2 creates a new result; it does not replace scores unless you assign the result back. clip() can enforce an upper or lower boundary when that is the stated rule.

**Read:** Does scores + 2 work on one value or every value?

Answer: It adds 2 to every value in the Series, keeping the index labels.

**Predict:** What is the new result for S003, and what remains in scores for S003?

Answer: The new result is 101. The original scores value remains 99.

**Explain:** Why might 101 be unacceptable for a score measured out of 100?

Answer: It exceeds the permitted range. If the rule allows bonus marks but caps scores at 100, clipping can apply that explicit rule.



In [ ]:
# Predict before running: What is the new result for S003, and what remains in scores for S003?
scores = pd.Series([88, 72, 99], index=["S001", "S002", "S003"], name="Score")
result = scores + 2
print("Original:")
print(scores)
print("After adding two:")
print(result)

**Your turn:** Add 5 to each score, then cap the result at 100. Keep the original scores unchanged.

For the check, keep these result names: `result`, `scores`.

**Hint:** Apply .clip(upper=100) to the calculated Series.

**Watch for this:** Use clipping only when a justified business or teaching rule calls for it.

In [ ]:
# Change this code to complete the task.
# Question: Add 5 to each score, then cap the result at 100. Keep the original scores unchanged.
scores = pd.Series([88, 72, 99], index=["S001", "S002", "S003"], name="Score")
result = (scores + 5).clip(upper=100)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [93, 77, 100]
assert scores.tolist() == [88, 72, 99]
print("Code checks passed. Review the written explanation as well.")

The new scores are 93, 77, and 100. Clipping applies the stated maximum; it should not be used to hide unexplained invalid source data.

**Quick check:** What does result = scores + 2 normally do?

1. Changes the original scores in place.
2. Creates a new Series assigned to result.
3. Adds a new dictionary key.

Answer: 2. The arithmetic expression produces a new Series; scores stays unchanged.

**Explain what you learned:** I can calculate a new labeled column without writing a loop.

### M03C04. Labels control Series alignment

When combining two Series, pandas matches index labels, even when their order differs. It does not simply pair the first value in one Series with the first value in the other. A label missing from one side normally produces a missing result for arithmetic.

**Read:** Which project value is added to the quiz score for S001?

Answer: 44, because its label is S001, even though it is the second value in project.

**Predict:** What total will S003 have?

Answer: 65, from 19 + 46.

**Explain:** Why is label alignment useful for student records?

Answer: It keeps values matched to the same student when the two inputs have different row orders, provided their labels identify students correctly.



In [ ]:
# Predict before running: What total will S003 have?
quiz = pd.Series({"S001": 17, "S002": 14, "S003": 19})
project = pd.Series({"S003": 46, "S001": 44, "S002": 36})
result = quiz + project
print(result)

**Your turn:** Remove S002 from project, add the two Series again, and print which result is missing. Do not replace the missing project with zero.

For the check, keep these result names: `result`.

**Hint:** Build project with only S003 and S001; use result.isna() to inspect missing results.

**Watch for this:** Equal lengths do not guarantee correct alignment; inspect the actual index labels.

In [ ]:
# Change this code to complete the task.
# Question: Remove S002 from project, add the two Series again, and print which result is missing. Do not replace the missing project with zero.
quiz = pd.Series({"S001": 17, "S002": 14, "S003": 19})
project = pd.Series({"S003": 46, "S001": 44})
result = quiz + project
print(result)
print("Missing total:")
print(result.isna())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc['S001'] == 61 and result.loc['S003'] == 65
assert pd.isna(result.loc['S002']) and result.isna().sum() == 1
print("Code checks passed. Review the written explanation as well.")

S001 totals 61 and S003 totals 65. S002 has a missing total because its project value is absent. A missing project is not automatically a zero score.

**Quick check:** How does pandas add two Series with index labels?

1. By matching the labels.
2. Only by their printed order.
3. By sorting their numerical values first.

Answer: 1. Series arithmetic aligns matching index labels before combining values.

**Explain what you learned:** I can explain why label alignment can create missing values.

## Module 04. Build and inspect a DataFrame

Turn Python data into a table and inspect its structure before analysis.

Before you start: Know dictionaries, lists, Series values, and index labels.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/01_table_oriented.html

### M04C01. Make a table from a dictionary

A DataFrame is a two-dimensional table with rows and named columns. Pass a dictionary of equally long lists to pd.DataFrame(). Each dictionary key becomes a column name. Each list position supplies the values for one row.

**Read:** What becomes the two column names?

Answer: The dictionary keys Name and Age.

**Predict:** What does df.shape print?

Answer: (3, 2): three rows and two columns.

**Explain:** Does the default index count as a third data column?

Answer: No. The index labels the rows and is separate from the two named data columns.



In [ ]:
# Predict before running: What does df.shape print?
data = {"Name": ["Amira", "Bilal", "Chloe"], "Age": [20, 21, 19]}
df = pd.DataFrame(data)
print(df)
print(type(df).__name__)
print("Rows and columns:", df.shape)

**Your turn:** Add a Campus list with RAK, Dubai, and RAK to data before building df. Print the new table and shape.

For the check, keep these result names: `df`.

**Hint:** Add another key with a list of exactly three values.

**Watch for this:** Call pd.DataFrame(...) with a capital D and F; Python names are case-sensitive.

In [ ]:
# Change this code to complete the task.
# Question: Add a Campus list with RAK, Dubai, and RAK to data before building df. Print the new table and shape.
data = {
    "Name": ["Amira", "Bilal", "Chloe"],
    "Age": [20, 21, 19],
    "Campus": ["RAK", "Dubai", "RAK"]
}
df = pd.DataFrame(data)
print(df)
print("Rows and columns:", df.shape)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert df.shape == (3, 3) and df.loc[1, 'Campus'] == 'Dubai'
print("Code checks passed. Review the written explanation as well.")

The shape becomes (3, 3). The number of rows stays the same while the new key adds one data column.

**Quick check:** In df.shape, what comes first?

1. Column count.
2. Row count.
3. The first score.

Answer: 2. shape is a pair in the order (number of rows, number of columns).

**Explain what you learned:** I can turn a dictionary of lists into a DataFrame.

### M04C02. Build a table from records

A list of dictionaries can also become a DataFrame. Each dictionary contributes a row, and pandas matches fields by their keys. If one record lacks a field, pandas marks that cell as missing. The table therefore does not require every record to contain the same keys.

**Read:** Which record is missing the Campus key?

Answer: The second record, for Bilal.

**Predict:** Will the second row disappear because its campus is absent?

Answer: No. Bilal stays in the table with a missing Campus value.

**Explain:** Why is this different from unequal plain lists?

Answer: Record dictionaries name their fields, so pandas can identify the missing field for a particular row. Unequal plain lists do not state which row is missing a value.



In [ ]:
# Predict before running: Will the second row disappear because its campus is absent?
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
df = pd.DataFrame(rows)
print(df)
print("Missing campus:", df["Campus"].isna().sum())

**Your turn:** Add Campus as Dubai to Bilal's record before constructing df. Check that no campus values are missing.

For the check, keep these result names: `df`.

**Hint:** Update rows[1]["Campus"] before calling pd.DataFrame(rows).

**Watch for this:** Missing-value display can vary with dtype and pandas version; inspect it with isna().

In [ ]:
# Change this code to complete the task.
# Question: Add Campus as Dubai to Bilal's record before constructing df. Check that no campus values are missing.
rows = [
    {"Name": "Amira", "Age": 20, "Campus": "RAK"},
    {"Name": "Bilal", "Age": 21},
    {"Name": "Chloe", "Age": 19, "Campus": "RAK"}
]
rows[1]["Campus"] = "Dubai"
df = pd.DataFrame(rows)
print(df)
print("Missing campus:", df["Campus"].isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert df.loc[1, 'Campus'] == 'Dubai' and df['Campus'].isna().sum() == 0
print("Code checks passed. Review the written explanation as well.")

Bilal now has Dubai in the Campus column. There are zero missing Campus values.

**Quick check:** A record is missing one field. What can pd.DataFrame(rows) do?

1. Invent a likely value.
2. Delete all the other records.
3. Keep the row and mark the field as missing.

Answer: 3. Pandas aligns record fields by key and represents an absent field as missing.

**Explain what you learned:** I can build a DataFrame from records and identify a missing field.

### M04C03. Look at the beginning and the end

The setup provides students(), a helper that returns a fresh copy of the 12-row fictional practice table. head(n) shows the first n rows and tail(n) shows the last n rows. These previews help you understand a file without printing the whole dataset. A preview alone does not prove every row is valid.

**Read:** Is students() a built-in pandas function?

Answer: No. It is a helper defined in this workbook's setup to return the practice data.

**Predict:** Which student IDs appear in tail(2)?

Answer: S011 and S012.

**Explain:** Why should you avoid deciding that a whole dataset is clean after reading only head()?

Answer: Problems can occur in rows that the preview does not show. You also need checks across all rows.



In [ ]:
# Predict before running: Which student IDs appear in tail(2)?
df = students()
first = df.head(3)
last = df.tail(2)
print("First three rows:")
print(first)
print("Last two rows:")
print(last)

**Your turn:** Show the first two rows and last three rows. Print the full table's shape to confirm previews have not removed rows.

For the check, keep these result names: `df`, `first`, `last`.

**Hint:** Use head(2), tail(3), and df.shape.

**Watch for this:** head() without an argument defaults to five rows, or fewer when fewer exist.

In [ ]:
# Change this code to complete the task.
# Question: Show the first two rows and last three rows. Print the full table's shape to confirm previews have not removed rows.
df = students()
first = df.head(2)
last = df.tail(3)
print(first)
print(last)
print("Full shape:", df.shape)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert first['Student_ID'].tolist() == ['S001', 'S002']
assert last['Student_ID'].tolist() == ['S010', 'S011', 'S012'] and df.shape == (12, 9)
print("Code checks passed. Review the written explanation as well.")

The previews show 2 and 3 rows, while df remains a 12-row, 9-column table.

**Quick check:** What does df.head(3) do?

1. Returns the first three rows.
2. Permanently deletes all other rows.
3. Returns the first three columns.

Answer: 1. head(3) returns a row preview; it does not alter df.

**Explain what you learned:** I can preview a dataset while preserving its full contents.

### M04C04. Inspect names, types, and a first summary

Attributes such as shape and columns are stored information, so they do not use parentheses. Methods such as info() perform an action and do use parentheses. info() reports column names, non-missing counts, and dtypes; describe() returns summary statistics. Text dtype labels can differ across pandas versions, so focus on what the data represent.

**Read:** Why is it df.shape but df.info()?

Answer: shape is an attribute; info is a method that must be called to produce its report.

**Predict:** What will count be for both Age and Score in this clean table?

Answer: 12 for each, because all 12 rows have values in both columns.

**Explain:** Why is df.info() better than only looking at the first row?

Answer: It checks the full table's structure, non-missing counts, and types rather than showing one record.



In [ ]:
# Predict before running: What will count be for both Age and Score in this clean table?
df = students()
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
df.info()
summary = df[["Age", "Score"]].describe()
print(summary.round(2))

**Your turn:** Inspect dtypes, then create a describe() summary for Study_Hours and Attendance only. Round it to one decimal place.

For the check, keep these result names: `summary`.

**Hint:** Use df.dtypes and select two columns with a list inside the brackets.

**Watch for this:** Call df.info() directly. print(df.info()) also prints None because info() returns None.

In [ ]:
# Change this code to complete the task.
# Question: Inspect dtypes, then create a describe() summary for Study_Hours and Attendance only. Round it to one decimal place.
df = students()
print(df.dtypes)
summary = df[["Study_Hours", "Attendance"]].describe().round(1)
print(summary)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert summary.columns.tolist() == ['Study_Hours', 'Attendance']
assert summary.loc['count', 'Study_Hours'] == 12 and summary.loc['max', 'Attendance'] == 99
print("Code checks passed. Review the written explanation as well.")

The summary covers only Study_Hours and Attendance. Rounding changes the displayed summary, not the original data values.

**Quick check:** Which expression correctly accesses the table shape?

1. df.shape()
2. df[shape]
3. df.shape

Answer: 3. shape is an attribute, so it is accessed without parentheses.

**Explain what you learned:** I can inspect table structure before writing an analysis.

## Module 05. Select exactly what you need

Choose rows, columns, and individual cells using labels and positions.

Before you start: Know the difference between a Series and a DataFrame and recognize an index.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html

### M05C01. One bracket or two?

df["Score"] selects one column as a Series. df[["Score"]] passes a list of column names and returns a DataFrame, even with only one column. To select several columns, put their names in that inner list. The distinction matters when another operation expects a table.

**Read:** What is the inner ["Name", "Score"]?

Answer: A Python list containing the names of the columns to select.

**Predict:** What are the shapes of column and table?

Answer: (12,) for the Series and (12, 1) for the one-column DataFrame.

**Explain:** Why use brackets rather than df.Score as the main teaching pattern?

Answer: Brackets work consistently with names containing spaces and names that could conflict with DataFrame attributes or methods.



In [ ]:
# Predict before running: What are the shapes of column and table?
df = students()
column = df["Score"]
table = df[["Score"]]
result = df[["Name", "Score"]]
print(type(column).__name__, column.shape)
print(type(table).__name__, table.shape)
print(result.head(3))

**Your turn:** Select Campus and Program, in that order, into a DataFrame named result. Print its first four rows and shape.

For the check, keep these result names: `result`.

**Hint:** Place both exact column names inside one inner list.

**Watch for this:** df["Name", "Score"] is not the same as df[["Name", "Score"]] for these ordinary columns.

In [ ]:
# Change this code to complete the task.
# Question: Select Campus and Program, in that order, into a DataFrame named result. Print its first four rows and shape.
df = students()
result = df[["Campus", "Program"]]
print(result.head(4))
print(result.shape)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert isinstance(result, pd.DataFrame) and result.shape == (12, 2)
assert result.columns.tolist() == ['Campus', 'Program']
print("Code checks passed. Review the written explanation as well.")

result has 12 rows and two columns in the requested order: Campus, then Program.

**Quick check:** Which returns a one-column DataFrame?

1. df["Score"]
2. df[["Score"]]
3. df.shape

Answer: 2. Passing a list of column names preserves a DataFrame result.

**Explain what you learned:** I can predict whether a column selection returns a Series or a DataFrame.

### M05C02. Select row and column labels with loc

set_index("Student_ID") uses the student IDs as row labels and, by default, removes that field from the data columns. loc has the form table.loc[row_labels, column_labels]. A list keeps the selection table-shaped. at is convenient when you need a single value with a row label and column label.

**Read:** What separates row selection from column selection inside loc?

Answer: The comma. The row labels are on the left; the column labels are on the right.

**Predict:** Which two names appear, and what is one_score?

Answer: Amira and Chloe appear. one_score is 91.

**Explain:** Why must Student_ID be the index for this exact loc expression?

Answer: loc looks for row labels. Without set_index, the default row labels are numbers rather than S001 and S003.



In [ ]:
# Predict before running: Which two names appear, and what is one_score?
df = students().set_index("Student_ID")
result = df.loc[["S001", "S003"], ["Name", "Score"]]
one_score = df.at["S003", "Score"]
print(result)
print("One score:", one_score)

**Your turn:** Select S002 and S004 with Name and Campus into result. Use at to get S004's Score into one_score.

For the check, keep these result names: `one_score`, `result`.

**Hint:** Provide a list of IDs on the left of the comma and a list of column names on the right.

**Watch for this:** set_index does not automatically check that IDs are unique; uniqueness needs its own check.

In [ ]:
# Change this code to complete the task.
# Question: Select S002 and S004 with Name and Campus into result. Use at to get S004's Score into one_score.
df = students().set_index("Student_ID")
result = df.loc[["S002", "S004"], ["Name", "Campus"]]
one_score = df.at["S004", "Score"]
print(result)
print("One score:", one_score)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ['S002', 'S004'] and result['Campus'].tolist() == ['Dubai', 'Dubai']
assert one_score == 65
print("Code checks passed. Review the written explanation as well.")

Bilal and Dana both belong to Dubai. Dana's individual score is 65.

**Quick check:** What does loc use to select rows?

1. Only row positions.
2. Only alphabetical sorting.
3. Index labels or a Boolean mask.

Answer: 3. loc is label-based and can also accept a Boolean mask for row selection.

**Explain what you learned:** I can read selected records and one value using meaningful labels.

### M05C03. Select row and column positions with iloc

iloc uses numerical positions for both rows and columns. Positions start at 0 even when the printed index labels are student IDs. A slice excludes its stop position. iat reads one value using a row position and a column position.

**Read:** Which row positions are included by 1:4?

Answer: Positions 1, 2, and 3: Bilal, Chloe, and Dana.

**Predict:** Which columns do positions 0 and 2 represent here?

Answer: Name and Score. The table was first arranged with columns Name, Campus, Score.

**Explain:** Why can a hard-coded column position become risky after reordering columns?

Answer: The same numerical position can refer to a different column after the order changes.



In [ ]:
# Predict before running: Which columns do positions 0 and 2 represent here?
df = students()[["Name", "Campus", "Score"]]
result = df.iloc[1:4, [0, 2]]
one_score = df.iat[1, 2]
print(result)
print("One score:", one_score)

**Your turn:** Select the first two rows and all three columns into result. Read the first student's score with iat into one_score.

For the check, keep these result names: `one_score`, `result`.

**Hint:** Use :2 for the first two row positions and : for all column positions.

**Watch for this:** iloc expects positions, so df.iloc["S001"] is not valid positional selection.

In [ ]:
# Change this code to complete the task.
# Question: Select the first two rows and all three columns into result. Read the first student's score with iat into one_score.
df = students()[["Name", "Campus", "Score"]]
result = df.iloc[:2, :]
one_score = df.iat[0, 2]
print(result)
print("One score:", one_score)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 3) and result['Name'].tolist() == ['Amira', 'Bilal']
assert one_score == 88
print("Code checks passed. Review the written explanation as well.")

result contains Amira and Bilal across all three selected columns. Amira's score is 88.

**Quick check:** What does df.iloc[1:4] include?

1. Row positions 1, 2, and 3.
2. Row positions 1, 2, 3, and 4.
3. Only a row whose label is 4.

Answer: 1. The stop position is excluded, so 1:4 includes 1, 2, and 3.

**Explain what you learned:** I can select a rectangular portion of a table by position.

### M05C04. Compare the two slice endpoints

For this sorted, unique index, a loc label slice includes both its start and stop labels. An iloc position slice excludes its stop position. Compare the selected IDs instead of assuming the two expressions return the same rows. Label slicing works best when the index is appropriately sorted and its labels are understood.

**Read:** Which expression includes the stop value?

Answer: The loc label slice includes the existing stop label S004 in this sorted unique index.

**Predict:** How many rows appear in each result?

Answer: Three in by_label: S002, S003, S004. Two in by_position: S002 and S003.

**Explain:** Does df.iloc[1:3] mean index labels 1 through 3?

Answer: No. It means row positions 1 and 2, regardless of their index labels.



In [ ]:
# Predict before running: How many rows appear in each result?
df = students().set_index("Student_ID").sort_index()
by_label = df.loc["S002":"S004", ["Name", "Score"]]
by_position = df.iloc[1:3][["Name", "Score"]]
print("Label slice:")
print(by_label)
print("Position slice:")
print(by_position)

**Your turn:** Change by_position so that its position slice returns exactly the same three rows as by_label. Store the Boolean equality check in same, then print the selected rows and the check.

For the check, keep these result names: `by_position`, `same`.

**Hint:** To include row position 3, stop at position 4.

**Watch for this:** Selecting from a selection is fine for reading; use one .loc assignment when changing the original table.

In [ ]:
# Change this code to complete the task.
# Question: Change by_position so that its position slice returns exactly the same three rows as by_label. Store the Boolean equality check in same, then print the selected rows and the check.
df = students().set_index("Student_ID").sort_index()
by_label = df.loc["S002":"S004", ["Name", "Score"]]
by_position = df.iloc[1:4][["Name", "Score"]]
same = by_label.equals(by_position)
print(by_position)
print("Same result:", same)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert same is True and by_position.index.tolist() == ['S002', 'S003', 'S004']
print("Code checks passed. Review the written explanation as well.")

Both results now contain S002, S003, and S004 with the same column order and labels. equals() reports True.

**Quick check:** On this sorted index, which is true?

1. Both loc and iloc exclude the stop.
2. loc includes its stop label; iloc excludes its stop position.
3. Both selectors ignore their start.

Answer: 2. This label slice includes S004, while a positional slice stops before its endpoint.

**Explain what you learned:** I can explain why a label slice and a position slice can return different rows.

## Module 06. Filter with conditions

Build Boolean masks and combine conditions to answer precise questions.

Before you start: Know column selection and the basic comparison operators.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html

### M06C01. See the True and False values first

Comparing a Series with a value produces a Boolean Series, often called a mask. Each True marks a row that meets the condition. df.loc[mask, columns] keeps those rows and selects the requested columns. Keeping the mask in its own variable makes the logic easier to inspect.

**Read:** What is the type of value at each position in mask?

Answer: A Boolean value, True or False, for this clean non-missing Score column.

**Predict:** How many rows pass Score > 80?

Answer: Six: S001, S003, S006, S007, S010, and S012.

**Explain:** Why is df["Score"] > 80 not already the filtered table?

Answer: It only contains the condition results. A second selection uses that mask to retrieve the matching records.



In [ ]:
# Predict before running: How many rows pass Score > 80?
df = students()
mask = df["Score"] > 80
print(mask)
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)

**Your turn:** Change the condition to Score < 70. Select Student_ID, Name, and Score into result. Print the number of matches.

For the check, keep these result names: `mask`, `result`.

**Hint:** Change > to < and use 70 as the threshold.

**Watch for this:** A mask should correspond to the table being filtered; a mask from another table can misalign.

In [ ]:
# Change this code to complete the task.
# Question: Change the condition to Score < 70. Select Student_ID, Name, and Score into result. Print the number of matches.
df = students()
mask = df["Score"] < 70
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)
print("Matches:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S004', 'S008', 'S011'] and mask.sum() == 3
print("Code checks passed. Review the written explanation as well.")

Dana, Hassan, and Khalid match the new rule, giving three rows. The question changed from high scores to scores needing attention.

**Quick check:** What does a True in a filter mask mean?

1. Keep that row.
2. Delete the column.
3. Replace the value with 1.

Answer: 1. A True marks a row to keep when the mask is used for selection.

**Explain what you learned:** I can separate the condition from the table selection it controls.

### M06C02. Combine conditions with AND and OR

Use & for element-by-element AND and | for element-by-element OR. Put parentheses around each comparison before combining them. AND requires both conditions; OR requires at least one. Python's and and or do not combine pandas Series masks this way.

**Read:** What two conditions must a row satisfy?

Answer: Its Campus must equal RAK and its Score must be at least 80.

**Predict:** Which names remain?

Answer: Amira, Chloe, and Grace.

**Explain:** What changes if & is replaced by |?

Answer: A student can pass either condition: be at RAK, have Score at least 80, or satisfy both. More rows can qualify.



In [ ]:
# Predict before running: Which names remain?
df = students()
mask = (df["Campus"] == "RAK") & (df["Score"] >= 80)
result = df.loc[mask, ["Student_ID", "Name", "Campus", "Score"]]
print(result)

**Your turn:** Select students who are at RAK OR have Score at least 90. Keep the same four output columns.

For the check, keep these result names: `result`.

**Hint:** Use | between two parenthesized comparisons.

**Watch for this:** Do not omit the parentheses around comparisons; operator precedence can change the expression.

In [ ]:
# Change this code to complete the task.
# Question: Select students who are at RAK OR have Score at least 90. Keep the same four output columns.
df = students()
mask = (df["Campus"] == "RAK") | (df["Score"] >= 90)
result = df.loc[mask, ["Student_ID", "Name", "Campus", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S001', 'S003', 'S005', 'S006', 'S007', 'S009', 'S011']
print("Code checks passed. Review the written explanation as well.")

All six RAK students qualify, and Fatima qualifies through her score of 95. Chloe meets both conditions but appears only once, so there are seven rows.

**Quick check:** Which correctly combines two pandas conditions with AND?

1. condition_a and condition_b
2. (condition_a) & (condition_b)
3. condition_a + "and" + condition_b

Answer: 2. Use & for element-wise AND; parenthesize each comparison when constructing the masks.

**Explain what you learned:** I can translate both, either, and at least into correct filter logic.

### M06C03. Membership, ranges, and NOT

isin() tests whether a value belongs to an allowed list. between() checks a range; inclusive="both" includes its two endpoints. The ~ operator reverses a Boolean mask. Naming masks can make a longer rule easier to explain.

**Read:** What does inclusive="both" tell between() to do?

Answer: Include values exactly equal to 70 or 90, as well as values between them.

**Predict:** Which AI students have scores between 70 and 90 here?

Answer: Amira, Grace, and Jamal.

**Explain:** Why does isin(["AI"]) use a list even for one allowed value?

Answer: isin expects a list-like collection of allowed values. The same expression can later include several categories.



In [ ]:
# Predict before running: Which AI students have scores between 70 and 90 here?
df = students()
in_program = df["Program"].isin(["AI"])
in_range = df["Score"].between(70, 90, inclusive="both")
result = df.loc[in_program & in_range, ["Name", "Program", "Score"]]
print(result)

**Your turn:** Select students whose Program is NOT AI and whose Score is between 70 and 85 inclusive. Print their names, programs, and scores.

For the check, keep these result names: `result`.

**Hint:** Reverse the membership mask with ~ and combine it with the new range mask.

**Watch for this:** Do not use Program == ["AI", "IT"] to test membership; use .isin(["AI", "IT"]).

In [ ]:
# Change this code to complete the task.
# Question: Select students whose Program is NOT AI and whose Score is between 70 and 85 inclusive. Print their names, programs, and scores.
df = students()
not_ai = ~df["Program"].isin(["AI"])
in_range = df["Score"].between(70, 85, inclusive="both")
result = df.loc[not_ai & in_range, ["Name", "Program", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Bilal', 'Ethan', 'Ivy', 'Lina']
print("Code checks passed. Review the written explanation as well.")

Bilal, Ethan, Ivy, and Lina qualify. ~ reverses the AI membership test before the score range is applied.

**Quick check:** Which operator reverses a Boolean Series?

1. !
2. not
3. ~

Answer: 3. Use ~ for element-wise Boolean reversal of a pandas mask.

**Explain what you learned:** I can express allowed categories, numerical ranges, and excluded categories.

### M06C04. Read a filter written with query

query() is another way to express a row filter, using a short expression in a string. Column names appear inside that expression. @limit refers to a Python variable outside it. Learn the normal Boolean-mask form first so you can check what the query means.

**Read:** Why is there an @ before limit?

Answer: It tells query to use the Python variable limit instead of looking for a column named limit.

**Predict:** Which Dubai students meet the threshold?

Answer: Fatima and Jamal.

**Explain:** Why is and accepted here even though a previous cell used &?

Answer: query parses its expression string. Outside query, combining pandas Series directly requires the element-wise operators such as &.



In [ ]:
# Predict before running: Which Dubai students meet the threshold?
df = students()
limit = 85
result = df.query("Score >= @limit and Campus == 'Dubai'")[["Name", "Campus", "Score"]]
print(result)

**Your turn:** Use a normal Boolean mask instead of query to find Dubai students with Score at least 80. Save the same three columns in result.

For the check, keep these result names: `result`.

**Hint:** Combine (df["Score"] >= limit) and (df["Campus"] == "Dubai") with &.

**Watch for this:** Use fixed, trusted query expressions for these lessons; do not construct executable expressions from arbitrary user text.

In [ ]:
# Change this code to complete the task.
# Question: Use a normal Boolean mask instead of query to find Dubai students with Score at least 80. Save the same three columns in result.
df = students()
limit = 80
mask = (df["Score"] >= limit) & (df["Campus"] == "Dubai")
result = df.loc[mask, ["Name", "Campus", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Fatima', 'Jamal', 'Lina']
print("Code checks passed. Review the written explanation as well.")

Fatima, Jamal, and Lina meet the threshold of 80. A mask and a query can express the same selection rule.

**Quick check:** Inside query, what does @limit refer to?

1. A Python variable named limit.
2. Always a column called limit.
3. The last row.

Answer: 1. @ accesses a variable from the surrounding Python scope within the query expression.

**Explain what you learned:** I can translate a simple query into a Boolean mask and explain both.

## Module 07. Sorting and index decisions

Order records deliberately and control which labels identify rows.

Before you start: Know filtering, loc, iloc, and table shape.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html#the-index-object

### M07C01. Sort rows by a value

sort_values() orders whole rows by one or more column values. ascending=False puts larger values first. It returns a new table, so assign the result if you want to keep the order. The index labels travel with their rows and are not automatically renumbered.

**Read:** What does ascending=False request?

Answer: Descending order, from larger scores to smaller scores.

**Predict:** Who are the top three students by Score?

Answer: Fatima with 95, Chloe with 91, and Jamal with 89.

**Explain:** Why are the printed index labels 5, 2, and 9 instead of 0, 1, and 2?

Answer: The original row labels stay attached to the records when the rows are sorted.



In [ ]:
# Predict before running: Who are the top three students by Score?
df = students()
result = df.sort_values("Score", ascending=False).head(3)
print(result[["Student_ID", "Name", "Score"]])
print("Index labels:", result.index.tolist())

**Your turn:** Find the three lowest scores instead. Keep Student_ID, Name, and Score in result.

For the check, keep these result names: `result`.

**Hint:** Sort ascending and then take head(3).

**Watch for this:** head(3) before sorting chooses the first three source rows, not the three highest scores.

In [ ]:
# Change this code to complete the task.
# Question: Find the three lowest scores instead. Keep Student_ID, Name, and Score in result.
df = students()
result = df.sort_values("Score", ascending=True).head(3)[["Student_ID", "Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Hassan', 'Dana', 'Khalid']
assert result['Score'].tolist() == [58, 65, 68]
print("Code checks passed. Review the written explanation as well.")

The three lowest scores are Hassan 58, Dana 65, and Khalid 68. Sorting whole rows keeps each score matched with its student.

**Quick check:** Does sort_values rearrange only the selected column's values?

1. Yes, other columns stay fixed.
2. No, it reorders whole rows.
3. It always removes the column.

Answer: 2. Whole rows move together so their fields stay associated.

**Explain what you learned:** I can sort a table and identify the top or bottom records.

### M07C02. Specify a tie rule

Rows can tie on a sort column. With one column, kind="stable" preserves their existing order within a tie. To define a new tie rule, sort by several columns and give one ascending flag per column. Clear tie rules make results easier to reproduce and explain.

**Read:** Which rows share a score?

Answer: Chloe, Amira, and Dana all have 80.

**Predict:** What is their order after this stable sort?

Answer: Chloe, Amira, then Dana, preserving their order in the original table.

**Explain:** Why might a report need a second sort column?

Answer: It provides a clear order among equal values, such as alphabetical order among students with the same score.



In [ ]:
# Predict before running: What is their order after this stable sort?
df = pd.DataFrame({
    "Name": ["Chloe", "Amira", "Bilal", "Dana"],
    "Score": [80, 80, 95, 80]
})
result = df.sort_values("Score", ascending=False, kind="stable")
print(result)

**Your turn:** Sort by Score descending, then Name ascending to break ties alphabetically.

For the check, keep these result names: `result`.

**Hint:** Pass ["Score", "Name"] and ascending=[False, True].

**Watch for this:** Alphabetical tie order must be requested explicitly; a stable sort preserves source order instead.

In [ ]:
# Change this code to complete the task.
# Question: Sort by Score descending, then Name ascending to break ties alphabetically.
df = pd.DataFrame({
    "Name": ["Chloe", "Amira", "Bilal", "Dana"],
    "Score": [80, 80, 95, 80]
})
result = df.sort_values(["Score", "Name"], ascending=[False, True])
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Bilal', 'Amira', 'Chloe', 'Dana']
print("Code checks passed. Review the written explanation as well.")

Bilal remains first with 95. The three scores of 80 are then ordered Amira, Chloe, Dana.

**Quick check:** What does a stable sort preserve for equal sort values?

1. The existing relative order.
2. Only the highest value.
3. Alphabetical order automatically.

Answer: 1. Stability preserves the source order of rows that tie on the sort key.

**Explain what you learned:** I can make the order of tied records explicit.

### M07C03. Set, check, and reset an index

An index labels rows; it is not automatically a unique ID. set_index() chooses the labels, and index.is_unique tells you whether any label repeats. reset_index() moves the current index back into a column and supplies new default row labels. Use reset_index(drop=True) only when you deliberately want to discard the current labels.

**Read:** What does indexed.index.is_unique tell you?

Answer: It is True when no index label repeats. Separate checks are still needed for missing or invalid IDs.

**Predict:** After reset_index(), is Student_ID a data column again?

Answer: Yes. It is restored as a column, while the row index becomes 0 through 11.

**Explain:** Why could reset_index(drop=True) lose useful information here?

Answer: Student_ID is currently the index, so drop=True would discard those IDs instead of restoring them to a column.



In [ ]:
# Predict before running: After reset_index(), is Student_ID a data column again?
df = students()
indexed = df.set_index("Student_ID")
print("Unique index:", indexed.index.is_unique)
result = indexed.reset_index()
print(result[["Student_ID", "Name"]].head(3))

**Your turn:** Set Student_ID as the index, select S003 and S001 in that order, and reset the selected index without losing the IDs.

For the check, keep these result names: `result`.

**Hint:** Use .loc[["S003", "S001"]] followed by .reset_index().

**Watch for this:** Having an index does not guarantee that its labels are unique or non-missing.

In [ ]:
# Change this code to complete the task.
# Question: Set Student_ID as the index, select S003 and S001 in that order, and reset the selected index without losing the IDs.
df = students()
indexed = df.set_index("Student_ID")
selected = indexed.loc[["S003", "S001"]]
result = selected.reset_index()
print(result[["Student_ID", "Name"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S003', 'S001']
assert result.index.tolist() == [0, 1]
print("Code checks passed. Review the written explanation as well.")

The selected rows are Chloe and Amira. Their IDs remain in the Student_ID column, and their new row labels are 0 and 1.

**Quick check:** Which operation keeps the old index as a column?

1. reset_index(drop=True)
2. reset_index()
3. sort_index()

Answer: 2. reset_index() preserves the old index values in a column unless drop=True is specified.

**Explain what you learned:** I can change an index without accidentally discarding record IDs.

### M07C04. Reindex to a requested order

reindex() aligns a table to a specified set of labels and their requested order. Existing labels bring their rows with them; new labels create missing values. This differs from sorting because you supply the desired labels explicitly. Use isna() to inspect newly missing cells.

**Read:** Where did the S999 row come from?

Answer: It was requested in reindex but was absent from the source index, so pandas created a row of missing values.

**Predict:** Which order will the index have?

Answer: S003, S001, S999, matching the order passed to reindex.

**Explain:** Why is a missing S999 score not automatically a failing score?

Answer: There is no source record for S999. Missing data does not establish a measured score or a fail decision.



In [ ]:
# Predict before running: Which order will the index have?
df = students().set_index("Student_ID")
result = df.reindex(["S003", "S001", "S999"])[["Name", "Score"]]
print(result)
print("Missing name:")
print(result["Name"].isna())

**Your turn:** Reindex to S002, S001, and S004, in that order. Confirm that none of their names is missing.

For the check, keep these result names: `result`.

**Hint:** Use only existing IDs and count missing names with .isna().sum().

**Watch for this:** Reindexing can fail when source labels are duplicated; check uniqueness before using labels as identifiers.

In [ ]:
# Change this code to complete the task.
# Question: Reindex to S002, S001, and S004, in that order. Confirm that none of their names is missing.
df = students().set_index("Student_ID")
result = df.reindex(["S002", "S001", "S004"])[["Name", "Score"]]
print(result)
print("Missing names:", result["Name"].isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ['S002', 'S001', 'S004']
assert result['Name'].isna().sum() == 0
print("Code checks passed. Review the written explanation as well.")

The output is Bilal, Amira, Dana in the requested order, with no missing names. All requested labels exist.

**Quick check:** What can reindex do when a requested label does not exist?

1. Guess the nearest record.
2. Raise an error in every case.
3. Create a row with missing values.

Answer: 3. For this unique index, reindex creates missing data for an absent requested label.

**Explain what you learned:** I can distinguish sorting existing rows from aligning to a requested index.

## Module 08. Create and update columns

Calculate new fields, tidy column names, and change selected values safely.

Before you start: Know arithmetic, column selection, Boolean masks, and assignment.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html

### M08C01. Calculate a new column

Assign a Series to a new column name to add a field. Arithmetic across columns works row by row after index alignment. Here the artificial Score_Per_Hour ratio demonstrates division; it is not a validated measure of student ability. Always consider units and the meaning of a calculation.

**Read:** Which part of the assignment names the new column?

Answer: df["Score_Per_Hour"] on the left side.

**Predict:** What is Amira's Score_Per_Hour before rounding?

Answer: 11.0, from 88 divided by 8.

**Explain:** Why should a high ratio not be treated as proof of better learning?

Answer: The ratio alone does not measure learning quality or account for other factors. It is only an arithmetic example with fictional data.



In [ ]:
# Predict before running: What is Amira's Score_Per_Hour before rounding?
df = students()
df["Score_Per_Hour"] = df["Score"] / df["Study_Hours"]
result = df[["Name", "Score", "Study_Hours", "Score_Per_Hour"]]
print(result.head(3).round(2))

**Your turn:** Start with a fresh df = students(). Create Score_Fraction by dividing Score by 100. Store Name, Score, and Score_Fraction for all students in result, then print its first three rows.

For the check, keep these result names: `df`, `result`.

**Hint:** Assign the division result to df["Score_Fraction"].

**Watch for this:** Before dividing by a column, check for zero and missing denominators; they can create undefined results.

In [ ]:
# Change this code to complete the task.
# Question: Start with a fresh df = students(). Create Score_Fraction by dividing Score by 100. Store Name, Score, and Score_Fraction for all students in result, then print its first three rows.
df = students()
df["Score_Fraction"] = df["Score"] / 100
result = df[["Name", "Score", "Score_Fraction"]]
print(result.head(3))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert df.shape == (12, 10) and result['Score_Fraction'].head(3).tolist() == [0.88, 0.72, 0.91]
print("Code checks passed. Review the written explanation as well.")

The first fractions are 0.88, 0.72, and 0.91. The original Score remains measured out of 100.

**Quick check:** What happens when you assign to a column name that does not yet exist?

1. Pandas adds the column.
2. Pandas always deletes a row.
3. Pandas sorts the index.

Answer: 1. Assigning a value or aligned Series to a new name creates a column.

**Explain what you learned:** I can create a calculated column and state its units and limits.

### M08C02. Use assign, rename, and drop

assign() returns a DataFrame with new or replaced columns. rename(columns=...) changes column labels, while drop(columns=...) removes named columns. These methods return new objects by default. Assign the returned result so your intended changes are kept.

**Read:** What is stored in the new Passed column?

Answer: True for scores at least 60 and False otherwise, using the rule stated in this example.

**Predict:** Does result still have a column named Name?

Answer: No. In result it has been renamed Student_Name. The original df still has Name.

**Explain:** Why does the final table still have nine columns?

Answer: It starts with nine, adds Passed, then drops Join_Date: 9 + 1 - 1 = 9.



In [ ]:
# Predict before running: Does result still have a column named Name?
df = students()
result = (df.assign(Passed=df["Score"] >= 60)
            .rename(columns={"Name": "Student_Name"})
            .drop(columns=["Join_Date"]))
print(result[["Student_Name", "Score", "Passed"]].head(4))
print("Result shape:", result.shape)

**Your turn:** Use assign to create Needs_Support for scores below 70. Rename Attendance to Attendance_Percent and drop Join_Date. Keep the result in result.

For the check, keep these result names: `df`, `result`.

**Hint:** Provide a Boolean Series to assign, a mapping to rename, and a list of names to drop.

**Watch for this:** Calling df.drop(...) without assigning its returned result does not change df by default.

In [ ]:
# Change this code to complete the task.
# Question: Use assign to create Needs_Support for scores below 70. Rename Attendance to Attendance_Percent and drop Join_Date. Keep the result in result.
df = students()
result = (df.assign(Needs_Support=df["Score"] < 70)
            .rename(columns={"Attendance": "Attendance_Percent"})
            .drop(columns=["Join_Date"]))
print(result[["Name", "Score", "Needs_Support", "Attendance_Percent"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Needs_Support'].sum() == 3 and 'Attendance_Percent' in result
assert 'Attendance' not in result and 'Join_Date' not in result
assert 'Attendance' in df and 'Needs_Support' not in df
print("Code checks passed. Review the written explanation as well.")

Needs_Support is True for Dana, Hassan, and Khalid. The longer attendance name makes the unit explicit. df remains available in its original form.

**Quick check:** Which removes a named column from the returned table?

1. df.rename(columns={"Score": "Mark"})
2. df.drop(columns=["Score"])
3. df.assign(Score=0)

Answer: 2. drop(columns=[...]) removes the specified columns; rename changes a label and assign creates or replaces values.

**Explain what you learned:** I can add fields and tidy column structure while keeping the original available.

### M08C03. Update the original with one loc assignment

Use one .loc[row_condition, column_name] assignment to change selected values in a DataFrame. This explicitly identifies both the rows and the field being changed. Avoid chained assignment such as df[mask]["Score"] = value; it does not reliably update the intended original object and does not do so under pandas 3 Copy-on-Write.

**Read:** Which row and column are changed?

Answer: The Score column for the row whose Student_ID is S004, Dana.

**Predict:** What is Dana's score after the assignment?

Answer: 67.

**Explain:** Why does a single loc assignment state your intention more clearly than two chained brackets?

Answer: It tells pandas to change the named column in selected rows of df itself, instead of writing into an intermediate selection.



In [ ]:
# Predict before running: What is Dana's score after the assignment?
df = students()
mask = df["Student_ID"] == "S004"
df.loc[mask, "Score"] = 67
result = df.loc[mask, ["Student_ID", "Name", "Score"]]
print(result)

**Your turn:** Apply a practice bonus of 3 points to students with Attendance at least 95. Cap scores at 100 and show only the updated students.

For the check, keep these result names: `df`, `result`.

**Hint:** Use one .loc[mask, "Score"] assignment; calculate from the same selected Score values and clip at 100.

**Watch for this:** Repeatedly running an incremental update on the same table would repeat the bonus; this cell starts with a fresh table each time.

In [ ]:
# Change this code to complete the task.
# Question: Apply a practice bonus of 3 points to students with Attendance at least 95. Cap scores at 100 and show only the updated students.
df = students()
mask = df["Attendance"] >= 95
df.loc[mask, "Score"] = (df.loc[mask, "Score"] + 3).clip(upper=100)
result = df.loc[mask, ["Student_ID", "Name", "Score", "Attendance"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S001', 'S003', 'S006', 'S010']
assert result['Score'].tolist() == [91, 94, 98, 92]
assert df.loc[df['Student_ID'] == 'S004', 'Score'].iloc[0] == 65
print("Code checks passed. Review the written explanation as well.")

Amira, Chloe, Fatima, and Jamal receive the practice bonus. Their scores become 91, 94, 98, and 92. Students outside the mask keep their scores.

**Quick check:** Which is the clear way to update selected scores in df?

1. df[mask]["Score"] = 67
2. df["Score"][mask] = 67
3. df.loc[mask, "Score"] = 67

Answer: 3. One loc assignment directly targets the selected rows and column in df.

**Explain what you learned:** I can modify selected values using one explicit loc assignment.

### M08C04. Keep an independent working copy

Assigning original to another name creates an alias: both names refer to the same Python object. copy() creates a separate DataFrame for this ordinary scalar-valued teaching table. Use a fresh copy when you want to experiment without changing the original. Pandas copy rules do not replace the need to understand which object you are editing.

**Read:** What does .copy() let us do in this example?

Answer: Change values in working while keeping original unchanged.

**Predict:** What two first-score values are printed?

Answer: 88 for original and 0 for working.

**Explain:** What would change if the code used working = original instead?

Answer: Both names would refer to the same object; changing working would also be visible through original.



In [ ]:
# Predict before running: What two first-score values are printed?
original = students()
working = original.copy()
working.loc[0, "Score"] = 0
print("Original first score:", original.loc[0, "Score"])
print("Working first score:", working.loc[0, "Score"])
print("Same object:", working is original)

**Your turn:** Copy only the RAK rows into working. Add 1 to their scores in the copy and verify Amira's original score is still 88.

For the check, keep these result names: `original`, `working`.

**Hint:** Filter with loc, call .copy(), and update the selected table rather than original.

**Watch for this:** For columns containing nested mutable Python objects, even a deep DataFrame copy does not recursively copy those inner objects; this table uses ordinary scalar values.

In [ ]:
# Change this code to complete the task.
# Question: Copy only the RAK rows into working. Add 1 to their scores in the copy and verify Amira's original score is still 88.
original = students()
working = original.loc[original["Campus"] == "RAK"].copy()
working["Score"] = working["Score"] + 1
print(working[["Name", "Score"]])
print("Amira in original:", original.loc[0, "Score"])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(working) == 6 and working.loc[0, 'Score'] == 89
assert original.loc[0, 'Score'] == 88 and original.shape == (12, 9)
print("Code checks passed. Review the written explanation as well.")

working contains six RAK students with scores increased by 1. Amira has 89 in working and 88 in original.

**Quick check:** What does working = original do?

1. Makes two names refer to the same object.
2. Always creates a separate deep copy.
3. Exports the data to a file.

Answer: 1. Simple assignment binds another name to the existing object; copy() is a separate operation.

**Explain what you learned:** I can protect source values while testing changes on a separate table.

## Module 09. Data types and safe conversion

Choose a useful data type and identify values that cannot be converted.

Before you start: You can select a column and create a new column.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#dtypes

### M09C01. A number and its text are different

A data type tells pandas what a column contains and which operations make sense. Quotation marks make "20" text. `astype()` converts values when the requested conversion is possible.

**Read:** What do the quotation marks around the ages tell Python?

Answer: These values are strings, even though they contain digits.

**Predict:** What will the next-year ages be?

Answer: 21, 22, and 20.

**Explain:** Why convert before adding 1?

Answer: We want numeric addition; text values are not numeric ages yet.



In [ ]:
# Predict before running: What will the next-year ages be?
ages = pd.Series(["20", "21", "19"], dtype="string")
result = ages.astype("int64")
print("Before:", ages.dtype)
print("After:", result.dtype)
print("Next year:", (result + 1).tolist())

**Your turn:** Start again with the same text ages. Convert them to numbers and calculate their mean in `result`.

For the check, keep these result names: `result`.

**Hint:** Convert the Series first; then call `.mean()`.

**Watch for this:** Do not assume a column is numeric just because its displayed values look like numbers.

In [ ]:
# Change this code to complete the task.
# Question: Start again with the same text ages. Convert them to numbers and calculate their mean in `result`.
ages = pd.Series(["20", "21", "19"], dtype="string")
result = ages.astype("int64").mean()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == 20.0
print("Code checks passed. Review the written explanation as well.")

The mean is 20.0. The conversion makes arithmetic meaningful.

**Quick check:** Which expression requests a numeric type?

1. ages.astype("int64")
2. ages.astype("string")
3. str(ages)

Answer: 1. int64 is an integer data type; string is text.

**Explain what you learned:** Check types before doing arithmetic.

### M09C02. Convert messy numbers and keep the reason

`pd.to_numeric()` parses values as numbers. With `errors="coerce"`, values it cannot parse become missing. Preserve the source so you can distinguish an invalid label from a value that was already missing.

**Read:** What does `errors="coerce"` ask pandas to do?

Answer: Replace a value that cannot be parsed as a number with a missing value.

**Predict:** Which row is marked as invalid text?

Answer: Only row 1, containing "absent". Row 2 was already missing.

**Explain:** Why should we keep the original values?

Answer: Conversion alone loses the distinction between invalid text and an originally missing value; "absent" may also have a special policy meaning.



In [ ]:
# Predict before running: Which row is marked as invalid text?
source = pd.Series(["88", "absent", None, "72"], dtype="string")
numbers = pd.to_numeric(source, errors="coerce")
invalid = source.notna() & numbers.isna()
result = pd.DataFrame({"Original": source, "Number": numbers, "Invalid_text": invalid})
print(result)

**Your turn:** Change "absent" to "90" in the source. Rebuild the table and count the missing numbers.

For the check, keep these result names: `result`.

**Hint:** Use `.isna().sum()` after the conversion.

**Watch for this:** Do not silently turn every conversion failure into zero.

In [ ]:
# Change this code to complete the task.
# Question: Change "absent" to "90" in the source. Rebuild the table and count the missing numbers.
source = pd.Series(["88", "90", None, "72"], dtype="string")
numbers = pd.to_numeric(source, errors="coerce")
invalid = source.notna() & numbers.isna()
result = pd.DataFrame({"Original": source, "Number": numbers, "Invalid_text": invalid})
print(result)
print("Missing numbers:", numbers.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Number"].isna().sum() == 1
assert not result["Invalid_text"].any()
print("Code checks passed. Review the written explanation as well.")

Only the original None remains missing; there are no invalid text values.

**Quick check:** Does converting "absent" to a missing value mean the score was zero?

1. Yes, missing and zero are identical
2. No; a grading rule must determine what absent means
3. Yes, pandas automatically applies grading rules

Answer: 2. Parsing is a technical operation. It does not decide the meaning or grading treatment of absence.

**Explain what you learned:** Audit conversion failures before deciding how to handle them.

### M09C03. Integers that can be missing

Pandas nullable integer type `Int64` can store whole numbers and `pd.NA`. Its capital I matters. Comparisons can also be missing, so decide how missing comparisons should behave in a filter.

**Read:** What does the capital I in `Int64` select?

Answer: Pandas nullable integer type, which supports a missing-value marker.

**Predict:** What happens to the missing age when 1 is added?

Answer: It remains missing; pandas does not invent an age.

**Explain:** Why is None not an age of zero?

Answer: It means the age was not recorded, while zero would be an actual number.



In [ ]:
# Predict before running: What happens to the missing age when 1 is added?
ages = pd.Series([20, None, 22], dtype="Int64")
result = ages + 1
print(result)
print("Known ages:", ages.notna().sum())

**Your turn:** Start with the same ages and keep only known ages of at least 21. Store the filtered Series in `result`.

For the check, keep these result names: `result`.

**Hint:** Build `ages >= 21`; use `.fillna(False)` on that mask.

**Watch for this:** Do not write `if pd.NA:`; its truth value is ambiguous. Use `.isna()` or `.notna()`.

In [ ]:
# Change this code to complete the task.
# Question: Start with the same ages and keep only known ages of at least 21. Store the filtered Series in `result`.
ages = pd.Series([20, None, 22], dtype="Int64")
mask = (ages >= 21).fillna(False)
result = ages[mask]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [22]
assert result.index.tolist() == [2]
print("Code checks passed. Review the written explanation as well.")

Only 22 remains. The explicit False says an unknown age does not satisfy this selection rule.

**Quick check:** Which value remains missing after adding 1?

1. 20
2. None / pd.NA
3. 22

Answer: 2. Arithmetic preserves the unknown value.

**Explain what you learned:** Use nullable types when missing values are legitimate.

### M09C04. Categories and an explicit order

A category stores values drawn from a defined set, such as programs or rating bands. An ordered category has a meaningful sequence that you choose. Sorting follows that sequence instead of alphabetical order.

**Read:** Where is the rating order defined?

Answer: In the category list ["Low", "Medium", "High"].

**Predict:** Which value appears first after sorting?

Answer: Low, followed by the other Low, then Medium and High.

**Explain:** Why might alphabetical sorting be wrong here?

Answer: Alphabetical sorting puts High first, which does not represent the intended rating order.



In [ ]:
# Predict before running: Which value appears first after sorting?
ratings = pd.Series(["High", "Low", "Medium", "Low"], dtype="string")
rating_type = pd.CategoricalDtype(["Low", "Medium", "High"], ordered=True)
result = ratings.astype(rating_type).sort_values()
print(result)
print("Order:", result.cat.categories.tolist())

**Your turn:** Use students(). Make Program an ordered category with IT before AI, then sort by Program and Student_ID. Store the sorted Student_ID and Program columns in the DataFrame result.

For the check, keep these result names: `result`.

**Hint:** Create a `pd.CategoricalDtype(["IT", "AI"], ordered=True)`.

**Watch for this:** A value outside a declared category list becomes missing when converted; check the result.

In [ ]:
# Change this code to complete the task.
# Question: Use students(). Make Program an ordered category with IT before AI, then sort by Program and Student_ID. Store the sorted Student_ID and Program columns in the DataFrame result.
df = students()
program_type = pd.CategoricalDtype(["IT", "AI"], ordered=True)
df["Program"] = df["Program"].astype(program_type)
result = df.sort_values(["Program", "Student_ID"])[["Student_ID", "Program"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Program"].iloc[:6].tolist() == ["IT"] * 6
assert result.iloc[0]["Student_ID"] == "S002"
print("Code checks passed. Review the written explanation as well.")

All six IT students appear before the six AI students; each program is ordered by ID.

**Quick check:** What does ordered=True add?

1. An explicit meaningful order
2. Automatic numeric grades
3. A rule that removes duplicates

Answer: 1. The category sequence becomes the ordering used by sorting and comparisons.

**Explain what you learned:** Define category meaning and order deliberately.

## Module 10. Missing values and honest cleaning

Find missing entries and choose a treatment that fits the question.

Before you start: You can convert columns and use Boolean masks.

Official reference: https://pandas.pydata.org/docs/user_guide/missing_data.html

### M10C01. Missing is not the same as empty or zero

`isna()` gives True where a value is missing. Zero is a recorded value, and an empty string is text unless you explicitly treat it as missing. Counting True values with `.sum()` counts missing entries.

**Read:** What type of values does `isna()` return?

Answer: True/False values, one for each entry.

**Predict:** How many missing entries will be counted?

Answer: Three: None, pd.NA, and np.nan.

**Explain:** Why is the empty string not automatically missing?

Answer: It is a stored text value. A cleaning rule must say whether blank text represents missing data.



In [ ]:
# Predict before running: How many missing entries will be counted?
values = pd.Series([0, None, "", pd.NA, np.nan], dtype="object")
result = values.isna()
print(result.tolist())
print("Missing count:", result.sum())

**Your turn:** Treat the empty string as missing, but keep zero. Store the cleaned Series in `result` and count missing entries.

For the check, keep these result names: `result`.

**Hint:** Use `.replace("", pd.NA)`.

**Watch for this:** Use `.isna()`, not `== np.nan`, to detect missing values.

In [ ]:
# Change this code to complete the task.
# Question: Treat the empty string as missing, but keep zero. Store the cleaned Series in `result` and count missing entries.
values = pd.Series([0, None, "", pd.NA, np.nan], dtype="object")
result = values.replace("", pd.NA)
print(result)
print("Missing count:", result.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.isna().sum() == 4
assert result.iloc[0] == 0
print("Code checks passed. Review the written explanation as well.")

Four values are now missing. The recorded zero is still zero.

**Quick check:** Which original value is known rather than missing?

1. None
2. 0
3. np.nan

Answer: 2. Zero is a numeric value; it should not disappear just because some data are missing.

**Explain what you learned:** Define what missing means in your dataset.

### M10C02. Drop rows only for required fields

`dropna(subset=[...])` removes rows missing the specified fields. It does not remove a row just because an unrelated field is missing. Keep a count of excluded records and consider how the exclusion affects the answer.

**Read:** Which column controls row removal in this cell?

Answer: Score only.

**Predict:** Will S006 remain even though Study_Hours is missing?

Answer: Yes. S006 has a score, so it passes the Score-only requirement.

**Explain:** Why not use `df.dropna()` without thinking?

Answer: It may remove rows for fields the analysis does not need, changing the sample unnecessarily.



In [ ]:
# Predict before running: Will S006 remain even though Study_Hours is missing?
df = raw_students()
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
result = df.dropna(subset=["Score"])
print(result[["Student_ID", "Score", "Study_Hours"]])
print("Rows kept:", len(result))

**Your turn:** Keep only rows with both Score and Study_Hours available. Leave the intentional duplicate for the next module.

For the check, keep these result names: `result`.

**Hint:** Pass both required column names to `subset`.

**Watch for this:** Removing incomplete records can change the composition of your data; report what was excluded.

In [ ]:
# Change this code to complete the task.
# Question: Keep only rows with both Score and Study_Hours available. Leave the intentional duplicate for the next module.
df = raw_students()
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
result = df.dropna(subset=["Score", "Study_Hours"])
print(result[["Student_ID", "Score", "Study_Hours"]])
print("Rows kept:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 10
assert not set(["S004", "S006", "S008"]) & set(result["Student_ID"])
print("Code checks passed. Review the written explanation as well.")

Ten rows remain out of thirteen. S004, S008, and S006 were excluded for the stated requirements.

**Quick check:** What does subset specify?

1. The fields that must be present
2. The first few rows to show
3. A replacement for missing values

Answer: 1. Only missing values in those columns trigger removal.

**Explain what you learned:** Match required fields to the analysis question.

### M10C03. Fill a label without inventing a measurement

`fillna()` replaces missing values with a value you specify. A label such as "Unknown" can make a missing category visible in a report. The same idea does not justify inventing a score or age.

**Read:** What is being filled here: a measured number or a category label?

Answer: A category label.

**Predict:** How many Unknown labels will appear?

Answer: Two.

**Explain:** Does Unknown tell us the actual campus?

Answer: No. It makes the missing information explicit; it does not recover the campus.



In [ ]:
# Predict before running: How many Unknown labels will appear?
campuses = pd.Series(["RAK", None, "Dubai", None], dtype="string")
result = campuses.fillna("Unknown")
print(result.tolist())

**Your turn:** Use the same Series, but first record a Boolean Was_missing flag. Store a DataFrame named result with the filled Campus column and the Was_missing flag.

For the check, keep these result names: `result`.

**Hint:** Call `.isna()` before replacing missing values.

**Watch for this:** Do not use zero as a universal replacement for missing numbers.

In [ ]:
# Change this code to complete the task.
# Question: Use the same Series, but first record a Boolean Was_missing flag. Store a DataFrame named result with the filled Campus column and the Was_missing flag.
campuses = pd.Series(["RAK", None, "Dubai", None], dtype="string")
result = pd.DataFrame({"Campus": campuses.fillna("Unknown"), "Was_missing": campuses.isna()})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Was_missing"].sum() == 2
assert (result.loc[result["Was_missing"], "Campus"] == "Unknown").all()
print("Code checks passed. Review the written explanation as well.")

The flag preserves which labels were originally missing, even after every cell contains text.

**Quick check:** Why retain a missingness flag?

1. To restore information about which entries were filled
2. To make every row identical
3. To prove the missing values were incorrect

Answer: 1. After filling, the flag distinguishes observed labels from replacements.

**Explain what you learned:** Keep missingness visible when replacements are used.

### M10C04. Compare a fill policy with keeping missing data

Preview of Module 16: the groupby, transform, and named aggregation syntax in this cell will be explained there. First focus on the difference between a technical replacement and a justified data policy. A group median is one possible imputation rule, not a universal answer. `transform()` returns a group result for every original row. This small example compares a proposed fill with the original values so its effect is visible.

**Read:** Why does transform return five values instead of two group summaries?

Answer: It repeats each group median on the rows belonging to that group.

**Predict:** Which values fill the missing AI and IT hours?

Answer: AI gets 6.0, the median of 4 and 8; IT gets 2.0, its only observed value.

**Explain:** Why is filling the IT value with 2.0 a weakly supported choice?

Answer: It is based on only one observed IT value and can hide uncertainty or reduce apparent variation.



In [ ]:
# Predict before running: Which values fill the missing AI and IT hours?
df = pd.DataFrame({"Program": ["AI", "AI", "AI", "IT", "IT"], "Hours": [4.0, 8.0, np.nan, 2.0, np.nan]})
group_median = df.groupby("Program")["Hours"].transform("median")
result = df.assign(Hours_filled=df["Hours"].fillna(group_median), Was_missing=df["Hours"].isna())
print(result)

**Your turn:** Use the same original table. Keep the missing values and store a summary in result with Program as its index. Name the known Hours count Known_hours, the row count Rows, and the median Median_hours.

For the check, keep these result names: `df`, `result`.

**Hint:** Use named aggregation; count counts known Hours, while size counts rows.

**Watch for this:** In predictive modeling, fit any imputation rule on training data only; do not learn it from held-out test data.

In [ ]:
# Change this code to complete the task.
# Question: Use the same original table. Keep the missing values and store a summary in result with Program as its index. Name the known Hours count Known_hours, the row count Rows, and the median Median_hours.
df = pd.DataFrame({"Program": ["AI", "AI", "AI", "IT", "IT"], "Hours": [4.0, 8.0, np.nan, 2.0, np.nan]})
result = df.groupby("Program").agg(Known_hours=("Hours", "count"), Rows=("Hours", "size"), Median_hours=("Hours", "median"))
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["AI", "Known_hours"] == 2
assert result.loc["IT", "Rows"] == 2
assert df["Hours"].isna().sum() == 2
print("Code checks passed. Review the written explanation as well.")

AI has 2 known values in 3 rows; IT has 1 in 2. The report shows the missing information rather than replacing it.

**Quick check:** Should every missing measurement be filled with a mean or median?

1. Yes, always
2. Only when the policy fits the context and its effects are checked
3. Only if the column is first converted to text

Answer: 2. Choose and document a treatment; retaining missing values or excluding specific rows may be more appropriate.

**Explain what you learned:** Explain the assumption behind every fill policy.

## Module 11. Duplicates and data quality

Find repeated records and validate IDs, ranges, and allowed labels.

Before you start: You can filter rows, convert values, and count missing entries.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html#duplicate-data

### M11C01. Find an exact repeated record

`duplicated()` marks repeated rows. By default, the first occurrence is kept unmarked and later copies are marked True. With `keep=False`, every member of a repeated set is marked.

**Read:** Why use keep=False when investigating duplicates?

Answer: It shows both the original row and its duplicate so we can compare them.

**Predict:** How many displayed rows and how many extra copies will we see?

Answer: Two displayed rows for S003, but only one extra copy.

**Explain:** Is a repeated name enough to identify a duplicate person?

Answer: No. Different people can share a name; use the intended record key and relevant fields.



In [ ]:
# Predict before running: How many displayed rows and how many extra copies will we see?
df = raw_students()
result = df.loc[df.duplicated(keep=False), ["Student_ID", "Name", "Score"]]
print(result)
print("Extra copies:", df.duplicated().sum())

**Your turn:** Remove exact duplicate rows and reset the index. Store the result in `result`.

For the check, keep these result names: `result`.

**Hint:** Use `.drop_duplicates().reset_index(drop=True)`.

**Watch for this:** A duplicate record and a missing value are different problems.

In [ ]:
# Change this code to complete the task.
# Question: Remove exact duplicate rows and reset the index. Store the result in `result`.
df = raw_students()
result = df.drop_duplicates().reset_index(drop=True)
print(result[["Student_ID", "Name", "Score"]])
print("Rows:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert not result.duplicated().any()
assert result.index.tolist() == list(range(12))
print("Code checks passed. Review the written explanation as well.")

Twelve rows remain. Other data-quality problems are still present and need separate treatment.

**Quick check:** What does duplicated() count by default here?

1. Every occurrence of a repeated row
2. Only repeated copies after the first
3. Rows with missing values

Answer: 2. The default keep="first" leaves the first copy unmarked.

**Explain what you learned:** Inspect repeated records before removing them.

### M11C02. A repeated ID is not always an exact duplicate

`subset` lets you check only the columns that define a record key. Two rows can share an ID but disagree on a score. Keeping the last one is defensible only when a documented order says it is the latest record.

**Read:** Which field is being treated as the record key?

Answer: Student_ID.

**Predict:** Are the two S001 rows exact duplicates?

Answer: No. Their dates and scores differ.

**Explain:** Why would dropping one row without a rule be risky?

Answer: We could discard the correct observation; the meaning of the records and update order matters.



In [ ]:
# Predict before running: Are the two S001 rows exact duplicates?
df = pd.DataFrame({"Student_ID": ["S001", "S001", "S002"], "Updated": ["2026-10-01", "2026-10-03", "2026-10-02"], "Score": [80, 88, 72]})
result = df.loc[df.duplicated(subset=["Student_ID"], keep=False)]
print(result)
print("Exact duplicates:", df.duplicated().sum())

**Your turn:** Assume the later Updated date is the authoritative revision. Parse Updated, sort chronologically, and keep the latest row per Student_ID. Store the retained rows in result, sorted by Student_ID in ascending order.

For the check, keep these result names: `result`.

**Hint:** Date parsing is introduced in Module 13: use pd.to_datetime with format="%Y-%m-%d". Sort by Updated before drop_duplicates(subset="Student_ID", keep="last"), then sort the retained rows by Student_ID.

**Watch for this:** In a transaction table, repeated student IDs may be expected; define what one row represents.

In [ ]:
# Change this code to complete the task.
# Question: Assume the later Updated date is the authoritative revision. Parse Updated, sort chronologically, and keep the latest row per Student_ID. Store the retained rows in result, sorted by Student_ID in ascending order.
df = pd.DataFrame({"Student_ID": ["S001", "S001", "S002"], "Updated": ["2026-10-01", "2026-10-03", "2026-10-02"], "Score": [80, 88, 72]})
df["Updated"] = pd.to_datetime(df["Updated"], format="%Y-%m-%d")
result = df.sort_values("Updated").drop_duplicates(subset="Student_ID", keep="last").sort_values("Student_ID")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Score"].tolist() == [88, 72]
assert result["Student_ID"].is_unique
print("Code checks passed. Review the written explanation as well.")

S001 keeps score 88 from October 3; S002 keeps 72. The decision follows the stated update policy.

**Quick check:** Does keep="last" automatically know the newest date?

1. Yes
2. No; the row order must first match the date rule
3. Only when the ID is numeric

Answer: 2. Last means last in the current row order.

**Explain what you learned:** Choose a key and a documented conflict rule.

### M11C03. Flag impossible values instead of hiding them

Data can have the right type and still be invalid. If Attendance is a percentage, its allowed range is 0 through 100. A Boolean mask can create a review table before you decide how to repair the source.

**Read:** What does the ~ symbol do to the range check?

Answer: It reverses True and False, so rows outside the range are selected.

**Predict:** Which record will be flagged?

Answer: S011, whose Attendance value is 105.

**Explain:** Why should we not automatically change 105 to 100?

Answer: We do not know the correct value. Clipping can hide a recording error and overstate attendance.



In [ ]:
# Predict before running: Which record will be flagged?
df = raw_students()
invalid = ~df["Attendance"].between(0, 100)
result = df.loc[invalid, ["Student_ID", "Attendance"]]
print(result)

**Your turn:** Preserve the source Attendance values in Attendance_raw. Convert Attendance to a nullable numeric column and set its out-of-range values to missing. Store the affected rows in result with columns Student_ID, Attendance_raw, and Attendance.

For the check, keep these result names: `result`.

**Hint:** Use a copy, `astype("Float64")`, and one `.loc[mask, column]` assignment.

**Watch for this:** The allowed range comes from the meaning of the variable, not from pandas.

In [ ]:
# Change this code to complete the task.
# Question: Preserve the source Attendance values in Attendance_raw. Convert Attendance to a nullable numeric column and set its out-of-range values to missing. Store the affected rows in result with columns Student_ID, Attendance_raw, and Attendance.
df = raw_students()
df["Attendance_raw"] = df["Attendance"]
df["Attendance"] = df["Attendance"].astype("Float64")
invalid = ~df["Attendance"].between(0, 100)
df.loc[invalid, "Attendance"] = pd.NA
result = df.loc[invalid, ["Student_ID", "Attendance_raw", "Attendance"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Attendance_raw"].iloc[0] == 105
assert pd.isna(result["Attendance"].iloc[0])
print("Code checks passed. Review the written explanation as well.")

The original 105 is retained for follow-up. The cleaned measurement is marked unknown, not changed to a guessed value.

**Quick check:** Is a numeric value always a valid measurement?

1. Yes
2. No, it must also satisfy the measurement rules
3. Only when it has a decimal point

Answer: 2. Type validation and range validation check different things.

**Explain what you learned:** Validate values against explicit domain rules.

### M11C04. Turn expectations into assertions

An assertion stops execution if an expectation is false. It is useful for classroom checks and analysis pipelines. Validate identity, required values, and ranges separately so a failure has a clear meaning.

**Read:** What happens if an assertion condition is False?

Answer: Python raises an AssertionError and stops the normal execution of the cell.

**Predict:** Will these checks pass for students()?

Answer: Yes. The clean teaching table has 12 unique nonmissing IDs and scores within the allowed range.

**Explain:** Why test missing IDs separately from uniqueness?

Answer: A column with one missing ID can still be unique; uniqueness alone does not require every ID to be present.



In [ ]:
# Predict before running: Will these checks pass for students()?
df = students()
assert df["Student_ID"].notna().all(), "A student ID is missing"
assert df["Student_ID"].is_unique, "Student IDs must be unique here"
assert df["Score"].between(0, 100).all(), "Score outside 0-100"
result = {"rows": len(df), "valid_ids": True, "valid_scores": True}
print(result)

**Your turn:** Add checks requiring Campus to be RAK or Dubai and Attendance to be 0-100. Store the two Boolean results in a dictionary named result with keys campus_ok and attendance_ok.

For the check, keep these result names: `result`.

**Hint:** Use `.isin([...]).all()` for an allowed-label check.

**Watch for this:** Python can disable assert statements in optimized runs; formal applications may need explicit validation errors.

In [ ]:
# Change this code to complete the task.
# Question: Add checks requiring Campus to be RAK or Dubai and Attendance to be 0-100. Store the two Boolean results in a dictionary named result with keys campus_ok and attendance_ok.
df = students()
campus_ok = df["Campus"].isin(["RAK", "Dubai"]).all()
attendance_ok = df["Attendance"].between(0, 100).all()
assert campus_ok, "Unexpected campus label"
assert attendance_ok, "Attendance outside 0-100"
result = {"campus_ok": bool(campus_ok), "attendance_ok": bool(attendance_ok)}
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == {"campus_ok": True, "attendance_ok": True}
print("Code checks passed. Review the written explanation as well.")

Both checks are True for the clean table. Applying equivalent checks to raw data would expose problems that need review.

**Quick check:** For a Boolean Series with no missing values, what does .all() require?

1. At least one True
2. Every value to be True
3. Exactly twelve rows

Answer: 2. With no missing values, every tested value must be True. By default, .all() skips missing values. Validate required non-missing values separately, or explicitly treat missing checks as False.

**Explain what you learned:** Write data expectations as checks you can repeat.

## Module 12. Clean and search text

Standardize labels and extract useful pieces from text columns.

Before you start: You can select columns and filter with Boolean masks.

Official reference: https://pandas.pydata.org/docs/user_guide/text.html

### M12C01. Clean spaces and preserve meaningful labels

The `.str` accessor applies text operations to each entry in a Series. `strip()` removes spaces at the beginning and end. A label map can then standardize spelling while preserving the exact labels you want.

**Read:** What does `.str.strip()` remove?

Answer: Leading and trailing whitespace from each text value.

**Predict:** What will S001 Name become?

Answer: Amira, without the surrounding spaces.

**Explain:** Will strip remove the space inside a two-word name?

Answer: No. Internal spaces are preserved.



In [ ]:
# Predict before running: What will S001 Name become?
df = raw_students()
df["Name"] = df["Name"].astype("string").str.strip()
result = df.loc[df["Student_ID"] == "S001", ["Student_ID", "Name"]]
print(result)

**Your turn:** Standardize all Campus labels as exactly RAK or Dubai. Handle case and surrounding spaces before mapping. Store Student_ID and the cleaned Campus for every source row in result.

For the check, keep these result names: `result`.

**Hint:** Use `.str.strip().str.lower()` followed by `.map({"rak": "RAK", "dubai": "Dubai"})`.

**Watch for this:** An unrecognized value passed through a dictionary map becomes missing; inspect unmatched labels.

In [ ]:
# Change this code to complete the task.
# Question: Standardize all Campus labels as exactly RAK or Dubai. Handle case and surrounding spaces before mapping. Store Student_ID and the cleaned Campus for every source row in result.
df = raw_students()
campus_text = df["Campus"].astype("string").str.strip().str.lower()
df["Campus"] = campus_text.map({"rak": "RAK", "dubai": "Dubai"})
result = df[["Student_ID", "Campus"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert set(result["Campus"]) == {"RAK", "Dubai"}
assert result.loc[1, "Campus"] == "Dubai"
assert result["Campus"].notna().all()
print("Code checks passed. Review the written explanation as well.")

S002 changes from dubai to Dubai. The abbreviation RAK remains uppercase.

**Quick check:** Would .str.title() alone preserve RAK exactly?

1. Yes
2. No, it would produce Rak
3. It would remove the whole value

Answer: 2. A known-label map is useful when abbreviations require special capitalization.

**Explain what you learned:** Standardize categories without destroying their meaning.

### M12C02. Literal text and regular expressions

`str.contains()` checks whether text contains a pattern. Use `regex=False` when you want the characters literally. With a regular expression, characters such as a dot can have special meanings.

**Read:** Why does the code set regex=False?

Answer: The dot should be an actual dot, not a pattern matching any character.

**Predict:** Which two filenames are selected?

Answer: report.csv and notes.csv.

**Explain:** What does na=False do?

Answer: It treats a missing text entry as not matching the search.



In [ ]:
# Predict before running: Which two filenames are selected?
files = pd.Series(["report.csv", "reportXcsv", "notes.csv", None, "csv.backup"], dtype="string")
mask = files.str.contains(".csv", regex=False, na=False)
result = files[mask]
print(result)

**Your turn:** Select all nonmissing filenames that begin with report. Use a regular expression and store the result.

For the check, keep these result names: `result`.

**Hint:** The ^ symbol anchors a regular expression to the start of the text.

**Watch for this:** Always be clear whether a pattern is literal text or a regular expression.

In [ ]:
# Change this code to complete the task.
# Question: Select all nonmissing filenames that begin with report. Use a regular expression and store the result.
files = pd.Series(["report.csv", "reportXcsv", "notes.csv", None, "csv.backup"], dtype="string")
mask = files.str.contains(r"^report", regex=True, na=False)
result = files[mask]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == ["report.csv", "reportXcsv"]
print("Code checks passed. Review the written explanation as well.")

Both report.csv and reportXcsv begin with report. Their extensions do not matter for this question.

**Quick check:** Which argument makes a search literal?

1. regex=False
2. na=False
3. ascending=False

Answer: 1. regex=False turns off regular-expression interpretation.

**Explain what you learned:** Choose text-search semantics deliberately.

### M12C03. Split a value into separate columns

`str.split()` separates text at a delimiter. `expand=True` creates separate columns rather than a list in each cell. Limiting splits can protect the rest of a string when a delimiter appears repeatedly.

**Read:** What character divides the email into two parts?

Answer: The @ character.

**Predict:** What will the first Username be?

Answer: amira.

**Explain:** Why use expand=True?

Answer: We want two table columns, so each piece can be selected and analyzed separately.



In [ ]:
# Predict before running: What will the first Username be?
emails = pd.Series(["amira@example.edu", "bilal@example.edu", None], dtype="string")
result = emails.str.split("@", n=1, expand=True, regex=False)
result.columns = ["Username", "Domain"]
print(result)

**Your turn:** Use the same emails. Extract Domain as a Series, then store the frequency of each known domain in result as a value_counts() Series.

For the check, keep these result names: `result`.

**Hint:** Select the second split piece with `.str[1]`, then call `.value_counts()`.

**Watch for this:** This is a string-processing exercise, not full validation of every legal email address.

In [ ]:
# Change this code to complete the task.
# Question: Use the same emails. Extract Domain as a Series, then store the frequency of each known domain in result as a value_counts() Series.
emails = pd.Series(["amira@example.edu", "bilal@example.edu", None], dtype="string")
domains = emails.str.split("@", n=1, regex=False).str[1]
result = domains.value_counts()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["example.edu"] == 2
assert len(result) == 1
print("Code checks passed. Review the written explanation as well.")

example.edu has two known occurrences. Missing domains are excluded by value_counts() by default.

**Quick check:** What object does expand=True produce here?

1. A two-column DataFrame
2. A single Python string
3. A number

Answer: 1. The split pieces become columns in a DataFrame.

**Explain what you learned:** Split text into fields before summarizing its parts.

### M12C04. Extract only values that match a structure

`str.extract()` uses capture groups to pull parts out of a matching pattern. Named groups become useful column names. A nonmatching string produces missing fields, which you should inspect.

**Read:** What do Program and Number name in the pattern?

Answer: The two captured fields, which become output columns.

**Predict:** What happens to the value bad?

Answer: Both extracted fields are missing because the whole pattern does not match.

**Explain:** Why are ^ and $ useful here?

Answer: They require the full string to match the structure, rather than accepting a matching piece inside other text.



In [ ]:
# Predict before running: What happens to the value bad?
codes = pd.Series(["AI-101", "IT-205", None, "bad"], dtype="string")
result = codes.str.extract(r"^(?P<Program>[A-Z]+)-(?P<Number>\d+)$")
print(result)

**Your turn:** Use the same codes. Extract the fields and convert Number to nullable Int64.

For the check, keep these result names: `result`.

**Hint:** Pass the extracted Number column through `pd.to_numeric(..., errors="coerce")`.

**Watch for this:** A successful pattern match checks structure, not whether the code exists in an official registry.

In [ ]:
# Change this code to complete the task.
# Question: Use the same codes. Extract the fields and convert Number to nullable Int64.
codes = pd.Series(["AI-101", "IT-205", None, "bad"], dtype="string")
result = codes.str.extract(r"^(?P<Program>[A-Z]+)-(?P<Number>\d+)$")
result["Number"] = pd.to_numeric(result["Number"], errors="coerce").astype("Int64")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert str(result["Number"].dtype) == "Int64"
assert result.loc[0, "Number"] == 101
assert result["Number"].isna().sum() == 2
print("Code checks passed. Review the written explanation as well.")

The valid numeric codes are 101 and 205. The missing and nonmatching source entries remain missing.

**Quick check:** Does extract automatically turn captured digits into integers?

1. Yes
2. No; convert the extracted text explicitly
3. Only when the group has a name

Answer: 2. Matching digits is different from choosing a numeric data type.

**Explain what you learned:** Extract structured pieces and then assign appropriate types.

## Module 13. Dates, times, and durations

Parse dates explicitly and use calendar-aware operations.

Before you start: You can convert types and select rows by conditions.

Official reference: https://pandas.pydata.org/docs/user_guide/timeseries.html

### M13C01. Turn date text into dates

A date-looking string is still text until it is parsed. `to_datetime()` converts it to a date/time type. An explicit format removes ambiguity; impossible dates can be marked as NaT, pandas missing date value.

**Read:** What do %Y, %m, and %d represent?

Answer: Four-digit year, numeric month, and day of month.

**Predict:** What happens to February 30?

Answer: It becomes NaT because it is not a valid calendar date.

**Explain:** Why might 01/02/2026 be ambiguous without a format?

Answer: It could mean January 2 or February 1, depending on the intended date convention.



In [ ]:
# Predict before running: What happens to February 30?
source = pd.Series(["2026-09-01", "2026-09-03", "2026-02-30"], dtype="string")
result = pd.to_datetime(source, format="%Y-%m-%d", errors="coerce")
print(result)
print("Unparsed dates:", result.isna().sum())

**Your turn:** Parse ["01/09/2026", "03/09/2026", "30/02/2026"] as day/month/year. Count invalid dates.

For the check, keep these result names: `result`.

**Hint:** Use the explicit format "%d/%m/%Y".

**Watch for this:** Inspect parsing failures; errors="coerce" does not repair the source date.

In [ ]:
# Change this code to complete the task.
# Question: Parse ["01/09/2026", "03/09/2026", "30/02/2026"] as day/month/year. Count invalid dates.
source = pd.Series(["01/09/2026", "03/09/2026", "30/02/2026"], dtype="string")
result = pd.to_datetime(source, format="%d/%m/%Y", errors="coerce")
print(result)
print("Unparsed dates:", result.isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.iloc[0] == pd.Timestamp("2026-09-01")
assert result.iloc[1].day == 3
assert result.isna().sum() == 1
print("Code checks passed. Review the written explanation as well.")

The first two dates are September 1 and September 3. February 30 remains invalid.

**Quick check:** What is NaT?

1. A missing datetime value
2. A date-format instruction
3. The current time

Answer: 1. NaT marks a missing or unparseable date/time value.

**Explain what you learned:** Parse known date formats explicitly.

### M13C02. Extract useful calendar parts

After parsing dates, the `.dt` accessor exposes calendar parts such as year, month, and weekday. These features are useful for grouping and checking patterns. Do not try to use `.dt` on unparsed strings.

**Read:** Why can .dt be used on Join_Date here?

Answer: The column has been converted from text into a datetime type.

**Predict:** Will all Month values be the same in this dataset?

Answer: Yes, all join dates are in September, so every Month is 9.

**Explain:** Why might a month number need a year too in a larger dataset?

Answer: September from different years would otherwise be grouped together.



In [ ]:
# Predict before running: Will all Month values be the same in this dataset?
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
result = df[["Student_ID", "Join_Date"]].copy()
result["Month"] = result["Join_Date"].dt.month
result["Weekday"] = result["Join_Date"].dt.day_name()
print(result.head(4))

**Your turn:** Add Year and Day columns for the same join dates. Store only the first three rows in result, with columns Student_ID, Year, Month, and Day, then print result.

For the check, keep these result names: `result`.

**Hint:** Use `.dt.year`, `.dt.month`, and `.dt.day`.

**Watch for this:** A weekday pattern in twelve fictional records is a practice result, not evidence about real student behavior.

In [ ]:
# Change this code to complete the task.
# Question: Add Year and Day columns for the same join dates. Store only the first three rows in result, with columns Student_ID, Year, Month, and Day, then print result.
df = students()
dates = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
result = pd.DataFrame({"Student_ID": df["Student_ID"], "Year": dates.dt.year, "Month": dates.dt.month, "Day": dates.dt.day}).head(3)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Day"].tolist() == [1, 2, 3]
assert result["Year"].eq(2026).all()
print("Code checks passed. Review the written explanation as well.")

The years are 2026, the months are 9, and the day values are 1, 2, and 3.

**Quick check:** Which accessor provides calendar components?

1. .str
2. .dt
3. .iloc

Answer: 2. .dt exposes datetime properties and methods.

**Explain what you learned:** Create date features after parsing the date type.

### M13C03. Filter an explicit date interval

Date comparisons let you select a time interval. `between()` includes both boundaries by default; naming the inclusive rule makes your intent clearer. Here every date is a midnight date with no separate time-of-day information.

**Read:** Are September 3 and September 6 included?

Answer: Yes; inclusive="both" includes both endpoints.

**Predict:** Which IDs appear?

Answer: S003, S004, S005, and S006.

**Explain:** Why do interval boundary rules matter?

Answer: An included or excluded endpoint changes which records contribute to the analysis.



In [ ]:
# Predict before running: Which IDs appear?
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
mask = df["Join_Date"].between("2026-09-03", "2026-09-06", inclusive="both")
result = df.loc[mask, ["Student_ID", "Join_Date"]]
print(result)

**Your turn:** Select dates on or after September 3 and strictly before September 6 using two comparisons.

For the check, keep these result names: `result`.

**Hint:** Combine >= and < with & and parentheses.

**Watch for this:** With timestamps containing hours, an end date at midnight is not the whole final day; a next-day exclusive bound is often clearer.

In [ ]:
# Change this code to complete the task.
# Question: Select dates on or after September 3 and strictly before September 6 using two comparisons.
df = students()
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
mask = (df["Join_Date"] >= "2026-09-03") & (df["Join_Date"] < "2026-09-06")
result = df.loc[mask, ["Student_ID", "Join_Date"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Student_ID"].tolist() == ["S003", "S004", "S005"]
print("Code checks passed. Review the written explanation as well.")

S003, S004, and S005 remain. The exclusive upper boundary removes S006.

**Quick check:** Which operator excludes the upper boundary?

1. <=
2. <
3. >=

Answer: 2. A strict less-than comparison excludes an equal endpoint.

**Explain what you learned:** State the start and end rules of every date filter.

### M13C04. Calculate elapsed time with a fixed reference

Subtracting dates produces timedeltas, which represent durations. `.dt.days` extracts whole days from those durations. A fixed reference date makes a classroom result reproducible when the file is opened later.

**Read:** What object is produced by reference - joined?

Answer: A Series of time durations, called timedeltas.

**Predict:** How many elapsed days does S001 have on October 1?

Answer: 30 days from September 1 to October 1.

**Explain:** Why use a fixed reference instead of today()?

Answer: Everyone can reproduce the same answer, regardless of when they run the lesson.



In [ ]:
# Predict before running: How many elapsed days does S001 have on October 1?
df = students()
joined = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
reference = pd.Timestamp("2026-10-01")
result = pd.DataFrame({"Student_ID": df["Student_ID"], "Days_enrolled": (reference - joined).dt.days})
print(result.head(4))

**Your turn:** Change the reference to October 8 and select students with at least 30 elapsed days enrolled.

For the check, keep these result names: `result`.

**Hint:** Recalculate the duration, then filter Days_enrolled >= 30.

**Watch for this:** For real timestamps, keep time zones consistent before subtracting them.

In [ ]:
# Change this code to complete the task.
# Question: Change the reference to October 8 and select students with at least 30 elapsed days enrolled.
df = students()
joined = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")
reference = pd.Timestamp("2026-10-08")
df["Days_enrolled"] = (reference - joined).dt.days
result = df.loc[df["Days_enrolled"] >= 30, ["Student_ID", "Days_enrolled"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 8
assert result.iloc[-1]["Student_ID"] == "S008"
assert result.iloc[-1]["Days_enrolled"] == 30
print("Code checks passed. Review the written explanation as well.")

S001 through S008 meet the rule. S008 has exactly 30 elapsed days.

**Quick check:** Does elapsed time count both the start and end date as full extra days?

1. Yes
2. No; it is the duration between the two dates
3. Only during September

Answer: 2. Date subtraction measures elapsed time. Inclusive calendar-day counts are a separate reporting rule.

**Explain what you learned:** Use timedelta arithmetic instead of subtracting date strings.

## Module 14. Map, apply, and vectorize

Choose an appropriate way to transform values and create categories.

Before you start: You can create columns, filter values, and define a short Python function.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#function-application

### M14C01. Map codes to meaningful labels

`Series.map()` can look up each value in a dictionary. A code missing from the dictionary produces a missing result. Keep the source codes so you can investigate labels that were not mapped.

**Read:** What do the keys of labels represent?

Answer: The codes we want to look up.

**Predict:** Which Label entries will be missing?

Answer: The entry for CS, which is not mapped, and the entry whose source code is missing.

**Explain:** Why should the unmapped CS and missing source code be investigated separately?

Answer: One is an observed but unrecognized code; the other was never recorded.



In [ ]:
# Predict before running: Which Label entries will be missing?
codes = pd.Series(["AI", "IT", "CS", None], dtype="string")
labels = {"AI": "Artificial Intelligence", "IT": "Information Technology"}
result = pd.DataFrame({"Code": codes, "Label": codes.map(labels)})
print(result)

**Your turn:** For this exercise, define CS as Cybersecurity. Add it to the mapping and rebuild the result while leaving the original missing code missing.

For the check, keep these result names: `result`.

**Hint:** Add one dictionary key-value pair; do not fill every missing label.

**Watch for this:** The meaning of a code must come from a stated codebook, not a guess.

In [ ]:
# Change this code to complete the task.
# Question: For this exercise, define CS as Cybersecurity. Add it to the mapping and rebuild the result while leaving the original missing code missing.
codes = pd.Series(["AI", "IT", "CS", None], dtype="string")
labels = {"AI": "Artificial Intelligence", "IT": "Information Technology", "CS": "Cybersecurity"}
result = pd.DataFrame({"Code": codes, "Label": codes.map(labels)})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Label"].isna().sum() == 1
assert result.loc[2, "Label"] == "Cybersecurity"
print("Code checks passed. Review the written explanation as well.")

The CS row now has its specified label; the row without a source code remains missing.

**Quick check:** What happens to an unmapped key in Series.map with a plain dictionary?

1. It is always copied unchanged
2. It produces a missing result
3. It becomes zero

Answer: 2. Unlike a general text replacement, a dictionary map returns missing for keys it does not contain.

**Explain what you learned:** Check what a lookup leaves unmatched.

### M14C02. Apply a function to each value

A Python function can express a custom rule for one value. `Series.apply()` calls that function for each entry. When the same rule can be written with built-in column operations, those operations are often simpler and more efficient.

**Read:** What does the function receive each time it is called?

Answer: One score value, not the whole Series.

**Predict:** Which label does a score of exactly 80 receive?

Answer: Secure, because the condition uses >=.

**Explain:** Is apply required for every calculation on a column?

Answer: No. Built-in arithmetic, comparisons, string methods, and aggregations often do the job directly.



In [ ]:
# Predict before running: Which label does a score of exactly 80 receive?
scores = pd.Series([58, 79, 80, 95])
def status(score):
    return "Secure" if score >= 80 else "Review"
result = scores.apply(status)
print(result.tolist())

**Your turn:** Create the same labels without a custom function. Use a Boolean comparison and a mapping from True/False to labels.

For the check, keep these result names: `result`.

**Hint:** Start with `scores.ge(80)`.

**Watch for this:** These labels use an exercise rule, not an official grading policy.

In [ ]:
# Change this code to complete the task.
# Question: Create the same labels without a custom function. Use a Boolean comparison and a mapping from True/False to labels.
scores = pd.Series([58, 79, 80, 95])
result = scores.ge(80).map({True: "Secure", False: "Review"})
print(result.tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == ["Review", "Review", "Secure", "Secure"]
print("Code checks passed. Review the written explanation as well.")

The labels are unchanged. The column comparison makes the threshold visible without a separate function.

**Quick check:** What does ge(80) mean?

1. Greater than or equal to 80
2. Exactly 80
3. Less than 80

Answer: 1. ge is the named form of the >= comparison.

**Explain what you learned:** Use a custom function when it adds clarity, not by default.

### M14C03. A row function versus column arithmetic

`DataFrame.apply(..., axis=1)` passes one row at a time to a function. Here Quiz is out of 20 and Project is out of 50, so both must be scaled before weighting them. A direct column formula can express the same calculation.

**Read:** Why is axis=1 used here?

Answer: The function needs Quiz and Project from the same row.

**Predict:** What is the first weighted percentage?

Answer: 80.0: 16/20 contributes 32 and 40/50 contributes 48.

**Explain:** Why would Quiz + Project be a different measure?

Answer: It would sum raw points without the stated 40% and 60% weights.



In [ ]:
# Predict before running: What is the first weighted percentage?
df = pd.DataFrame({"Quiz": [16, 12, 18], "Project": [40, 35, 45]})
result = df.apply(lambda row: (row["Quiz"] / 20) * 40 + (row["Project"] / 50) * 60, axis=1)
print(result.round(2).tolist())

**Your turn:** Compute the same weighted percentage with direct column arithmetic, without apply or lambda.

For the check, keep these result names: `df`, `result`.

**Hint:** Perform each calculation on the entire df["Quiz"] or df["Project"] Series.

**Watch for this:** Check the maximum points and weights before calculating a percentage.

In [ ]:
# Change this code to complete the task.
# Question: Compute the same weighted percentage with direct column arithmetic, without apply or lambda.
df = pd.DataFrame({"Quiz": [16, 12, 18], "Project": [40, 35, 45]})
result = (df["Quiz"] / 20) * 40 + (df["Project"] / 50) * 60
print(result.round(2).tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert np.allclose(result.to_numpy(), [80.0, 66.0, 90.0])
assert result.index.equals(df.index)
print("Code checks passed. Review the written explanation as well.")

The percentages remain 80.0, 66.0, and 90.0. Direct arithmetic also preserves the row index.

**Quick check:** With DataFrame.apply, axis=1 passes which object to the function?

1. A row
2. A file
3. The column names only

Answer: 1. axis=1 applies the function across the columns of each row.

**Explain what you learned:** Prefer direct column formulas for ordinary arithmetic.

### M14C04. Fixed intervals versus quantile groups

`cut()` groups numbers using value boundaries that you choose. `qcut()` uses sample quantiles to form groups with roughly similar counts. They answer different questions, and equal-count groups need not have equal numeric widths.

**Read:** Which method uses the explicit boundary of 50?

Answer: cut().

**Predict:** How many values are in the lower fixed band and lower quantile group?

Answer: Three values are in Lower; two are in Lower_half.

**Explain:** Why would quantile bands be unsuitable for an official fixed pass threshold?

Answer: Their boundaries depend on the current sample rather than a fixed scoring rule.



In [ ]:
# Predict before running: How many values are in the lower fixed band and lower quantile group?
values = pd.Series([10, 20, 30, 100])
fixed = pd.cut(values, bins=[0, 50, 100], labels=["Lower", "Upper"], include_lowest=True)
quantile = pd.qcut(values, q=2, labels=["Lower_half", "Upper_half"])
result = pd.DataFrame({"Value": values, "Fixed_band": fixed, "Quantile_band": quantile})
print(result)

**Your turn:** Use students(). Create Score_band using boundaries 0, 60, 80, 100 with labels Developing, Progressing, Secure. Include both 0 and 100, with 60 in Developing and 80 in Progressing. Store the count of each band in a Series named result.

For the check, keep these result names: `result`.

**Hint:** Use cut with right=True and include_lowest=True, then value_counts(sort=False).

**Watch for this:** Tied values can prevent qcut from making the requested number of distinct bins; inspect boundaries and group counts.

In [ ]:
# Change this code to complete the task.
# Question: Use students(). Create Score_band using boundaries 0, 60, 80, 100 with labels Developing, Progressing, Secure. Include both 0 and 100, with 60 in Developing and 80 in Progressing. Store the count of each band in a Series named result.
df = students()
df["Score_band"] = pd.cut(df["Score"], bins=[0, 60, 80, 100], labels=["Developing", "Progressing", "Secure"], right=True, include_lowest=True)
result = df["Score_band"].value_counts(sort=False)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_dict() == {"Developing": 1, "Progressing": 5, "Secure": 6}
print("Code checks passed. Review the written explanation as well.")

The counts are 1 Developing, 5 Progressing, and 6 Secure. These are practice labels with explicit boundary rules.

**Quick check:** Which method is appropriate for fixed score boundaries?

1. qcut
2. cut
3. sort_index

Answer: 2. cut accepts the exact value boundaries you need.

**Explain what you learned:** Choose bands by meaning or by sample distribution, as required.

## Module 15. Statistics you can explain

Summarize center, spread, completeness, and association with clear interpretations.

Before you start: You can select numeric columns and identify missing values.

Official reference: https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics

### M15C01. Mean and median respond differently

The mean uses every numeric value in an arithmetic average. The median is the middle value after sorting. Comparing them can help you notice how an unusually large or small observation affects a summary.

**Read:** What calculation produces the mean here?

Answer: (10 + 20 + 30 + 40 + 50) / 5 = 30.

**Predict:** What is the median?

Answer: 30, the middle value of the five sorted costs.

**Explain:** Does a mean equal to the median prove the data are correct?

Answer: No. It only describes this set of numbers; validation is a separate task.



In [ ]:
# Predict before running: What is the median?
costs = pd.Series([10, 20, 30, 40, 50])  # Fictional costs in AED
result = pd.Series({"Mean_AED": costs.mean(), "Median_AED": costs.median()})
print(result)

**Your turn:** Change the last cost from 50 to 500. Recalculate mean and median in result.

For the check, keep these result names: `result`.

**Hint:** Recreate the Series with the changed final value.

**Watch for this:** An extreme value is not automatically a mistake; investigate its source and meaning.

In [ ]:
# Change this code to complete the task.
# Question: Change the last cost from 50 to 500. Recalculate mean and median in result.
costs = pd.Series([10, 20, 30, 40, 500])
result = pd.Series({"Mean_AED": costs.mean(), "Median_AED": costs.median()})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Mean_AED"] == 120
assert result["Median_AED"] == 30
print("Code checks passed. Review the written explanation as well.")

The mean rises to AED 120, while the median stays AED 30. The large value has a stronger effect on the mean.

**Quick check:** Which summary stays 30 after this change?

1. Mean
2. Median
3. Maximum

Answer: 2. The middle sorted observation is still 30.

**Explain what you learned:** Explain which summary fits the question and distribution.

### M15C02. Spread and the standard-deviation convention

Standard deviation measures spread around the mean in the same units as the values. pandas uses sample standard deviation by default, with ddof=1. ddof=0 uses the population-style denominator; choose based on what the data represent.

**Read:** What does ddof change?

Answer: It changes the denominator from n to n minus ddof in the variance calculation.

**Predict:** Which standard deviation is larger for these data?

Answer: The sample value, 2.0, is larger than the population-style value, about 1.633.

**Explain:** Would changing hours to minutes leave the standard deviation numerically unchanged?

Answer: No. The standard deviation would also be multiplied by 60 because it has the same units.



In [ ]:
# Predict before running: Which standard deviation is larger for these data?
hours = pd.Series([2.0, 4.0, 6.0])
result = pd.Series({"Mean_hours": hours.mean(), "Sample_std_hours": hours.std(ddof=1), "Population_std_hours": hours.std(ddof=0)})
print(result.round(4))

**Your turn:** Replace the hours with [4.0, 4.0, 4.0]. Recalculate both standard deviations.

For the check, keep these result names: `result`.

**Hint:** A set of identical values has no spread around its mean.

**Watch for this:** With only one known value, sample std is missing because n - 1 is zero.

In [ ]:
# Change this code to complete the task.
# Question: Replace the hours with [4.0, 4.0, 4.0]. Recalculate both standard deviations.
hours = pd.Series([4.0, 4.0, 4.0])
result = pd.Series({"Sample_std_hours": hours.std(ddof=1), "Population_std_hours": hours.std(ddof=0)})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Sample_std_hours"] == 0
assert result["Population_std_hours"] == 0
print("Code checks passed. Review the written explanation as well.")

Both standard deviations are zero because every value equals the mean.

**Quick check:** Which ddof does pandas std() use by default?

1. 0
2. 1
3. The number of rows

Answer: 2. The default is ddof=1, the sample standard-deviation convention.

**Explain what you learned:** State the spread measure and its convention.

### M15C03. Describe, count, and quantiles

`describe()` summarizes a numeric Series with count, mean, spread, and selected percentiles. Count excludes missing values; size counts all entries. A quantile describes a position in the distribution, and its value may be interpolated between observations.

**Read:** Does describe count the missing score as an observed score?

Answer: No. Its count is 4, while the Series has 5 entries.

**Predict:** What is the 50% value?

Answer: 75.0, halfway between the two middle known values, 70 and 80.

**Explain:** Why does a reported average need a known-value count?

Answer: Readers need to know how many observations support the average and how much information is missing.



In [ ]:
# Predict before running: What is the 50% value?
scores = pd.Series([60.0, 70.0, np.nan, 80.0, 90.0])
result = scores.describe()
print(result)
print("Rows:", scores.size, "Known:", scores.count())

**Your turn:** For the same scores, report the 25th, 50th, and 75th percentiles explicitly using quantile.

For the check, keep these result names: `result`.

**Hint:** Pass a list of probabilities [0.25, 0.5, 0.75].

**Watch for this:** Describe is a starting point. It does not check whether IDs, units, ranges, or collection procedures are valid.

In [ ]:
# Change this code to complete the task.
# Question: For the same scores, report the 25th, 50th, and 75th percentiles explicitly using quantile.
scores = pd.Series([60.0, 70.0, np.nan, 80.0, 90.0])
result = scores.quantile([0.25, 0.5, 0.75])
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_dict() == {0.25: 67.5, 0.5: 75.0, 0.75: 82.5}
print("Code checks passed. Review the written explanation as well.")

With the default linear interpolation, the quartiles are 67.5, 75.0, and 82.5.

**Quick check:** Which quantity includes a missing entry?

1. Series.count()
2. Series.size
3. The count row of describe()

Answer: 2. size counts entries; count counts nonmissing values.

**Explain what you learned:** Report completeness alongside summary statistics.

### M15C04. Correlation describes association

`corr()` computes pairwise correlations between numeric variables; its default method is Pearson correlation. Positive values indicate that higher values tend to occur together in a linear pattern. Correlation does not establish that one variable causes the other.

**Read:** Why are only three columns selected before corr()?

Answer: They are numeric measurements relevant to this question; IDs and labels do not belong in this calculation.

**Predict:** What values appear on the diagonal?

Answer: 1.0 for each of these nonconstant columns correlated with itself.

**Explain:** Does a positive Study_Hours-Score correlation prove that studying longer caused these scores?

Answer: No. Other factors and the way the data were collected could explain the association. These records are also fictional teaching data.



In [ ]:
# Predict before running: What values appear on the diagonal?
df = students()
result = df[["Study_Hours", "Score", "Attendance"]].corr(method="pearson")
print(result.round(3))

**Your turn:** Calculate Spearman rank correlation for Study_Hours and Score. Store the two-by-two matrix in result.

For the check, keep these result names: `result`.

**Hint:** Change method to "spearman" after selecting the two columns.

**Watch for this:** Missing entries are handled pairwise, so different correlations can use different numbers of observations.

In [ ]:
# Change this code to complete the task.
# Question: Calculate Spearman rank correlation for Study_Hours and Score. Store the two-by-two matrix in result.
df = students()
result = df[["Study_Hours", "Score"]].corr(method="spearman")
print(result.round(3))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 2)
assert 0 < result.loc["Study_Hours", "Score"] <= 1
assert np.isclose(result.loc["Study_Hours", "Score"], result.loc["Score", "Study_Hours"])
print("Code checks passed. Review the written explanation as well.")

Spearman compares ranks rather than raw values. The association is positive in this practice table; it still does not support a causal claim.

**Quick check:** What claim does correlation alone justify?

1. A causal effect has been proved
2. An association was measured in the available data
3. The model will predict perfectly

Answer: 2. Association is the appropriate interpretation without a design supporting causal inference.

**Explain what you learned:** Interpret a correlation as an association with limitations.

## Module 16. Group, summarize, and compare

Distinguish group summaries, row-aligned transformations, and group filters.

Before you start: You can select columns, use summary methods, and create new columns.

Official reference: https://pandas.pydata.org/docs/user_guide/groupby.html

### M16C01. One question for each group

`groupby()` splits rows according to a label, then a summary method calculates one result per group. Named aggregation lets you choose readable output names. Here each campus receives its own row count and mean score.

**Read:** What does each row of the result represent?

Answer: One campus, with its number of student records and average score.

**Predict:** How many rows should result contain?

Answer: Two, one for Dubai and one for RAK.

**Explain:** Why use as_index=False?

Answer: It keeps Campus as an ordinary column rather than placing it in the result index.



In [ ]:
# Predict before running: How many rows should result contain?
df = students()
result = df.groupby("Campus", as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))

**Your turn:** Group by Program instead. Store a DataFrame named result with Program as a normal column and named summary columns Students, Median_score, and Highest_score.

For the check, keep these result names: `result`.

**Hint:** Use named aggregations with size, median, and max.

**Watch for this:** A mean without its supporting count can hide a very small group.

In [ ]:
# Change this code to complete the task.
# Question: Group by Program instead. Store a DataFrame named result with Program as a normal column and named summary columns Students, Median_score, and Highest_score.
df = students()
result = df.groupby("Program", as_index=False).agg(Students=("Student_ID", "size"), Median_score=("Score", "median"), Highest_score=("Score", "max"))
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.set_index("Program").loc["AI", "Median_score"] == 88.5
assert result.set_index("Program").loc["IT", "Highest_score"] == 82
assert result["Students"].sum() == 12
print("Code checks passed. Review the written explanation as well.")

Both programs have six students. AI has median 88.5 and maximum 95; IT has median 74.0 and maximum 82.

**Quick check:** How many values does an ordinary mean aggregation return for one group?

1. One summary value
2. One value for every original row
3. One value for every cell

Answer: 1. An aggregation reduces the group to a summary.

**Explain what you learned:** Summarize each group with readable labels and counts.

### M16C02. Multiple keys and missing group labels

Grouping by a list of columns forms one group for each observed combination. By default, rows with a missing group key are excluded. `dropna=False` keeps those rows in a separate missing-label group so the totals can be audited.

**Read:** What does one result row represent now?

Answer: One Campus-Program combination, including a combination whose Campus is missing.

**Predict:** How many student records are represented in total?

Answer: All 12, because dropna=False retains the missing-campus record.

**Explain:** What would the default dropna=True do to S001?

Answer: It would exclude S001 from this grouped result because Campus is missing.



In [ ]:
# Predict before running: How many student records are represented in total?
df = students()
df.loc[0, "Campus"] = None
result = df.groupby(["Campus", "Program"], dropna=False, as_index=False).agg(Students=("Student_ID", "size"), Mean_score=("Score", "mean"))
print(result.round(2))
print("Total represented:", result["Students"].sum())

**Your turn:** Run the same grouping with dropna=True. Compare its represented row total with len(df), and store the grouped table in result.

For the check, keep these result names: `df`, `result`.

**Hint:** Keep the original table; add the group sizes and compare the sum.

**Watch for this:** When group keys are categorical, set observed explicitly if you need predictable inclusion of unused categories.

In [ ]:
# Change this code to complete the task.
# Question: Run the same grouping with dropna=True. Compare its represented row total with len(df), and store the grouped table in result.
df = students()
df.loc[0, "Campus"] = None
result = df.groupby(["Campus", "Program"], dropna=True, as_index=False).agg(Students=("Student_ID", "size"))
print(result)
print("Represented:", result["Students"].sum(), "Original:", len(df))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Students"].sum() == 11
assert len(df) == 12
print("Code checks passed. Review the written explanation as well.")

The grouped counts sum to 11, while the original table still has 12 rows. S001 is excluded only from the grouping output.

**Quick check:** Which option retains rows whose grouping key is missing?

1. dropna=False
2. dropna=True
3. ascending=False

Answer: 1. dropna=False creates a missing-key group.

**Explain what you learned:** Reconcile grouped totals with the original table.

### M16C03. A summary for every original row

`transform()` returns values aligned with the original rows. A group mean can therefore be attached to every student in that group. This makes comparisons with a group benchmark possible without reducing the table to one row per group.

**Read:** How many rows does result keep?

Answer: All 12 original student rows.

**Predict:** Will all AI students have the same Program_mean?

Answer: Yes. The same AI group mean is aligned with each AI student.

**Explain:** Why would assigning a two-row group mean Series directly be risky?

Answer: Pandas aligns by index labels, which may not match the original row index. transform is designed to return a row-aligned result.



In [ ]:
# Predict before running: Will all AI students have the same Program_mean?
df = students()
df["Program_mean"] = df.groupby("Program")["Score"].transform("mean")
df["Difference"] = df["Score"] - df["Program_mean"]
result = df[["Student_ID", "Program", "Score", "Program_mean", "Difference"]]
print(result.round(2))

**Your turn:** Calculate the median score within each campus in a column named Campus_median. Keep only students scoring strictly above that median. Store Student_ID, Campus, Score, and Campus_median in result.

For the check, keep these result names: `result`.

**Hint:** Use groupby("Campus")["Score"].transform("median") to build the benchmark.

**Watch for this:** A group-relative label and a fixed-threshold label answer different questions.

In [ ]:
# Change this code to complete the task.
# Question: Calculate the median score within each campus in a column named Campus_median. Keep only students scoring strictly above that median. Store Student_ID, Campus, Score, and Campus_median in result.
df = students()
df["Campus_median"] = df.groupby("Campus")["Score"].transform("median")
result = df.loc[df["Score"] > df["Campus_median"], ["Student_ID", "Campus", "Score", "Campus_median"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Student_ID"].tolist() == ["S001", "S003", "S006", "S007", "S010", "S012"]
assert (result["Score"] > result["Campus_median"]).all()
print("Code checks passed. Review the written explanation as well.")

Six students are above their campus median: S001, S003, S006, S007, S010, and S012.

**Quick check:** Which operation returns a group statistic aligned to every original row?

1. agg
2. transform
3. drop_duplicates

Answer: 2. transform preserves the original row alignment while applying group-specific calculations.

**Explain what you learned:** Use transform when the result belongs beside the original records.

### M16C04. Keep whole groups and control unused categories

A group filter keeps or removes whole groups based on a group-level rule. Categorical columns can also contain labels with no rows. `observed=True` limits a group result to categories actually present in the data.

**Read:** Does declaring CS as a category create CS student records?

Answer: No. It declares a possible label without adding rows.

**Predict:** Will CS appear in this grouped result?

Answer: No, because observed=True reports only categories that occur in the data.

**Explain:** How is filtering whole groups different from filtering scores >= 80?

Answer: A group filter keeps all rows in an accepted group, including individual scores below 80.



In [ ]:
# Predict before running: Will CS appear in this grouped result?
df = students()
df["Program"] = pd.Categorical(df["Program"], categories=["AI", "IT", "CS"])
result = df.groupby("Program", observed=True)["Score"].agg(["size", "mean"])
print(result.round(2))

**Your turn:** Start from students(). Keep all students in programs whose mean Score is at least 80 using groupby().filter().

For the check, keep these result names: `result`.

**Hint:** The filter function receives each group DataFrame; compare its Score mean with 80.

**Watch for this:** Unused categories and missing labels are different: observed controls unused categories; dropna controls missing keys.

In [ ]:
# Change this code to complete the task.
# Question: Start from students(). Keep all students in programs whose mean Score is at least 80 using groupby().filter().
df = students()
result = df.groupby("Program").filter(lambda group: group["Score"].mean() >= 80)
print(result[["Student_ID", "Program", "Score"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 6
assert set(result["Program"]) == {"AI"}
assert "S011" in result["Student_ID"].tolist()
print("Code checks passed. Review the written explanation as well.")

All six AI students remain, including S011 with score 68. The rule concerns the program mean, not each individual score.

**Quick check:** A program passes the group-mean rule. Which records are kept by groupby().filter()?

1. Only records above the threshold
2. All records in that program
3. Only the highest-scoring record

Answer: 2. A group filter accepts or rejects the group as a whole.

**Explain what you learned:** Separate group inclusion rules from row inclusion rules.

## Module 17. Merge related tables

Connect tables by a key and audit which records matched.

Before you start: Use column selection and understand unique Student_ID values.

Official reference: https://pandas.pydata.org/docs/user_guide/merging.html

### M17C01. Attach the department

The Program column is a key shared by two tables. A many-to-one merge lets several students match one department row. The validation checks that the lookup table has only one row per program.

**Read:** What does on="Program" tell pandas?

Answer: Match the two tables using Program values.

**Predict:** Will the result contain 2 rows or 12 rows?

Answer: 12 rows, one for each student, because each Program matches exactly one lookup row.

**Explain:** Why validate="many_to_one"?

Answer: Students may repeat a Program, but each Program must appear at most once in the lookup table.



In [ ]:
# Predict before running: Will the result contain 2 rows or 12 rows?
df = students()
result = df.merge(departments(), on="Program", how="left", validate="many_to_one")
print(result[["Student_ID", "Program", "Department"]])

**Your turn:** Use DataFrame.join to attach the same lookup fields. Set Program as the lookup index and keep the student index.

For the check, keep these result names: `result`.

**Hint:** Use departments().set_index("Program"), then join(..., on="Program").

**Watch for this:** Do not join tables by row position when they describe entities identified by keys.

In [ ]:
# Change this code to complete the task.
# Question: Use DataFrame.join to attach the same lookup fields. Set Program as the lookup index and keep the student index.
df = students()
lookup = departments().set_index("Program")
result = df.join(lookup, on="Program", how="left", validate="many_to_one")
print(result[["Student_ID", "Program", "Department"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert result["Advisor"].notna().all()
print("Code checks passed. Review the written explanation as well.")

join can use the right table index as the key. Both forms preserve all 12 students here.

**Quick check:** Which side must have unique Program values?

1. The student table
2. The department lookup
3. Both tables

Answer: 2. The lookup side must be unique for many-to-one matching.

**Explain what you learned:** Choose and validate the matching key.

### M17C02. Keep matches or keep every student

An inner merge keeps only keys found in both tables. A left merge keeps every key from the left table. In this complete assessment lookup, a missing Quiz comes from an unmatched student. With other data, a matched assessment could itself have a missing Quiz; use the merge indicator to test for unmatched records.

**Read:** Which table supplies Quiz?

Answer: The assessment table.

**Predict:** Which registered student disappears from the inner merge?

Answer: S012, Lina, because the assessment table has no S012 row.

**Explain:** Does an unmatched quiz mean the student earned zero?

Answer: No. A missing match is unknown and needs investigation.



In [ ]:
# Predict before running: Which registered student disappears from the inner merge?
result = students().merge(assessment(), on="Student_ID", how="inner", validate="one_to_one")
print(result[["Student_ID", "Name", "Quiz"]])
print("Rows:", len(result))

**Your turn:** Change the merge to left and store the full merged table in merged. Select only students whose Quiz is missing into result, keeping Student_ID, Name, and Quiz, then print result.

For the check, keep these result names: `merged`, `result`.

**Hint:** Use how="left" and then Quiz.isna().

**Watch for this:** Filling an unmatched assessment with zero changes its meaning; agree on a policy first.

In [ ]:
# Change this code to complete the task.
# Question: Change the merge to left and store the full merged table in merged. Select only students whose Quiz is missing into result, keeping Student_ID, Name, and Quiz, then print result.
merged = students().merge(assessment(), on="Student_ID", how="left", validate="one_to_one")
result = merged.loc[merged["Quiz"].isna(), ["Student_ID", "Name", "Quiz"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(merged) == 12
assert result["Student_ID"].tolist() == ["S012"]
print("Code checks passed. Review the written explanation as well.")

The left merge retains 12 students; the missing-match report contains only S012.

**Quick check:** Which merge preserves every registered student?

1. inner
2. left, with students on the left
3. right, with assessments on the right

Answer: 2. A left merge preserves the left table keys even when there is no right-side match.

**Explain what you learned:** Merge type determines which records remain.

### M17C03. Audit both unmatched sides

An outer merge keeps keys from either table. indicator=True adds a _merge column showing left_only, right_only, or both. This is a simple way to find records that need review.

**Read:** What does right_only mean here?

Answer: The key exists in assessments but not in the student table.

**Predict:** Which IDs will the audit show?

Answer: S012 is left_only; S013 is right_only.

**Explain:** Why inspect both unmatched sides?

Answer: One side can reveal missing assessments; the other can reveal assessments for an unregistered or mistyped ID.



In [ ]:
# Predict before running: Which IDs will the audit show?
audit = students().merge(assessment(), on="Student_ID", how="outer", indicator=True, validate="one_to_one")
result = audit.loc[audit["_merge"].ne("both"), ["Student_ID", "_merge"]]
print(result)
print("All merged rows:", len(audit))

**Your turn:** Use a right merge instead. Keep indicator=True and report the row that has no registered student name.

For the check, keep these result names: `audit`, `result`.

**Hint:** A right merge preserves the assessment keys; inspect Name.isna().

**Watch for this:** Pandas may match missing keys to other missing keys; audit or reject null keys before important merges.

In [ ]:
# Change this code to complete the task.
# Question: Use a right merge instead. Keep indicator=True and report the row that has no registered student name.
audit = students().merge(assessment(), on="Student_ID", how="right", indicator=True, validate="one_to_one")
result = audit.loc[audit["Name"].isna(), ["Student_ID", "_merge"]]
print(result)
print("All merged rows:", len(audit))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(audit) == 12
assert result["Student_ID"].tolist() == ["S013"]
print("Code checks passed. Review the written explanation as well.")

The right merge keeps 12 assessment records. S013 remains even though no student profile matches.

**Quick check:** How many distinct keys does the outer merge contain?

1. 11
2. 12
3. 13

Answer: 3. The union contains S001 through S013: 11 shared keys plus 2 unmatched keys.

**Explain what you learned:** Use indicator to make unmatched records visible.

### M17C04. Stop duplicate-key expansion

If a lookup repeats a key, one input row can produce several output rows. The code deliberately adds a duplicate AI lookup. Catching MergeError lets us demonstrate the problem without stopping the lesson.

**Read:** Why is try/except used?

Answer: To catch the expected validation error and continue the lesson.

**Predict:** How many rows appear without validation?

Answer: 18: each of the 6 AI students matches twice, and the 6 IT students match once.

**Explain:** Should every duplicate key be dropped automatically?

Answer: No. Repeated keys may describe different facts. This example repeats an identical lookup row, so removing that exact duplicate is justified.



In [ ]:
# Predict before running: How many rows appear without validation?
bad_lookup = pd.concat([departments(), departments().iloc[[0]]], ignore_index=True)
blocked = False
try:
    students().merge(bad_lookup, on="Program", validate="many_to_one")
except pd.errors.MergeError:
    blocked = True
    print("Blocked: Program is not unique in the lookup.")
result = students().merge(bad_lookup, on="Program")
print("Rows without validation:", len(result))

**Your turn:** Remove only exact duplicate lookup rows, confirm Program is unique, and perform the validated merge.

For the check, keep these result names: `result`.

**Hint:** drop_duplicates() removes exact repeats; is_unique checks the key.

**Watch for this:** validate checks the relationship; it does not decide how to repair conflicting records.

In [ ]:
# Change this code to complete the task.
# Question: Remove only exact duplicate lookup rows, confirm Program is unique, and perform the validated merge.
bad_lookup = pd.concat([departments(), departments().iloc[[0]]], ignore_index=True)
lookup = bad_lookup.drop_duplicates()
assert lookup["Program"].is_unique
result = students().merge(lookup, on="Program", validate="many_to_one")
print("Validated rows:", len(result))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert result["Student_ID"].is_unique
print("Code checks passed. Review the written explanation as well.")

The duplicated lookup was identical, so removing the exact repeat restores 12 rows.

**Quick check:** What can repeated keys on both sides cause?

1. A multiplication of matching rows
2. Automatic averaging
3. Automatic removal of duplicates

Answer: 1. Each matching pair produces a row, so duplicate keys can expand the result.

**Explain what you learned:** Check key uniqueness before trusting merged row counts.

## Module 18. Concatenate and align

Combine rows or columns while respecting index labels.

Before you start: Understand DataFrame indexes and missing values.

Official reference: https://pandas.pydata.org/docs/user_guide/merging.html#concat

### M18C01. Stack row batches

concat stacks tables when axis=0, the default. ignore_index=True assigns new position labels. This changes the index, not the Student_ID values.

**Read:** What is the list inside concat?

Answer: It contains the two DataFrames to combine.

**Predict:** How many student rows will result have?

Answer: 12, the 6 rows from each batch.

**Explain:** Does ignore_index remove duplicate students?

Answer: No. It only creates a fresh integer index; duplicate data would still remain.



In [ ]:
# Predict before running: How many student rows will result have?
df = students()
first = df.iloc[:6]
second = df.iloc[6:]
result = pd.concat([first, second], ignore_index=True)
print(result[["Student_ID", "Name"]])

**Your turn:** Stack the second batch before the first, and reset the index in the combined result.

For the check, keep these result names: `result`.

**Hint:** Reverse the order of the two DataFrames passed to concat.

**Watch for this:** Build a list of batches and concatenate once instead of repeatedly growing a DataFrame in a loop.

In [ ]:
# Change this code to complete the task.
# Question: Stack the second batch before the first, and reset the index in the combined result.
df = students()
result = pd.concat([df.iloc[6:], df.iloc[:6]], ignore_index=True)
print(result[["Student_ID", "Name"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.iloc[0]["Student_ID"] == "S007"
assert result.index.tolist() == list(range(12))
print("Code checks passed. Review the written explanation as well.")

S007 is first because concat keeps the order of the input batches.

**Quick check:** What changes with ignore_index=True?

1. The Student_ID values
2. The output row index
3. The column names

Answer: 2. A fresh range index is assigned; data values stay unchanged.

**Explain what you learned:** Use concat for batches with the same meaning and schema.

### M18C02. Align columns by labels

concat with axis=1 places objects side by side. It matches index labels, even when their order differs. This is why an ID index is safer than assuming row positions match.

**Read:** Why are the two Series given names?

Answer: Their names become output column labels.

**Predict:** Which score will Amira receive?

Answer: 88, because both values use index label S001.

**Explain:** Why is assigning scores.to_numpy() risky here?

Answer: It discards labels, so the first score, 72, would be assigned by position to Amira.



In [ ]:
# Predict before running: Which score will Amira receive?
names = pd.Series(["Amira", "Bilal"], index=["S001", "S002"], name="Name")
scores = pd.Series([72, 88], index=["S002", "S001"], name="Score")
result = pd.concat([names, scores], axis=1)
print(result)

**Your turn:** Add score 91 for S003, then concatenate using join="inner" so only shared IDs remain.

For the check, keep these result names: `result`.

**Hint:** Construct a three-value scores Series and set join="inner".

**Watch for this:** Resetting indexes before concatenating unrelated tables can hide a mismatched-key problem.

In [ ]:
# Change this code to complete the task.
# Question: Add score 91 for S003, then concatenate using join="inner" so only shared IDs remain.
names = pd.Series(["Amira", "Bilal"], index=["S001", "S002"], name="Name")
scores = pd.Series([72, 88, 91], index=["S002", "S001", "S003"], name="Score")
result = pd.concat([names, scores], axis=1, join="inner")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert set(result.index) == {"S001", "S002"}
assert result.loc["S001", "Score"] == 88
print("Code checks passed. Review the written explanation as well.")

S003 is excluded because its label is absent from names.

**Quick check:** What does axis=1 concatenate side by side?

1. Columns aligned on row labels
2. Rows matched by physical position
3. Only numeric values

Answer: 1. Column concatenation aligns the row indexes.

**Explain what you learned:** Alignment follows labels, not visual row order.

### M18C03. Request a specific index order

reindex requests specific labels in a specific order. A requested label that does not exist produces missing values. It can also omit existing labels that were not requested.

**Read:** What controls the row order?

Answer: The label list supplied to reindex.

**Predict:** What will appear for S099?

Answer: Missing Name and Score values because the ID does not exist.

**Explain:** How is this different from loc with a missing label?

Answer: With a list of IDs, loc raises KeyError if any requested ID is absent. reindex can create rows with missing values for absent labels. Label slices have different endpoint rules.



In [ ]:
# Predict before running: What will appear for S099?
df = students().set_index("Student_ID")
result = df.reindex(["S003", "S001", "S099"])[["Name", "Score"]]
print(result)

**Your turn:** Reindex to S012, S002, S001 and print the Name and Score columns in that order.

For the check, keep these result names: `result`.

**Hint:** Supply a different list of labels to reindex.

**Watch for this:** Reindexing generally requires unique labels on the axis being reindexed.

In [ ]:
# Change this code to complete the task.
# Question: Reindex to S012, S002, S001 and print the Name and Score columns in that order.
df = students().set_index("Student_ID")
result = df.reindex(["S012", "S002", "S001"])[["Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ["S012", "S002", "S001"]
assert result.notna().all().all()
print("Code checks passed. Review the written explanation as well.")

All three labels exist, so this result has no missing values.

**Quick check:** What does an absent label create during reindex?

1. A KeyError in every case
2. A row of missing values
3. A copy of the preceding row

Answer: 2. reindex creates missing entries for labels not present in the original unique index.

**Explain what you learned:** Reindex is an explicit request for labels and order.

### M18C04. Fill from a second source

combine_first fills missing values from matching labels in another object. Existing non-missing values in the first source win. Agree which source has priority before applying this rule.

**Read:** Which source has priority?

Answer: primary, because combine_first is called on it.

**Predict:** Will S001 become 85?

Answer: No. Its primary score 88 is already present.

**Explain:** Why is this different from an unconditional overwrite?

Answer: Only missing primary entries are filled; existing values remain.



In [ ]:
# Predict before running: Will S001 become 85?
primary = pd.Series([88, pd.NA, 91], index=["S001", "S002", "S003"], dtype="Int64")
backup = pd.Series([85, 72, 90], index=["S001", "S002", "S003"], dtype="Int64")
result = primary.combine_first(backup)
print(result)

**Your turn:** Reverse the source priority by calling combine_first on backup. Explain the S001 result.

For the check, keep these result names: `result`.

**Hint:** Place backup before the dot.

**Watch for this:** An unmatched backup label can expand the index; inspect the result size.

In [ ]:
# Change this code to complete the task.
# Question: Reverse the source priority by calling combine_first on backup. Explain the S001 result.
primary = pd.Series([88, pd.NA, 91], index=["S001", "S002", "S003"], dtype="Int64")
backup = pd.Series([85, 72, 90], index=["S001", "S002", "S003"], dtype="Int64")
result = backup.combine_first(primary)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [85, 72, 90]
print("Code checks passed. Review the written explanation as well.")

Backup now wins wherever it has a value; S001 is 85.

**Quick check:** Which primary values are replaced by combine_first?

1. Every value
2. Only missing values with an available backup
3. Only zero values

Answer: 2. Existing primary values are retained, including legitimate zero values.

**Explain what you learned:** Filling from another source is a policy as well as a method.

## Module 19. Reshape tables

Move between long and wide layouts and choose when to aggregate.

Before you start: Understand keys, groups, and summary functions.

Official reference: https://pandas.pydata.org/docs/user_guide/reshaping.html

### M19C01. Melt wide columns into rows

A wide table stores different measures in different columns. melt turns selected measure columns into a label column and a value column. The ID is repeated so every new row still belongs to a student.

**Read:** Which column stays as an identifier?

Answer: Student_ID.

**Predict:** How many long rows will 3 students and 2 measures produce?

Answer: 6 rows.

**Explain:** Are Score and Attendance being added together?

Answer: No. melt changes the layout; the values remain separate and their measure labels are retained.



In [ ]:
# Predict before running: How many long rows will 3 students and 2 measures produce?
wide = students().head(3)[["Student_ID", "Score", "Attendance"]]
result = wide.melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
print(result)

**Your turn:** Use the first four students and melt Score and Study_Hours instead.

For the check, keep these result names: `result`.

**Hint:** Change head(3) to head(4) and the two measure names.

**Watch for this:** A shared Value column may contain measures with different units; keep and use Measure.

In [ ]:
# Change this code to complete the task.
# Question: Use the first four students and melt Score and Study_Hours instead.
wide = students().head(4)[["Student_ID", "Score", "Study_Hours"]]
result = wide.melt(id_vars="Student_ID", var_name="Measure", value_name="Value")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 8
assert set(result["Measure"]) == {"Score", "Study_Hours"}
print("Code checks passed. Review the written explanation as well.")

Four students with two measures produce eight rows.

**Quick check:** What does melt mainly change?

1. The values by averaging them
2. The table layout
3. The student IDs

Answer: 2. melt reorganizes values into a long layout without aggregating them.

**Explain what you learned:** Long form keeps one measured value per row.

### M19C02. Pivot a unique pair

pivot moves distinct category labels into columns. Every index-and-column pair must identify at most one value. Here each student has one Score and one Attendance value.

**Read:** Which labels become the output columns?

Answer: The distinct Measure values: Attendance and Score.

**Predict:** What is the Score for S002 after pivoting?

Answer: 72, unchanged.

**Explain:** Why can repeated Student_ID and Measure pairs cause an error?

Answer: pivot cannot choose which of several values belongs in the same output cell.



In [ ]:
# Predict before running: What is the Score for S002 after pivoting?
long = students().head(3).melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
result = long.pivot(index="Student_ID", columns="Measure", values="Value")
print(result)

**Your turn:** Pivot the same long table so Measure labels form the rows and Student_ID labels form the columns.

For the check, keep these result names: `result`.

**Hint:** Exchange the arguments assigned to index and columns.

**Watch for this:** Use pivot_table only when you actually intend to aggregate repeated pairs.

In [ ]:
# Change this code to complete the task.
# Question: Pivot the same long table so Measure labels form the rows and Student_ID labels form the columns.
long = students().head(3).melt(id_vars="Student_ID", value_vars=["Score", "Attendance"], var_name="Measure", value_name="Value")
result = long.pivot(index="Measure", columns="Student_ID", values="Value")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 3)
assert result.loc["Score", "S002"] == 72
print("Code checks passed. Review the written explanation as well.")

The values stay the same, but the two table axes exchange roles.

**Quick check:** What must be unique for pivot?

1. Each row label alone
2. Each column label alone
3. Each index-and-column pair

Answer: 3. The pair identifies one output cell and must have a single value.

**Explain what you learned:** pivot rearranges uniquely identified values.

### M19C03. Aggregate with pivot_table

Several students share the same Campus and Program. pivot_table can reduce those repeated combinations using an explicit aggregation function. Naming aggfunc makes the summary rule clear.

**Read:** What does one cell summarize?

Answer: The mean Score of students with that Campus and Program.

**Predict:** How many campus-by-program cells are there?

Answer: Four: two campuses times two programs.

**Explain:** Why specify aggfunc?

Answer: It documents the statistic; mean, sum, and count answer different questions.



In [ ]:
# Predict before running: How many campus-by-program cells are there?
df = students()
result = df.pivot_table(index="Campus", columns="Program", values="Score", aggfunc="mean", observed=True)
print(result.round(2))

**Your turn:** Change the table to count Student_ID records for each campus and program.

For the check, keep these result names: `result`.

**Hint:** Use values="Student_ID" and aggfunc="count".

**Watch for this:** Filling an absent combination with zero implies a meaning; it is not automatically appropriate for a mean.

In [ ]:
# Change this code to complete the task.
# Question: Change the table to count Student_ID records for each campus and program.
df = students()
result = df.pivot_table(index="Campus", columns="Program", values="Student_ID", aggfunc="count", observed=True)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_numpy().sum() == 12
assert result.loc["RAK", "AI"] == 4
print("Code checks passed. Review the written explanation as well.")

The counts are Dubai AI 2, Dubai IT 4, RAK AI 4, and RAK IT 2.

**Quick check:** Which method handles repeated pairs by summarizing them?

1. pivot_table
2. pivot without any changes
3. set_index alone

Answer: 1. pivot_table applies an aggregation to values sharing the same output cell.

**Explain what you learned:** Choose an aggregation that matches the question.

### M19C04. Count combinations with crosstab

crosstab counts how often two categories occur together. normalize="index" instead reports proportions within each row. The denominator changes, so always label the result.

**Read:** What is counted in each cell?

Answer: Students in that Campus and Program combination.

**Predict:** How many RAK students are in AI?

Answer: 4.

**Explain:** Would a larger count alone prove a stronger preference?

Answer: No. Group sizes matter; proportions may be more suitable for that question.



In [ ]:
# Predict before running: How many RAK students are in AI?
df = students()
result = pd.crosstab(df["Campus"], df["Program"])
print(result)

**Your turn:** Calculate program percentages within each campus and keep the unrounded percentages in result. Print a version rounded to one decimal place.

For the check, keep these result names: `result`.

**Hint:** Use normalize="index" and multiply by 100.

**Watch for this:** Say whether percentages are by row, by column, or of the whole table.

In [ ]:
# Change this code to complete the task.
# Question: Calculate program percentages within each campus and keep the unrounded percentages in result. Print a version rounded to one decimal place.
df = students()
result = pd.crosstab(df["Campus"], df["Program"], normalize="index") * 100
print(result.round(1))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert np.allclose(result.sum(axis=1), 100)
assert np.isclose(result.loc["Dubai", "AI"], 100 / 3)
print("Code checks passed. Review the written explanation as well.")

Each campus row sums to 100%. AI accounts for about 66.7% in RAK and 33.3% in Dubai in this fictional table.

**Quick check:** normalize="index" uses which denominator?

1. Each row total
2. The entire table total
3. Each column total

Answer: 1. Each cell is divided by its own row total.

**Explain what you learned:** Counts and proportions answer different questions.

## Module 20. Read and write files

Move data between pandas and common local file formats.

Before you start: Know DataFrame construction, column selection, and paths.

Official reference: https://pandas.pydata.org/docs/user_guide/io.html

### M20C01. Read and export CSV

CSV stores text rows separated by delimiters. read_csv creates a DataFrame and attempts to infer column types. to_csv writes the selected data; index=False avoids saving the pandas row index as an extra column.

**Read:** What does DATA_DIR / "students.csv" build?

Answer: A platform-independent path to a file inside the data directory.

**Predict:** How many students score at least 85?

Answer: 4 students: S001, S003, S006, and S010.

**Explain:** Why keep Student_ID as text?

Answer: IDs are labels, not quantities; text preserves formats such as leading zeros.



In [ ]:
# Predict before running: How many students score at least 85?
df = pd.read_csv(DATA_DIR / "students.csv", dtype={"Student_ID": "string"})
result = df.loc[df["Score"].ge(85), ["Student_ID", "Name", "Score"]]
result.to_csv(OUTPUT_DIR / "high_scores.csv", index=False)
print(result)
print("Saved outputs/high_scores.csv")

**Your turn:** Read only Student_ID, Name, and Attendance. Save students with Attendance below 85 to attendance_followup.csv.

For the check, keep these result names: `result`.

**Hint:** Use usecols when reading and .lt(85) when filtering.

**Watch for this:** CSV does not preserve every pandas dtype. Check types and missing-value rules when reloading.

In [ ]:
# Change this code to complete the task.
# Question: Read only Student_ID, Name, and Attendance. Save students with Attendance below 85 to attendance_followup.csv.
df = pd.read_csv(DATA_DIR / "students.csv", usecols=["Student_ID", "Name", "Attendance"], dtype={"Student_ID": "string"})
result = df.loc[df["Attendance"].lt(85)]
result.to_csv(OUTPUT_DIR / "attendance_followup.csv", index=False)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Student_ID"].tolist() == ["S004", "S008", "S011"]
assert (OUTPUT_DIR / "attendance_followup.csv").is_file()
print("Code checks passed. Review the written explanation as well.")

The report contains S004, S008, and S011. All written files go in outputs.

**Quick check:** Why use index=False on export?

1. To sort the records
2. To omit the pandas row index
3. To delete Student_ID

Answer: 2. The row index is otherwise written as an additional CSV field.

**Explain what you learned:** Inspect imports and choose export columns deliberately.

### M20C02. Move records through JSON

JSON can store a list of records, each represented by a dictionary. orient="records" makes this structure explicit. Re-reading a file checks that the exported shape matches what you intended.

**Read:** What does one JSON record represent?

Answer: One student row, with column names as dictionary keys.

**Predict:** How many dictionaries are written?

Answer: 3, because head(3) selects three rows.

**Explain:** Why inspect the exported JSON as text?

Answer: It shows the actual file structure, not only its DataFrame representation.



In [ ]:
# Predict before running: How many dictionaries are written?
df = pd.read_json(DATA_DIR / "students.json", orient="records")
result = df[["Student_ID", "Campus", "Score"]].head(3)
result.to_json(OUTPUT_DIR / "student_preview.json", orient="records", indent=2)
print(result)
print((OUTPUT_DIR / "student_preview.json").read_text())

**Your turn:** Export Student_ID and Program for the first two students, then read the saved JSON back into result.

For the check, keep these result names: `result`.

**Hint:** Write with orient="records" and read with the same orientation.

**Watch for this:** File formats can change date and nullable dtype representations; a round trip needs validation.

In [ ]:
# Change this code to complete the task.
# Question: Export Student_ID and Program for the first two students, then read the saved JSON back into result.
df = students().head(2)[["Student_ID", "Program"]]
path = OUTPUT_DIR / "program_preview.json"
df.to_json(path, orient="records", indent=2)
result = pd.read_json(path, orient="records")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 2)
assert result["Student_ID"].tolist() == ["S001", "S002"]
print("Code checks passed. Review the written explanation as well.")

The round trip preserves these two records and their field names.

**Quick check:** What does orient="records" produce?

1. A list of row dictionaries
2. One image
3. An Excel worksheet

Answer: 1. Each output dictionary describes a row using field names and values.

**Explain what you learned:** Choose a file orientation that matches the receiving system.

### M20C03. Select an Excel worksheet

An Excel workbook can contain several sheets. sheet_name selects one by name. The openpyxl engine handles the provided .xlsx workbook and is listed in the installation requirements.

**Read:** What is the difference between the file and the sheet?

Answer: students.xlsx is the workbook file; Students identifies a worksheet inside it.

**Predict:** How many rows are in the summary?

Answer: 2, one for Dubai and one for RAK.

**Explain:** Why use a new output filename?

Answer: It preserves the supplied source data and clearly identifies the generated report.



In [ ]:
# Predict before running: How many rows are in the summary?
df = pd.read_excel(DATA_DIR / "students.xlsx", sheet_name="Students", engine="openpyxl")
result = df.groupby("Campus", as_index=False)["Score"].mean()
result.to_excel(OUTPUT_DIR / "campus_report.xlsx", sheet_name="Summary", index=False, engine="openpyxl")
print(result.round(2))

**Your turn:** Read the Students sheet. Store a summary in result with columns Program and Students, where Students counts rows per program. Export it to program_counts.xlsx, sheet Counts.

For the check, keep these result names: `result`.

**Hint:** groupby("Program").size() counts rows; name the count column.

**Watch for this:** If openpyxl is missing, install the provided requirements before running this cell.

In [ ]:
# Change this code to complete the task.
# Question: Read the Students sheet. Store a summary in result with columns Program and Students, where Students counts rows per program. Export it to program_counts.xlsx, sheet Counts.
df = pd.read_excel(DATA_DIR / "students.xlsx", sheet_name="Students", engine="openpyxl")
result = df.groupby("Program").size().rename("Students").reset_index()
result.to_excel(OUTPUT_DIR / "program_counts.xlsx", sheet_name="Counts", index=False, engine="openpyxl")
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Students"].tolist() == [6, 6]
assert (OUTPUT_DIR / "program_counts.xlsx").is_file()
print("Code checks passed. Review the written explanation as well.")

The workbook contains two counts: six AI students and six IT students.

**Quick check:** Which argument chooses a worksheet?

1. sheet_name
2. index=False
3. dtype alone

Answer: 1. sheet_name identifies which worksheet to read or create.

**Explain what you learned:** Excel I/O needs a workbook path and the correct worksheet.

### M20C04. Query a small SQLite table

SQL retrieves data from tables using a query. This example creates a temporary in-memory database from the fictional data. The parameter placeholder passes a value separately from the query text.

**Read:** What does :memory: mean?

Answer: The database exists in memory for this connection and is not a disk database file.

**Predict:** Which student appears first?

Answer: Fatima, S006, with Score 95.

**Explain:** Why use a placeholder and params?

Answer: It passes values safely without manually inserting them into SQL text.



In [ ]:
# Predict before running: Which student appears first?
with sqlite3.connect(":memory:") as connection:
    students().to_sql("students", connection, index=False, if_exists="replace")
    result = pd.read_sql_query(
        "SELECT Student_ID, Name, Score FROM students WHERE Score >= ? ORDER BY Score DESC",
        connection, params=(85,))
print(result)

**Your turn:** Use SQL to calculate mean Score and student count for each Campus. Store the returned DataFrame in result with columns Campus, Mean_Score, and Students, ordered by Campus.

For the check, keep these result names: `result`.

**Hint:** Use GROUP BY Campus with AVG(Score) AS Mean_Score and COUNT(*) AS Students.

**Watch for this:** Only use if_exists="replace" here on the temporary classroom table; on a real database it replaces the existing table.

In [ ]:
# Change this code to complete the task.
# Question: Use SQL to calculate mean Score and student count for each Campus. Store the returned DataFrame in result with columns Campus, Mean_Score, and Students, ordered by Campus.
with sqlite3.connect(":memory:") as connection:
    students().to_sql("students", connection, index=False, if_exists="replace")
    result = pd.read_sql_query(
        "SELECT Campus, AVG(Score) AS Mean_Score, COUNT(*) AS Students FROM students GROUP BY Campus ORDER BY Campus",
        connection)
print(result.round(2))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Students"].tolist() == [6, 6]
assert np.isclose(result.loc[result["Campus"].eq("RAK"), "Mean_Score"].item(), 81)
print("Code checks passed. Review the written explanation as well.")

This performs the grouping inside SQL and returns the two-row result as a DataFrame.

**Quick check:** What does read_sql_query return?

1. A DataFrame
2. A database server
3. An Excel file

Answer: 1. The query result becomes a pandas DataFrame.

**Explain what you learned:** Pandas can receive SQL results without loading every source row.

## Module 21. Work with time series

Distinguish calendar dates, observed records, windows, and changes over time.

Before you start: Parse dates and understand missing values and sorting.

Official reference: https://pandas.pydata.org/docs/user_guide/timeseries.html

### M21C01. Expose a missing day

A DatetimeIndex lets pandas work with calendar dates. asfreq("D") requests a daily index between the first and last observations. The missing October 4 row appears as unknown, not zero.

**Read:** What does freq="D" represent?

Answer: One index label per calendar day.

**Predict:** How many rows will result contain?

Answer: 7 calendar days, although the source has only 6 observations.

**Explain:** Why not fill the missing day with zero automatically?

Answer: No record does not prove that nobody visited; the value is unknown.



In [ ]:
# Predict before running: How many rows will result contain?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
result = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
print(result)

**Your turn:** Build the daily Series and select only entries whose visitor count is missing. Store that filtered Series, retaining its date index, in result and print it.

For the check, keep these result names: `result`.

**Hint:** Filter the daily Series with its isna() mask.

**Watch for this:** asfreq does not aggregate. Ensure the date index is unique before requesting a frequency.

In [ ]:
# Change this code to complete the task.
# Question: Build the daily Series and select only entries whose visitor count is missing. Store that filtered Series, retaining its date index, in result and print it.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = daily[daily.isna()]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 1
assert result.index[0] == pd.Timestamp("2026-10-04")
print("Code checks passed. Review the written explanation as well.")

Only 2026-10-04 is missing between the observed start and end dates.

**Quick check:** What can asfreq("D") reveal?

1. Missing calendar dates
2. The true counts for missing days
3. Why the sensor failed

Answer: 1. It creates the requested date grid; it cannot reconstruct unknown measurements.

**Explain what you learned:** A gap in records is not automatically a zero.

### M21C02. Resample into calendar bins

resample groups observations into time bins. Here each bin starts at a three-day boundary measured from the first day. Report the number of observations beside the total so a partial bin is visible.

**Read:** What is grouped by resample("3D")?

Answer: Observations falling in successive three-day calendar bins.

**Predict:** What total is recorded for October 1 through October 3?

Answer: 62: 20 + 24 + 18.

**Explain:** Is the total of 58 for the next bin a complete three-day total?

Answer: No. October 4 is missing, so it includes only the recorded counts for October 5 and 6.



In [ ]:
# Predict before running: What total is recorded for October 1 through October 3?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
s = df.set_index("Date")["Visitors"].sort_index()
groups = s.resample("3D")
result = pd.DataFrame({"Recorded_Total": groups.sum(min_count=1), "Observed_Days": groups.count()})
print(result)

**Your turn:** Resample to individual days and use sum(min_count=1) so October 4 stays missing. Store only the daily total Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Change "3D" to "D" and require at least one known value for a sum.

**Watch for this:** Time-bin labels and boundaries depend on frequency and options; state the reporting period.

In [ ]:
# Change this code to complete the task.
# Question: Resample to individual days and use sum(min_count=1) so October 4 stays missing. Store only the daily total Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
s = df.set_index("Date")["Visitors"].sort_index()
result = s.resample("D").sum(min_count=1)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 7
assert pd.isna(result.loc["2026-10-04"])
assert result.loc["2026-10-05"] == 30
print("Code checks passed. Review the written explanation as well.")

min_count=1 keeps a day with no observations unknown instead of reporting an empty sum as zero.

**Quick check:** Why show Observed_Days beside totals?

1. To reveal incomplete bins
2. To make every bin complete
3. To replace missing days

Answer: 1. A total based on two observed days should not be interpreted as a complete three-day total.

**Explain what you learned:** Time summaries need a coverage check.

### M21C03. Calculate a rolling mean

A rolling window calculates a statistic repeatedly as it moves through the data. On this daily grid, rolling(3) means three calendar rows. min_periods=3 requires three known counts, so windows crossing the missing day stay unknown.

**Read:** What does min_periods=3 require?

Answer: At least three non-missing observations within the current window.

**Predict:** What is the rolling mean on October 7?

Answer: 30: (30 + 28 + 32) / 3.

**Explain:** Would rolling(3) mean three days on the original irregular index?

Answer: No. It would mean three observed rows, which can span more than three days.



In [ ]:
# Predict before running: What is the rolling mean on October 7?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Mean_3_Days": daily.rolling(3, min_periods=3).mean()})
print(result.round(2))

**Your turn:** Calculate a two-day rolling mean requiring two known daily counts. Store only the resulting mean Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Apply rolling(2, min_periods=2).mean() to the daily Series and save that Series directly.

**Watch for this:** Sort time data before rolling. Otherwise the window follows an unintended order.

In [ ]:
# Change this code to complete the task.
# Question: Calculate a two-day rolling mean requiring two known daily counts. Store only the resulting mean Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = daily.rolling(2, min_periods=2).mean()
print(result.round(2))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["2026-10-02"] == 22
assert result.loc["2026-10-07"] == 30
assert pd.isna(result.loc["2026-10-05"])
print("Code checks passed. Review the written explanation as well.")

The last two-day mean is also 30, using only October 6 and 7. Earlier windows differ.

**Quick check:** Without a regular daily grid, rolling(3) means what?

1. Exactly three calendar days
2. Three rows
3. Three months

Answer: 2. An integer window size counts rows, regardless of the spacing between timestamps.

**Explain what you learned:** Window size and missing-data rules both affect a moving average.

### M21C04. Compare with a previous period

shift(1) places the previous row value beside the current row. pct_change(fill_method=None) calculates fractional change without filling gaps. Multiplying by 100 expresses that change as a percentage.

**Read:** Why is the first change missing?

Answer: There is no previous value in this series for comparison.

**Predict:** What percentage change is shown on October 2?

Answer: 20%, from 20 to 24.

**Explain:** Why is the October 5 change unknown?

Answer: The previous calendar day, October 4, has no known visitor count.



In [ ]:
# Predict before running: What percentage change is shown on October 2?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Previous_Day": daily.shift(1), "Change_Percent": daily.pct_change(fill_method=None) * 100})
print(result.round(2))

**Your turn:** Calculate the signed difference in visitor counts from two calendar days earlier using shift(2). Store the difference Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Subtract the shifted daily Series from the original daily Series.

**Watch for this:** Percent change from a previous zero is undefined; inspect zero denominators.

In [ ]:
# Change this code to complete the task.
# Question: Calculate the signed difference in visitor counts from two calendar days earlier using shift(2). Store the difference Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = daily - daily.shift(2)
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.loc["2026-10-07"] == 2
assert result.loc["2026-10-03"] == -2
print("Code checks passed. Review the written explanation as well.")

October 7 is two visitors higher than October 5. Comparisons involving missing dates stay unknown.

**Quick check:** Why multiply pct_change by 100?

1. It returns fractional change
2. It returns rounded percentage points
3. It returns a count

Answer: 1. A fractional change of 0.20 corresponds to 20%.

**Explain what you learned:** Be explicit about the time interval and denominator.

## Module 22. Explore richer structures

Use multiple index levels, list-valued columns, and categorical encodings.

Before you start: Understand indexes, reshaping, and basic data types.

Official reference: https://pandas.pydata.org/docs/user_guide/advanced.html

### M22C01. Create a MultiIndex

A MultiIndex stores more than one label per row. Here Campus and Student_ID jointly locate a student. sort_index arranges the levels for predictable selection.

**Read:** What does the tuple ("RAK", "S003") identify?

Answer: A row with Campus RAK and Student_ID S003.

**Predict:** What Name and Score will appear?

Answer: Chloe and 91.

**Explain:** Why keep Student_ID as an additional level?

Answer: Campus alone is shared by several students; the second level identifies the individual record.



In [ ]:
# Predict before running: What Name and Score will appear?
df = students().set_index(["Campus", "Student_ID"]).sort_index()
result = df.loc[("RAK", "S003"), ["Name", "Score"]]
print(result)

**Your turn:** Use xs to select all Dubai students at the Campus level and print Name and Score.

For the check, keep these result names: `result`.

**Hint:** Call xs("Dubai", level="Campus") on the indexed table.

**Watch for this:** A tuple used as a MultiIndex key is different from a list of separate row keys.

In [ ]:
# Change this code to complete the task.
# Question: Use xs to select all Dubai students at the Campus level and print Name and Score.
df = students().set_index(["Campus", "Student_ID"]).sort_index()
result = df.xs("Dubai", level="Campus")[["Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 6
assert result.index.name == "Student_ID"
assert "S002" in result.index
print("Code checks passed. Review the written explanation as well.")

The selection contains six Dubai students; xs drops the selected level by default.

**Quick check:** What is a MultiIndex?

1. Several labels per row or column axis entry
2. Several unrelated DataFrames
3. A duplicate-free column

Answer: 1. It represents an axis with multiple levels of labels.

**Explain what you learned:** Name index levels so selections are understandable.

### M22C02. Stack and unstack

stack moves column labels into an additional row-index level. unstack moves an index level back into columns. The explicit future_stack=True selects the modern stack implementation on pandas 2.2 and 3.x.

**Read:** How many index levels does the stacked result have?

Answer: Two: Student_ID and the original measure column label.

**Predict:** How many values are in the stacked result?

Answer: 4, from two students and two measures.

**Explain:** Were the measurements averaged during stacking?

Answer: No. Their labels moved; the original values remain.



In [ ]:
# Predict before running: How many values are in the stacked result?
wide = students().head(2).set_index("Student_ID")[["Score", "Attendance"]]
result = wide.stack(future_stack=True)
print(result)

**Your turn:** Stack the first three students and unstack the last level back into a DataFrame. Check that it equals the original wide table.

For the check, keep these result names: `result`, `wide`.

**Hint:** Use .unstack() on the stacked Series, then .equals(wide).

**Watch for this:** Missing combinations can appear when reshaping sparse data; inspect the resulting shape.

In [ ]:
# Change this code to complete the task.
# Question: Stack the first three students and unstack the last level back into a DataFrame. Check that it equals the original wide table.
wide = students().head(3).set_index("Student_ID")[["Score", "Attendance"]]
stacked = wide.stack(future_stack=True)
result = stacked.unstack().reindex(columns=wide.columns)
print(result)
print("Values restored:", result.equals(wide))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.equals(wide)
assert result.shape == (3, 2)
print("Code checks passed. Review the written explanation as well.")

The explicit column reindex restores the original column order for the equality check.

**Quick check:** Which operation returns the last index level to columns?

1. unstack
2. drop_duplicates
3. head

Answer: 1. unstack reshapes one index level into column labels.

**Explain what you learned:** Stack and unstack move labels between axes.

### M22C03. Expand lists with explode

Sometimes one cell contains several items, such as a list of skills. explode creates a separate row for each item and repeats the other columns. This changes the unit of analysis from one row per student to one row per student-skill.

**Read:** What kind of value is stored in each Skills cell before explode?

Answer: A Python list.

**Predict:** How many rows result?

Answer: 3: two skills for S001 and one skill for S002.

**Explain:** Why would counting rows after explode overcount students?

Answer: A student with multiple skills appears in multiple rows.



In [ ]:
# Predict before running: How many rows result?
df = pd.DataFrame({"Student_ID": ["S001", "S002"], "Skills": [["Python", "SQL"], ["Excel"]]})
result = df.explode("Skills", ignore_index=True)
print(result)

**Your turn:** Give S002 a second skill, SQL. Explode the table, then store the count of student-skill rows for each skill in result as a value_counts() Series.

For the check, keep these result names: `result`.

**Hint:** After explode, use the Skills column value_counts().

**Watch for this:** An empty list becomes a row with a missing item; a string that looks like a list is still a string.

In [ ]:
# Change this code to complete the task.
# Question: Give S002 a second skill, SQL. Explode the table, then store the count of student-skill rows for each skill in result as a value_counts() Series.
df = pd.DataFrame({"Student_ID": ["S001", "S002"], "Skills": [["Python", "SQL"], ["Excel", "SQL"]]})
long = df.explode("Skills", ignore_index=True)
result = long["Skills"].value_counts()
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["SQL"] == 2
assert result.sum() == 4
print("Code checks passed. Review the written explanation as well.")

SQL appears twice; Python and Excel appear once each.

**Quick check:** After explode, one row represents what?

1. Always one unique student
2. One listed item belonging to a student
3. One average skill score

Answer: 2. Each item becomes a row; the student identifier can repeat.

**Explain what you learned:** Check what a row represents after changing the layout.

### M22C04. Order and encode categories

Categorical values can follow a chosen order instead of alphabetical order. get_dummies makes one indicator column per category. These indicator values describe membership, not a numerical ranking.

**Read:** Which Program sorts first?

Answer: IT, because it is listed first in the ordered categories.

**Predict:** What are Program_IT and Program_AI for S001?

Answer: 0 and 1 respectively.

**Explain:** Does a dummy value of 1 mean AI is better than IT?

Answer: No. It only marks whether that row belongs to the AI category.



In [ ]:
# Predict before running: What are Program_IT and Program_AI for S001?
df = students().head(4).copy()
df["Program"] = pd.Categorical(df["Program"], categories=["IT", "AI"], ordered=True)
ordered = df.sort_values("Program", kind="stable")
result = pd.get_dummies(ordered["Program"], prefix="Program", dtype=int)
print(ordered[["Student_ID", "Program"]])
print(result)

**Your turn:** Change the category order to AI then IT and print the sorted student IDs and dummy columns.

For the check, keep these result names: `ordered`, `result`.

**Hint:** Reverse the categories list before sorting.

**Watch for this:** Values outside a specified category list become missing; validate allowed categories first.

In [ ]:
# Change this code to complete the task.
# Question: Change the category order to AI then IT and print the sorted student IDs and dummy columns.
df = students().head(4).copy()
df["Program"] = pd.Categorical(df["Program"], categories=["AI", "IT"], ordered=True)
ordered = df.sort_values("Program", kind="stable")
result = pd.get_dummies(ordered["Program"], prefix="Program", dtype=int)
print(ordered[["Student_ID", "Program"]])
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert ordered["Student_ID"].tolist() == ["S001", "S003", "S002", "S004"]
assert result.loc[0, "Program_AI"] == 1
print("Code checks passed. Review the written explanation as well.")

AI students now appear first; each row still has one indicator value equal to 1.

**Quick check:** What does get_dummies encode?

1. Category membership
2. Student achievement
3. A guaranteed useful order

Answer: 1. Each column indicates membership in one category.

**Explain what you learned:** Category order and indicator encoding are separate choices.

## Module 23. Work efficiently and reproducibly

Read less data, combine chunk summaries correctly, and make repeatable samples.

Before you start: Understand grouping, CSV I/O, and data types.

Official reference: https://pandas.pydata.org/docs/user_guide/scale.html

### M23C01. Measure memory use

memory_usage(deep=True) estimates memory including text stored by object columns. Repeated labels may use less memory as categories. The benefit depends on the data, so measure it instead of assuming.

**Read:** Why repeat only two labels many times?

Answer: It demonstrates data with few unique text values and many repeated occurrences.

**Predict:** Will converting to category change the label values?

Answer: No. It changes their representation, not the label meanings.

**Explain:** Are exact byte counts guaranteed on every computer?

Answer: No. They depend on versions and storage details; compare the measurements in your environment.



In [ ]:
# Predict before running: Will converting to category change the label values?
labels = pd.Series(["RAK", "Dubai"] * 500, dtype=object)
category = labels.astype("category")
result = pd.Series({"Object_bytes": labels.memory_usage(deep=True), "Category_bytes": category.memory_usage(deep=True)})
print(result)

**Your turn:** Measure 1,000 unique text labels before and after category conversion. Explain why the saving may shrink or disappear.

For the check, keep these result names: `category`, `labels`.

**Hint:** Build labels such as "ID_0" through "ID_999".

**Watch for this:** This is a small storage demonstration, not a benchmark of all pandas workloads.

In [ ]:
# Change this code to complete the task.
# Question: Measure 1,000 unique text labels before and after category conversion. Explain why the saving may shrink or disappear.
labels = pd.Series([f"ID_{i}" for i in range(1000)], dtype=object)
category = labels.astype("category")
result = pd.Series({"Object_bytes": labels.memory_usage(deep=True), "Category_bytes": category.memory_usage(deep=True)})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(category.cat.categories) == 1000
assert category.astype(object).equals(labels)
print("Code checks passed. Review the written explanation as well.")

Categories also store the distinct labels and codes. With nearly every label unique, that overhead can outweigh any benefit.

**Quick check:** When is category often useful for memory?

1. Many repeated values from a small set
2. Every value is unique, always
3. Only when the column is numeric

Answer: 1. A small category dictionary can represent many repeated labels compactly.

**Explain what you learned:** Choose data types based on meaning and measurement.

### M23C02. Load only needed columns

usecols avoids reading columns that the question does not need. Explicit dtypes make the import contract clearer. This small file demonstrates a habit that becomes useful with larger files.

**Read:** Which six source columns are omitted?

Answer: Name, Campus, Age, Study_Hours, Attendance, and Join_Date.

**Predict:** How many rows are loaded?

Answer: All 12 rows; usecols limits columns, not rows.

**Explain:** Why not load every field just in case?

Answer: Unneeded columns use memory and add distractions to the analysis.



In [ ]:
# Predict before running: How many rows are loaded?
result = pd.read_csv(DATA_DIR / "students.csv", usecols=["Student_ID", "Program", "Score"], dtype={"Student_ID": "string", "Program": "category"})
print(result.head())
print(result.dtypes)

**Your turn:** Read only Student_ID and Attendance, and limit the preview to the first four file rows with nrows.

For the check, keep these result names: `result`.

**Hint:** Use nrows=4 inside read_csv.

**Watch for this:** A file preview can miss problems that occur later in the file.

In [ ]:
# Change this code to complete the task.
# Question: Read only Student_ID and Attendance, and limit the preview to the first four file rows with nrows.
result = pd.read_csv(DATA_DIR / "students.csv", usecols=["Student_ID", "Attendance"], nrows=4, dtype={"Student_ID": "string"})
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (4, 2)
assert result.iloc[-1]["Student_ID"] == "S004"
print("Code checks passed. Review the written explanation as well.")

Only four rows and two columns are loaded. This preview does not represent the whole dataset.

**Quick check:** What does nrows=4 do?

1. Loads only the first four data rows
2. Chooses four random rows
3. Loads four columns

Answer: 1. nrows limits the number of parsed data rows from the beginning of the file.

**Explain what you learned:** Read the fields and rows needed for the current step.

### M23C03. Combine chunk totals correctly

chunksize reads a CSV in pieces. A total and a non-missing count can be accumulated across pieces without keeping all rows. Divide the overall total by the overall count to calculate the overall mean.

**Read:** What is chunk in each loop iteration?

Answer: A DataFrame containing the next group of file rows, up to five rows here.

**Predict:** How many chunks are processed?

Answer: 3 chunks, with 5, 5, and 2 rows.

**Explain:** Why not average the three chunk means equally?

Answer: The chunks have different counts; equal weighting would give the two-row chunk too much influence.



In [ ]:
# Predict before running: How many chunks are processed?
total = 0
count = 0
for chunk in pd.read_csv(DATA_DIR / "students.csv", usecols=["Score"], chunksize=5):
    total += chunk["Score"].sum()
    count += chunk["Score"].count()
result = total / count
print("Known scores:", count)
print("Mean score:", round(result, 2))

**Your turn:** Use chunks of four rows and calculate how many students score at least 80 without retaining all rows.

For the check, keep these result names: `result`.

**Hint:** Add the sum of a Boolean condition for each chunk.

**Watch for this:** Not every operation is easy to compute in chunks. A global sort or median needs a different strategy.

In [ ]:
# Change this code to complete the task.
# Question: Use chunks of four rows and calculate how many students score at least 80 without retaining all rows.
result = 0
for chunk in pd.read_csv(DATA_DIR / "students.csv", usecols=["Score"], chunksize=4):
    result += chunk["Score"].ge(80).sum()
print("Students with Score >= 80:", result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == 6
print("Code checks passed. Review the written explanation as well.")

Six students meet the threshold, regardless of the chosen chunk boundaries.

**Quick check:** What is needed for an exact combined mean?

1. An unweighted mean of chunk means
2. A combined sum and non-missing count
3. Only the last chunk

Answer: 2. The overall mean is the total of known values divided by their total count.

**Explain what you learned:** Combine summaries using their correct mathematical weights.

### M23C04. Take a repeatable sample

sample chooses rows randomly. random_state makes the selection repeatable for this data and environment. Reproducibility does not make a tiny sample representative of the population.

**Read:** What does n=3 request?

Answer: Three sampled rows.

**Predict:** Will the repeated call with the same input and seed match?

Answer: Yes, within the same sampling environment.

**Explain:** Does a fixed seed remove sampling bias?

Answer: No. It makes the selection reproducible; it does not guarantee a balanced or representative sample.



In [ ]:
# Predict before running: Will the repeated call with the same input and seed match?
df = students()
result = df.sample(n=3, random_state=7)
repeat = df.sample(n=3, random_state=7)
print(result[["Student_ID", "Campus", "Score"]])
print("Same sample:", result.equals(repeat))

**Your turn:** Take two students from each campus using groupby.sample with random_state=7.

For the check, keep these result names: `result`.

**Hint:** Group by Campus before calling sample(n=2, random_state=7).

**Watch for this:** Document input data, row order, library versions, and seed when an exact sample matters.

In [ ]:
# Change this code to complete the task.
# Question: Take two students from each campus using groupby.sample with random_state=7.
df = students()
result = df.groupby("Campus", group_keys=False).sample(n=2, random_state=7)
print(result[["Student_ID", "Campus", "Score"]])
print(result["Campus"].value_counts())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 4
assert result["Campus"].value_counts().to_dict() == {"Dubai": 2, "RAK": 2}
print("Code checks passed. Review the written explanation as well.")

The result has four students, two from each campus. Equal counts here are a chosen design, not a universal sampling rule.

**Quick check:** What does a seed mainly provide?

1. Guaranteed representativeness
2. Repeatability under the same setup
3. Higher scores

Answer: 2. A seed controls the random sequence; it does not establish statistical validity.

**Explain what you learned:** Repeatability and representativeness are different goals.

## Module 24. Plot and interpret

Select a plot that answers a clear question and label its units.

Before you start: Use summaries, numeric columns, and sorted dates.

Official reference: https://pandas.pydata.org/docs/user_guide/visualization.html

### M24C01. Compare groups with bars

A bar chart compares categories. First calculate one statistic per group, then plot those values. The axis label must say whether the height is a mean, count, or total.

**Read:** What does the height of one bar represent?

Answer: The mean student Score for that campus in the fictional table.

**Predict:** Which campus has the higher mean?

Answer: RAK, with 81.00, compared with Dubai at about 76.83.

**Explain:** Does the chart prove campus causes a score difference?

Answer: No. These are descriptive summaries from a small fictional dataset, without causal evidence.



In [ ]:
# Predict before running: Which campus has the higher mean?
df = students()
result = df.groupby("Campus")["Score"].mean().sort_index()
ax = result.plot.bar(color="#28756a", rot=0, figsize=(6, 3.6))
ax.set(title="Mean score by campus (practice data)", xlabel="Campus", ylabel="Mean score (out of 100)", ylim=(0, 100))
plt.tight_layout()
print(result.round(2))
plt.show()

**Your turn:** Plot the number of students in each Program instead, with a count label on the y-axis.

For the check, keep these result names: `result`.

**Hint:** Use groupby("Program").size() before plot.bar().

**Watch for this:** Use a zero baseline for ordinary bar comparisons so bar length remains meaningful.

In [ ]:
# Change this code to complete the task.
# Question: Plot the number of students in each Program instead, with a count label on the y-axis.
df = students()
result = df.groupby("Program").size()
ax = result.plot.bar(color="#28756a", rot=0, figsize=(6, 3.6))
ax.set(title="Students by program (practice data)", xlabel="Program", ylabel="Number of students")
ax.set_ylim(bottom=0)
plt.tight_layout()
print(result)
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.to_dict() == {"AI": 6, "IT": 6}
print("Code checks passed. Review the written explanation as well.")

Both programs have six students, so their bars have the same height.

**Quick check:** Why label the y-axis "Mean score"?

1. To identify the statistic and units
2. To make the bar taller
3. To remove uncertainty

Answer: 1. A viewer needs to know what each plotted quantity means.

**Explain what you learned:** Aggregate first, then label exactly what was plotted.

### M24C02. Inspect a distribution

A histogram counts numeric observations within intervals called bins. It shows the distribution of individual values, rather than a summary for each named category. Bin choices can change the visual impression.

**Read:** What is counted along the vertical axis?

Answer: Students whose scores fall inside each interval.

**Predict:** How many scores are in [90, 100)?

Answer: 2: 91 and 95.

**Explain:** How is this different from a bar chart of campus means?

Answer: A histogram groups numeric values into intervals; campus bars compare named groups.



In [ ]:
# Predict before running: How many scores are in [90, 100)?
df = students()
result = pd.cut(df["Score"], bins=[50, 60, 70, 80, 90, 100], right=False).value_counts(sort=False)
ax = df["Score"].plot.hist(bins=[50, 60, 70, 80, 90, 100], color="#28756a", edgecolor="white", figsize=(6, 3.6))
ax.set(title="Distribution of scores (practice data)", xlabel="Score (out of 100)", ylabel="Number of students")
plt.tight_layout()
print(result)
plt.show()

**Your turn:** Use wider bins [50, 75, 100] and compare the count table and visual shape.

For the check, keep these result names: `result`.

**Hint:** Change the bin edges in both pd.cut and plot.hist.

**Watch for this:** Histogram libraries usually include the upper endpoint of the last bin; pd.cut(right=False) does not. These sample scores contain no endpoint value of 100.

In [ ]:
# Change this code to complete the task.
# Question: Use wider bins [50, 75, 100] and compare the count table and visual shape.
df = students()
result = pd.cut(df["Score"], bins=[50, 75, 100], right=False).value_counts(sort=False)
ax = df["Score"].plot.hist(bins=[50, 75, 100], color="#28756a", edgecolor="white", figsize=(6, 3.6))
ax.set(title="Scores with wider bins (practice data)", xlabel="Score (out of 100)", ylabel="Number of students")
plt.tight_layout()
print(result)
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.tolist() == [4, 8]
assert result.sum() == 12
print("Code checks passed. Review the written explanation as well.")

The two wider bins contain four scores below 75 and eight scores from 75 up to 100.

**Quick check:** What happens when bin widths change?

1. The raw scores change
2. The grouped visual summary changes
3. The number of students changes

Answer: 2. Binning changes the display, not the underlying observations.

**Explain what you learned:** Check bin boundaries when interpreting distributions.

### M24C03. Compare two measurements

A scatter plot places one point at each pair of numeric measurements. This can reveal a pattern, an unusual observation, or a gap. Correlation describes association; it does not establish why the pattern occurs.

**Read:** What does one dot represent?

Answer: One student with a Study_Hours value and a Score value.

**Predict:** Will the overall pattern slope upward or downward?

Answer: Upward in this fictional table: higher study hours generally accompany higher scores.

**Explain:** Can this prove that adding one hour causes a fixed score increase?

Answer: No. The table is fictional and descriptive, and the plot does not control other factors.



In [ ]:
# Predict before running: Will the overall pattern slope upward or downward?
df = students()
result = df[["Study_Hours", "Score"]].corr().loc["Study_Hours", "Score"]
ax = df.plot.scatter(x="Study_Hours", y="Score", color="#28756a", figsize=(6, 3.6))
ax.set(title="Study time and score (practice data)", xlabel="Study hours per week", ylabel="Score (out of 100)")
plt.tight_layout()
print("Pearson correlation:", round(result, 3))
plt.show()

**Your turn:** Plot Attendance against Score and calculate their Pearson correlation.

For the check, keep these result names: `result`.

**Hint:** Replace Study_Hours with Attendance in both the correlation and plot.

**Watch for this:** Outliers, restricted ranges, and missing data can affect correlation; inspect the actual points.

In [ ]:
# Change this code to complete the task.
# Question: Plot Attendance against Score and calculate their Pearson correlation.
df = students()
result = df[["Attendance", "Score"]].corr().loc["Attendance", "Score"]
ax = df.plot.scatter(x="Attendance", y="Score", color="#28756a", figsize=(6, 3.6))
ax.set(title="Attendance and score (practice data)", xlabel="Attendance (%)", ylabel="Score (out of 100)")
plt.tight_layout()
print("Pearson correlation:", round(result, 3))
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert 0 < result <= 1
assert np.isclose(result, students()["Attendance"].corr(students()["Score"]))
print("Code checks passed. Review the written explanation as well.")

The revised plot compares attendance percentages with scores. Its association still does not establish causation.

**Quick check:** What does a positive correlation indicate?

1. One variable must cause the other
2. Values tend to increase together
3. Every point lies on a straight line

Answer: 2. A positive correlation describes the direction of association, not a causal mechanism.

**Explain what you learned:** Use scatter plots to inspect relationships without claiming causes.

### M24C04. Show change over time

A line chart uses an ordered horizontal axis to show change. Reindexing to a daily grid keeps the missing date visible as a gap. A line across an unobserved day can otherwise suggest continuity that was not measured.

**Read:** Why convert Date before plotting?

Answer: It gives the horizontal axis chronological meaning and supports a regular daily index.

**Predict:** Where will the gap appear?

Answer: October 4, whose visitor count is missing.

**Explain:** Does the gap mean no visitors came?

Answer: No. The count is unknown for that day.



In [ ]:
# Predict before running: Where will the gap appear?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
result = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(result.index, result, marker="o", color="#28756a")
ax.set(title="Daily visitors (practice data)", xlabel="Date", ylabel="Recorded visitors")
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
plt.tight_layout()
print(result)
plt.show()

**Your turn:** Store the daily counts in daily and the three-day rolling mean in result, requiring three known daily values. Plot daily and result together with labels for both lines.

For the check, keep these result names: `result`.

**Hint:** Calculate result from daily.rolling(3, min_periods=3).mean(), then plot both Series on the same axes.

**Watch for this:** A smooth line does not create new observations; report how any smoothing was calculated.

In [ ]:
# Change this code to complete the task.
# Question: Store the daily counts in daily and the three-day rolling mean in result, requiring three known daily values. Plot daily and result together with labels for both lines.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = daily.rolling(3, min_periods=3).mean()
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(daily.index, daily, marker="o", label="Daily count", color="#28756a")
ax.plot(result.index, result, marker="s", label="3-day mean", color="#9c6b28")
ax.set(title="Daily visitors and rolling mean (practice data)", xlabel="Date", ylabel="Visitors")
ax.set_ylim(bottom=0)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
print(result.round(2))
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.notna().sum() == 2
assert result.loc["2026-10-07"] == 30
print("Code checks passed. Review the written explanation as well.")

The rolling mean is available only on October 3 and October 7 because other windows lack three known values.

**Quick check:** Why leave a visible gap for the missing day?

1. It communicates an unknown measurement
2. It proves the count was zero
3. It increases the sample size

Answer: 1. A gap prevents an unknown value from being mistaken for an observed zero.

**Explain what you learned:** Time order, gaps, and transformations belong in the explanation.

## Module 25. Debugging clinic

Find the reason a pandas result is wrong, then repair it.

Before you start: Modules 1–20; revisit selection, cleaning, and joins.

Official reference: https://pandas.pydata.org/docs/user_guide/copy_on_write.html

### M25C01. A label is not a row position

A table can use student IDs as its index. Position 0 still means the first row, but the label of that row is now S001. Read the index before choosing a selector.

**Read:** What labels does df use now?

Answer: The Student_ID values, such as S001, are index labels.

**Predict:** Why do both selections contain two rows?

Answer: iloc excludes stop position 2; loc includes both existing endpoint labels S001 and S002.

**Explain:** Would df.loc[0] mean the first row here?

Answer: No. It asks for label 0, which is absent, so it raises KeyError.



In [ ]:
# Predict before running: Why do both selections contain two rows?
df = students().set_index("Student_ID")
by_position = df.iloc[0:2][["Name", "Score"]]
by_label = df.loc["S001":"S002", ["Name", "Score"]]
print(by_position)
print("Same rows:", by_position.equals(by_label))

**Your turn:** Select S003 through S005 by label, then select the same three students by position. Print both and confirm equality.

For the check, keep these result names: `by_label`, `by_position`.

**Hint:** The first selected row has position 2. The excluded stop must be one position after S005.

**Watch for this:** An integer inside loc is a label, even when it looks like a row number.

In [ ]:
# Change this code to complete the task.
# Question: Select S003 through S005 by label, then select the same three students by position. Print both and confirm equality.
df = students().set_index("Student_ID")
by_label = df.loc["S003":"S005", ["Name", "Score"]]
by_position = df.iloc[2:5][["Name", "Score"]]
print(by_label)
print(by_position)
print("Same rows:", by_label.equals(by_position))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert by_position.equals(by_label)
assert by_label.index.tolist() == ['S003','S004','S005']
print("Code checks passed. Review the written explanation as well.")

Both results contain Chloe, Dana, and Ethan. Labels and positions happen to select the same records here, but they express different rules.

**Quick check:** Which stop is excluded?

1. The stop in an iloc slice
2. The stop in every loc label slice
3. Both stops are always excluded

Answer: 1. iloc uses Python's stop-exclusive positional slicing. Existing loc slice endpoints are included.

**Explain what you learned:** Choose a selector from the meaning of the request, not from habit.

### M25C02. Repair a condition with two rules

A filter must produce one True or False value per row. Combine pandas conditions with & for AND or | for OR, with parentheses around each comparison. The broken expression is a comment so this example can run.

**Read:** What does each True in mask represent?

Answer: A student meeting both the score and attendance conditions.

**Predict:** Will Lina be included?

Answer: Yes. Her score is 82 and attendance is 94.

**Explain:** Why is and wrong for these Series?

Answer: and expects a single truth value. Each comparison returns a Series of truth values, which & combines element by element.



In [ ]:
# Predict before running: Will Lina be included?
df = students()
# Broken: df[(df["Score"] >= 80) and (df["Attendance"] >= 90)]
mask = (df["Score"] >= 80) & (df["Attendance"] >= 90)
result = df.loc[mask, ["Student_ID", "Score", "Attendance"]]
print(result)

**Your turn:** Find students whose Score is below 70 OR whose Attendance is below 80. Print their IDs and both values.

For the check, keep these result names: `result`.

**Hint:** Use | and keep both comparisons in parentheses.

**Watch for this:** Do not remove the parentheses around each comparison.

In [ ]:
# Change this code to complete the task.
# Question: Find students whose Score is below 70 OR whose Attendance is below 80. Print their IDs and both values.
df = students()
mask = (df["Score"] < 70) | (df["Attendance"] < 80)
result = df.loc[mask, ["Student_ID", "Score", "Attendance"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S004','S008','S011']
print("Code checks passed. Review the written explanation as well.")

S004, S008, and S011 meet at least one condition. S011 qualifies by score even though attendance is 82.

**Quick check:** A student meets only one of two conditions. Which operator keeps that student?

1. &
2. |
3. and

Answer: 2. OR (|) accepts a row when either condition is True; AND (&) requires both.

**Explain what you learned:** Translate AND, OR, and NOT into &, |, and ~.

### M25C03. Change the table you actually intend to change

A filtered table and the original table are different objects. Use a single .loc assignment when the original must change. Use .copy() when creating a separate working table, and inspect both afterward.

**Read:** Which object receives the assignment?

Answer: df itself, through one .loc operation.

**Predict:** What score will S004 have now?

Answer: 70. The original example value was 65.

**Explain:** Why should you avoid the commented chained assignment?

Answer: It writes through an intermediate object. Under pandas Copy-on-Write it cannot update the original DataFrame.



In [ ]:
# Predict before running: What score will S004 have now?
df = students()
mask = df["Student_ID"].eq("S004")
# Avoid: df[mask]["Score"] = 70
df.loc[mask, "Score"] = 70
result = df.loc[mask, ["Student_ID", "Score"]]
print(result)

**Your turn:** Start with a fresh df = students() and create a separate copy of its RAK rows named rak. Add 5 points to scores in rak. Show that Amira's copied score is 93 while her score in df remains 88.

For the check, keep these result names: `df`, `rak`.

**Hint:** Start with df.loc[df['Campus'].eq('RAK')].copy().

**Watch for this:** A method returning a new object is not an instruction to update the old one.

In [ ]:
# Change this code to complete the task.
# Question: Start with a fresh df = students() and create a separate copy of its RAK rows named rak. Add 5 points to scores in rak. Show that Amira's copied score is 93 while her score in df remains 88.
df = students()
rak = df.loc[df["Campus"].eq("RAK")].copy()
rak["Score"] = rak["Score"] + 5
print("Copied score:", rak.loc[rak["Student_ID"].eq("S001"), "Score"].iloc[0])
print("Original score:", df.loc[df["Student_ID"].eq("S001"), "Score"].iloc[0])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert rak.loc[rak['Student_ID'].eq('S001'),'Score'].iloc[0] == 93
assert df.loc[df['Student_ID'].eq('S001'),'Score'].iloc[0] == 88
print("Code checks passed. Review the written explanation as well.")

The working copy changes; the original data stays available for comparison. For these numeric columns, .copy() makes the intended separation explicit.

**Quick check:** Which expression clearly updates df?

1. df[mask]['Score'] = 70
2. df.loc[mask, 'Score'] = 70
3. df.loc[mask].copy()['Score'] = 70

Answer: 2. Name the target DataFrame and select its rows and columns in one .loc assignment.

**Explain what you learned:** Make every data change explicit.

### M25C04. A correct total can still hide a bad merge

Two tables may contain the same key more than once. Matching repeated keys can multiply records. Validate the intended relationship and inspect unmatched IDs before calculating a report.

**Read:** What does validate='one_to_one' check?

Answer: The merge keys must be unique in each input table.

**Predict:** Which IDs will be unmatched?

Answer: S012 appears only in students; S013 appears only in assessment.

**Explain:** Why is a left merge alone insufficient to find extra assessment IDs?

Answer: It preserves left-side students but discards right-only records, including S013.



In [ ]:
# Predict before running: Which IDs will be unmatched?
left = students()[["Student_ID", "Name"]]
right = assessment()
result = left.merge(right, on="Student_ID", how="outer",
                    validate="one_to_one", indicator=True)
print(result.loc[result["_merge"].ne("both"), ["Student_ID", "_merge"]])
print("Rows:", len(result))

**Your turn:** Keep only enrolled students with a left merge stored in result. Keep the merge indicator column, and store the count of students with no assessment in missing_count.

For the check, keep these result names: `missing_count`, `result`.

**Hint:** Use how='left', validate='one_to_one', and indicator=True.

**Watch for this:** Do not remove duplicate keys without understanding why they exist.

In [ ]:
# Change this code to complete the task.
# Question: Keep only enrolled students with a left merge stored in result. Keep the merge indicator column, and store the count of students with no assessment in missing_count.
result = students().merge(assessment(), on="Student_ID", how="left",
                          validate="one_to_one", indicator=True)
missing_count = result["_merge"].eq("left_only").sum()
print(result[["Student_ID", "Quiz", "_merge"]])
print("Enrolled students without assessment:", missing_count)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 12
assert missing_count == 1
assert result.loc[result['_merge'].eq('left_only'),'Student_ID'].tolist() == ['S012']
print("Code checks passed. Review the written explanation as well.")

The result has 12 enrolled students and one missing assessment, S012. The earlier outer audit also exposed S013 for follow-up.

**Quick check:** What can a many-to-many merge do?

1. Always delete duplicates
2. Guarantee one row per student
3. Create multiple matches for each repeated key

Answer: 3. Repeated keys match every compatible combination. Check uniqueness and use validate before summarizing.

**Explain what you learned:** Audit the join before trusting the summary.

## Module 26. End-to-end classroom project

Clean, audit, summarize, visualize, and export a small dataset with an explanation of every choice.

Before you start: Core modules and plotting; keep raw and clean data distinct.

Official reference: https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html

### M26C01. Clean the raw table and keep an audit trail

Bring the cleaning skills together. Remove the known exact duplicate, normalize labels, preserve why a score is unavailable, and flag impossible attendance as missing. This exercise makes no assumption that an absent score equals zero.

**Read:** Why is Score_Status created before numeric conversion?

Answer: The original text distinguishes an absent student from an already missing score. Conversion alone would erase that distinction.

**Predict:** How many rows and missing numeric scores remain?

Answer: 12 unique students remain; two scores are missing numerically, S004 and S008.

**Explain:** Why not replace attendance 105 with 100 automatically?

Answer: The correct attendance is unknown. Capping would invent a value; flagging allows later correction from a source.



In [ ]:
# Predict before running: How many rows and missing numeric scores remain?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

print(df[["Student_ID", "Name", "Campus", "Score", "Score_Status", "Attendance"]])
print("Rows:", len(df))
print("Unavailable scores:", df["Score"].isna().sum())

**Your turn:** Use the cleaned table to create a review list containing students with a missing Score, Study_Hours, or Attendance. Store it in result with columns Student_ID, Score, Study_Hours, and Attendance, then print result.

For the check, keep these result names: `result`.

**Hint:** Use isna().any(axis=1) across the three columns.

**Watch for this:** Do not discard the raw labels before you have recorded their meaning.

In [ ]:
# Change this code to complete the task.
# Question: Use the cleaned table to create a review list containing students with a missing Score, Study_Hours, or Attendance. Store it in result with columns Student_ID, Score, Study_Hours, and Attendance, then print result.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

review_columns = ["Score", "Study_Hours", "Attendance"]
result = df.loc[df[review_columns].isna().any(axis=1), ["Student_ID"] + review_columns]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S004','S006','S008','S011']
print("Code checks passed. Review the written explanation as well.")

Review S004, S006, S008, and S011. Each record has at least one unavailable or invalid value. A review flag does not tell you how the missing value should be replaced.

**Quick check:** What evidence supports changing 105 to missing?

1. Attendance is defined as a percentage from 0 to 100
2. Every high value is an outlier to remove
3. Pandas does not support numbers above 100

Answer: 1. The column's documented range makes 105 invalid. Pandas itself has no such numeric limit.

**Explain what you learned:** Clean with documented rules and preserve the reasons for missing values.

### M26C02. Report a mean with its denominator

A group average is easier to interpret when its sample count is visible. Count all students and count recorded scores separately. Pandas mean skips missing numeric scores by default; say that in your report.

**Read:** How are Students and Recorded_Scores different?

Answer: Students counts rows. Recorded_Scores counts non-missing scores.

**Predict:** Which campus has fewer recorded scores than students?

Answer: Dubai has six students but only four recorded scores in the cleaned raw table.

**Explain:** May you report the mean as an average of all six Dubai students?

Answer: No. It is based on four recorded scores. The missing observations may also affect the interpretation.



In [ ]:
# Predict before running: Which campus has fewer recorded scores than students?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Campus", as_index=False)
          .agg(Students=("Student_ID", "size"),
               Recorded_Scores=("Score", "count"),
               Mean_Score=("Score", "mean")))
print(result.round(2))

**Your turn:** Produce the same report by Program. Add Missing_Scores = Students - Recorded_Scores, then sort from highest to lowest Mean_Score.

For the check, keep these result names: `result`.

**Hint:** Use groupby('Program'), named aggregation, and a new column.

**Watch for this:** A polished mean without the number of observed values can mislead.

In [ ]:
# Change this code to complete the task.
# Question: Produce the same report by Program. Add Missing_Scores = Students - Recorded_Scores, then sort from highest to lowest Mean_Score.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Program", as_index=False)
          .agg(Students=("Student_ID", "size"),
               Recorded_Scores=("Score", "count"),
               Mean_Score=("Score", "mean")))
result["Missing_Scores"] = result["Students"] - result["Recorded_Scores"]
result = result.sort_values("Mean_Score", ascending=False)
print(result.round(2))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Program'].tolist() == ['AI','IT']
assert result['Missing_Scores'].tolist() == [0,2]
assert np.isclose(result.iloc[0]['Mean_Score'],515/6)
assert np.isclose(result.iloc[1]['Mean_Score'],77.25)
print("Code checks passed. Review the written explanation as well.")

AI has six recorded scores and a mean of 85.83. IT has four recorded scores, two missing scores, and a mean of 77.25. These fictional data do not establish a general difference between programs.

**Quick check:** Which denominator does mean use here?

1. Every row, treating missing values as zero
2. Only non-missing score values in each group
3. The number of campuses

Answer: 2. mean skips missing numeric values by default. Reporting count beside it makes the denominator clear.

**Explain what you learned:** Attach counts and missingness to summary statistics.

### M26C03. Join, audit, and select a follow-up list

Combine the cleaned records with assessment results, checking both match quality and row counts. The follow-up rule below uses an observed Score under 75 or a missing assessment. Unknown scores need separate review, not automatic failure.

**Read:** Why is fillna(False) used in the score condition?

Answer: An unknown score should not be classified as below 75. Missing scores are handled by a separate review process.

**Predict:** Which student is selected because an assessment is missing?

Answer: S012, Lina.

**Explain:** Does excluding an unknown score mean the student passed?

Answer: No. It only means this specific rule does not classify that student. Missing data still need review.



In [ ]:
# Predict before running: Which student is selected because an assessment is missing?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

joined = df.merge(assessment(), on="Student_ID", how="left",
                  validate="one_to_one", indicator=True)
mask = joined["Score"].lt(75).fillna(False) | joined["_merge"].eq("left_only")
result = joined.loc[mask, ["Student_ID", "Score", "Quiz", "_merge"]]
print(result)

**Your turn:** Create a separate audit with an outer merge. Show only unmatched IDs and whether each ID belongs to the student table or the assessment table.

For the check, keep these result names: `result`.

**Hint:** Use how='outer', indicator=True, then filter _merge != 'both'.

**Watch for this:** A missing comparison may propagate pd.NA; decide how the rule treats unknowns.

In [ ]:
# Change this code to complete the task.
# Question: Create a separate audit with an outer merge. Show only unmatched IDs and whether each ID belongs to the student table or the assessment table.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

audit = df.merge(assessment(), on="Student_ID", how="outer",
                 validate="one_to_one", indicator=True)
result = audit.loc[audit["_merge"].ne("both"), ["Student_ID", "_merge"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S012','S013']
print("Code checks passed. Review the written explanation as well.")

S012 is student-only; S013 is assessment-only. Investigate both before treating the assessment file as complete.

**Quick check:** A missing score should automatically become a failing score.

1. True for every dataset
2. True whenever mean() is used
3. False; a documented rule or source correction is needed

Answer: 3. A missing observation is not an observed zero or a confirmed failure.

**Explain what you learned:** Keep classification rules separate from data-quality review.

### M26C04. Create a table, a plot, and a reproducible export

Finish with a compact report and a plot that answers a stated question. Keep the clean table and the summary as separate outputs. On a new run, rebuild the report from the source and the written cleaning rules.

**Read:** Why does the summary CSV keep its index?

Answer: Campus is stored in the pivot table index and carries meaning. Removing it would lose the campus labels.

**Predict:** How many mean values appear in this 2-by-2 table?

Answer: Four: AI and IT within Dubai and RAK.

**Explain:** What must accompany this chart in a report?

Answer: The number of recorded scores per group, the missing-score policy, and a note that these are fictional teaching data. Bar differences do not prove causation.



In [ ]:
# Predict before running: How many mean values appear in this 2-by-2 table?
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = df.pivot_table(index="Campus", columns="Program", values="Score",
                        aggfunc="mean", observed=True)
counts = df.pivot_table(index="Campus", columns="Program", values="Score",
                        aggfunc="count", observed=True)
print("Recorded mean scores:")
print(result.round(2))
print("Recorded score counts:")
print(counts)
ax = result.plot.bar(color=["#216b62", "#b58b45"], rot=0)
ax.set_title("Recorded mean score by campus and program")
ax.set_ylabel("Mean score (0–100)")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()
df.to_csv(OUTPUT_DIR / "capstone_cleaned_students.csv", index=False)
result.round(2).to_csv(OUTPUT_DIR / "capstone_score_summary.csv")
counts.to_csv(OUTPUT_DIR / "capstone_recorded_counts.csv")
print("Saved cleaned data, summary, and recorded counts in outputs/.")

**Your turn:** Make a Program-level report with columns Program, Recorded_Scores, and Mean_Score. Export it without a DataFrame index to capstone_program_report.csv. Read the exported file back into read_back and verify its columns and row count.

For the check, keep these result names: `read_back`.

**Hint:** Use groupby, agg, to_csv(index=False), and read_csv.

**Watch for this:** CSV does not preserve every pandas dtype. Recheck dtypes when loading it later.

In [ ]:
# Change this code to complete the task.
# Question: Make a Program-level report with columns Program, Recorded_Scores, and Mean_Score. Export it without a DataFrame index to capstone_program_report.csv. Read the exported file back into read_back and verify its columns and row count.
df = raw_students().drop_duplicates().copy()
df["Name"] = df["Name"].astype("string").str.strip()
df["Campus"] = df["Campus"].astype("string").str.strip().str.title()
df["Campus"] = df["Campus"].replace({"Rak": "RAK"})
df["Score_Status"] = "recorded"
df.loc[df["Score"].isna(), "Score_Status"] = "missing"
df.loc[df["Score"].eq("absent"), "Score_Status"] = "absent"
df["Score"] = pd.to_numeric(df["Score"], errors="coerce").astype("Float64")
df["Study_Hours"] = pd.to_numeric(df["Study_Hours"], errors="coerce").astype("Float64")
df["Attendance"] = df["Attendance"].astype("Float64")
df.loc[~df["Attendance"].between(0, 100), "Attendance"] = pd.NA
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="%Y-%m-%d")

result = (df.groupby("Program", as_index=False)
          .agg(Recorded_Scores=("Score", "count"), Mean_Score=("Score", "mean")))
result["Mean_Score"] = result["Mean_Score"].round(2)
path = OUTPUT_DIR / "capstone_program_report.csv"
result.to_csv(path, index=False)
read_back = pd.read_csv(path)
print(read_back)
print("Columns preserved:", read_back.columns.tolist() == result.columns.tolist())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert read_back.columns.tolist() == ['Program','Recorded_Scores','Mean_Score']
assert read_back['Recorded_Scores'].sum() == 10
assert read_back.shape == (2,3)
print("Code checks passed. Review the written explanation as well.")

The CSV contains Program, Recorded_Scores, and Mean_Score, with no extra unnamed index column. Reading the file back checks the delivered table, not just the in-memory object.

**Quick check:** When is index=False appropriate for this Program report?

1. When all meaningful labels already live in columns
2. Whenever the table contains numbers
3. Only for Excel files

Answer: 1. This report has Program as a normal column. Its default numeric index adds no useful information to the CSV.

**Explain what you learned:** Deliver a report that another person can reproduce and interpret.

## Module 27. Import CSV and text with confidence

Choose explicit parsing rules, preserve identifiers, and verify imported records.

Before you start: You can inspect a DataFrame, identify missing values, and read a CSV file.

Official reference: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

### M27C01. Inspect a real CSV before analyzing it

The bundled Palmer Penguins CSV is a real public dataset, preserved from its authoritative upstream source. Read the field definitions in data/public/README.md before interpreting its values. The simplified table records 344 observations, eight fields, and source missing values written as NA.

**Read:** What does one row describe, according to the source documentation?

Answer: One observed penguin in this simplified research dataset. The table does not provide an animal ID.

**Predict:** Will the preview contain only complete observations?

Answer: No. The fourth source record lacks its measurements and sex, so missing values are visible in the first five rows.

**Explain:** Why does a successful read_csv call not prove the data are ready for every analysis?

Answer: You still need to check field meaning, units, types, missingness, and which records the question requires.



In [ ]:
# Predict before running: Will the preview contain only complete observations?
path = DATA_DIR / "public" / "penguins.csv"
df = pd.read_csv(path, na_values=["NA"])
result = df.head(5)
print(result)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isna().sum())

**Your turn:** Read only species, island, and body_mass_g using usecols. Store the full three-column table in result and print its shape and the number of known body masses.

For the check, keep these result names: `result`.

**Hint:** Pass the three names as a list to usecols; count() counts known values.

**Watch for this:** These are historical study observations from 2007–2009, not current measurements or a population census.

In [ ]:
# Change this code to complete the task.
# Question: Read only species, island, and body_mass_g using usecols. Store the full three-column table in result and print its shape and the number of known body masses.
path = DATA_DIR / "public" / "penguins.csv"
result = pd.read_csv(path, usecols=["species", "island", "body_mass_g"], na_values=["NA"])
print(result.head())
print("Shape:", result.shape)
print("Known body masses:", result["body_mass_g"].count())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (344, 3)
assert result.columns.tolist() == ["species", "island", "body_mass_g"]
assert result["body_mass_g"].count() == 342
print("Code checks passed. Review the written explanation as well.")

The result has 344 rows and three columns. There are 342 known body masses; two missing measurements remain missing.

**Quick check:** How many body masses support a summary of the available measurements?

1. 344
2. 342
3. 333

Answer: 2. Two of the 344 body-mass entries are missing. The 333 complete-record count answers a different question.

**Explain what you learned:** Inspect the parsed table and its documentation together.

### M27C02. The separator is part of the file contract

A text file does not need a .csv extension to hold a table. Its delimiter determines where fields begin and end. The TSV and TXT files here are documented format conversions of the same public observations, so you can compare the imports directly.

**Read:** Why does the comma-based import have only one column?

Answer: The TSV uses tabs between fields; a comma separator does not split those fields.

**Predict:** What shape should the correctly parsed TSV have?

Answer: 344 rows and eight columns, matching the original CSV.

**Explain:** Does changing a filename extension change its internal delimiter?

Answer: No. Parsing must follow the actual file contents and documented format.



In [ ]:
# Predict before running: What shape should the correctly parsed TSV have?
path = DATA_DIR / "public" / "penguins.tsv"
wrong = pd.read_csv(path, sep=",")
result = pd.read_csv(path, sep="\t", na_values=["NA"])
print("With a comma separator:", wrong.shape)
print("With a tab separator:", result.shape)
print(result.head(3))

**Your turn:** Read penguins.txt into result using its semicolon separator and skip its two comment lines using comment="#". Read penguins.csv into reference and compare the two parsed tables.

For the check, keep these result names: `reference`, `result`.

**Hint:** Use pd.read_csv even though the extension is .txt.

**Watch for this:** Do not use skiprows to hide unexpected malformed records; first identify what those rows mean.

In [ ]:
# Change this code to complete the task.
# Question: Read penguins.txt into result using its semicolon separator and skip its two comment lines using comment="#". Read penguins.csv into reference and compare the two parsed tables.
folder = DATA_DIR / "public"
result = pd.read_csv(folder / "penguins.txt", sep=";", comment="#", encoding="utf-8", na_values=["NA"])
reference = pd.read_csv(folder / "penguins.csv", na_values=["NA"])
pd.testing.assert_frame_equal(result, reference)
print(result.head(3))
print("Same parsed values, columns, and row order as the source CSV.")

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (344, 8)
pd.testing.assert_frame_equal(result, reference)
print("Code checks passed. Review the written explanation as well.")

The comment lines are excluded from the table, and all 344 observations remain. No conversion to an Excel file is necessary to read this text format.

**Quick check:** Which setting separates a TSV into fields?

1. sep=";"
2. sep=","
3. sep="\t"

Answer: 3. TSV means tab-separated values, so the delimiter is a tab.

**Explain what you learned:** Read a table according to its actual delimiter and header rules.

### M27C03. Preserve leading zeros and literal NA codes

This small import fixture is explicitly fictional and separate from the penguin research data. Its IDs contain leading zeros, NA is a literal site code, and MISSING marks one unavailable amount. Encoding, quoting, decimal symbols, and missing-value rules all affect what a reader produces.

**Read:** Why is record_id read as text?

Answer: It is an identifier, not a quantity. Converting 001 to an integer would lose its leading zeros.

**Predict:** Does the NA site code become a missing value in this import?

Answer: No. keep_default_na=False preserves the literal code as text.

**Explain:** Why does the semicolon inside the second note not create an extra column?

Answer: The field is quoted using CSV quoting rules, so the embedded separator belongs to the note.



In [ ]:
# Predict before running: Does the NA site code become a missing value in this import?
path = DATA_DIR / "public" / "import_practice_semicolon.csv"
result = pd.read_csv(path, sep=";", comment="#", encoding="utf-8-sig", dtype="string", keep_default_na=False)
print(result)
print("IDs:", result["record_id"].tolist())
print("Site codes:", result["site_code"].tolist())

**Your turn:** Read the fixture again with record_id and site_code as strings, comma decimal symbols for amount_eur, and only MISSING in amount_eur treated as missing. Keep the NA site code unchanged. Print the known amount count and sum.

For the check, keep these result names: `result`.

**Hint:** Use decimal="," and a column-specific na_values dictionary while keeping keep_default_na=False.

**Watch for this:** The meanings NA=North Area and EU=East Unit apply only to this invented fixture. Do not impose those meanings on other sources.

In [ ]:
# Change this code to complete the task.
# Question: Read the fixture again with record_id and site_code as strings, comma decimal symbols for amount_eur, and only MISSING in amount_eur treated as missing. Keep the NA site code unchanged. Print the known amount count and sum.
path = DATA_DIR / "public" / "import_practice_semicolon.csv"
result = pd.read_csv(path, sep=";", comment="#", encoding="utf-8-sig", dtype={"record_id": "string", "site_code": "string"}, decimal=",", keep_default_na=False, na_values={"amount_eur": ["MISSING"]})
print(result)
print("Known amounts:", result["amount_eur"].count())
print("Known amount sum:", result["amount_eur"].sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["record_id"].tolist() == ["001", "010", "100"]
assert result["site_code"].tolist() == ["NA", "EU", "NA"]
assert result["amount_eur"].count() == 2
assert result["amount_eur"].sum() == 12.5
assert result.loc[2, "amount_eur"] == 0.0
print("Code checks passed. Review the written explanation as well.")

The amounts are 12.5, missing, and 0.0 EUR. Two known amounts sum to 12.5; the missing value was not replaced with zero. Both NA site codes and all leading-zero IDs are preserved.

**Quick check:** Which is a safe way to preserve leading-zero identifiers?

1. Read the identifier column with dtype="string"
2. Convert every column to integers
3. Remove all zeros before importing

Answer: 1. Identifiers need their full text representation even when they contain only digits.

**Explain what you learned:** Specify identifiers, missing tokens, and number formats deliberately.

### M27C04. Verify a CSV round trip with explicit types

A CSV stores text and does not carry a pandas dtype schema. A repeatable import can specify its expected text, integer, and measurement types. Exporting and reading back the table lets you check that values, missingness, and the chosen schema survive.

**Read:** Why use nullable Int64 for body_mass_g?

Answer: The source contains whole-number masses and missing entries; nullable Int64 represents both without inventing values.

**Predict:** How many columns should the exported CSV contain?

Answer: Eight, because index=False does not add the DataFrame index as a ninth field.

**Explain:** Why specify the schema again when reading the exported CSV?

Answer: CSV has no embedded pandas dtype contract, so the reader would otherwise infer types again.



In [ ]:
# Predict before running: How many columns should the exported CSV contain?
schema = {"species": "string", "island": "string", "bill_length_mm": "Float64", "bill_depth_mm": "Float64", "flipper_length_mm": "Int64", "body_mass_g": "Int64", "sex": "string", "year": "Int64"}
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", dtype=schema, na_values=["NA"])
path = OUTPUT_DIR / "penguins_round_trip.csv"
df.to_csv(path, index=False, na_rep="NA")
result = pd.read_csv(path, dtype=schema, na_values=["NA"])
pd.testing.assert_frame_equal(result, df)
print("Rows and columns:", result.shape)
print(result.dtypes)
print("Round-trip values and declared types match.")

**Your turn:** Using the same explicit schema, store only Gentoo observations in gentoo with a reset index. Export gentoo to penguins_gentoo.csv, read the file back into result, and compare result with gentoo.

For the check, keep these result names: `gentoo`, `result`.

**Hint:** Reset the filtered index because the CSV intentionally does not store it.

**Watch for this:** An export should go to OUTPUT_DIR, leaving the downloaded source unchanged.

In [ ]:
# Change this code to complete the task.
# Question: Using the same explicit schema, store only Gentoo observations in gentoo with a reset index. Export gentoo to penguins_gentoo.csv, read the file back into result, and compare result with gentoo.
schema = {"species": "string", "island": "string", "bill_length_mm": "Float64", "bill_depth_mm": "Float64", "flipper_length_mm": "Int64", "body_mass_g": "Int64", "sex": "string", "year": "Int64"}
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", dtype=schema, na_values=["NA"])
gentoo = df.loc[df["species"] == "Gentoo"].reset_index(drop=True)
path = OUTPUT_DIR / "penguins_gentoo.csv"
gentoo.to_csv(path, index=False, na_rep="NA")
result = pd.read_csv(path, dtype=schema, na_values=["NA"])
pd.testing.assert_frame_equal(result, gentoo)
print("Gentoo rows:", len(result))
print("Known body masses:", result["body_mass_g"].count())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (124, 8)
assert result["body_mass_g"].count() == 123
pd.testing.assert_frame_equal(result, gentoo)
print("Code checks passed. Review the written explanation as well.")

The subset contains 124 Gentoo observations, with 123 known body masses. Its row labels are newly created positions, not animal identities.

**Quick check:** Does a CSV automatically preserve a pandas nullable-integer dtype?

1. Yes, every dtype is stored inside CSV
2. No; provide the intended dtype when importing
3. Only if the filename contains Int64

Answer: 2. CSV stores fields as text; the receiving reader applies the type rules.

**Explain what you learned:** Treat an import/export schema as part of reproducibility.

## Module 28. Work across Excel and JSON formats

Read worksheets and JSON structures without changing the underlying observations.

Before you start: You can read files, interpret missing values, and compare DataFrames.

Official reference: https://pandas.pydata.org/docs/user_guide/io.html

### M28C01. Choose the worksheet you actually need

An Excel workbook can contain both observations and documentation. The bundled workbook has a Penguins data sheet and a Data_Dictionary sheet with field meanings and units. Specify the intended sheet so a change in tab order does not silently change your input.

**Read:** Why specify sheet_name="Penguins"?

Answer: It identifies the observations sheet directly rather than assuming it is the first worksheet.

**Predict:** Will the data sheet have the same 344 rows as the source CSV?

Answer: Yes. The workbook is a faithful format conversion, with missing source measurements represented by blank cells.

**Explain:** Why should a data dictionary be kept separate from the observation rows?

Answer: Descriptions and units are metadata, not extra penguin observations.



In [ ]:
# Predict before running: Will the data sheet have the same 344 rows as the source CSV?
path = DATA_DIR / "public" / "penguins.xlsx"
with pd.ExcelFile(path, engine="openpyxl") as book:
    print("Worksheets:", book.sheet_names)
    result = pd.read_excel(book, sheet_name="Penguins")
print(result.head(3))
print("Shape:", result.shape)
print("Known body masses:", result["body_mass_g"].count())

**Your turn:** Read only A:D from Data_Dictionary into a DataFrame named dictionary. Select the fields measured in mm or g into result and print those rows.

For the check, keep these result names: `dictionary`, `result`.

**Hint:** Use usecols="A:D", then filter unit with isin(["mm", "g"]).

**Watch for this:** The Excel lesson requires openpyxl. A .xlsx file is not plain delimited text and should not be passed to read_csv.

In [ ]:
# Change this code to complete the task.
# Question: Read only A:D from Data_Dictionary into a DataFrame named dictionary. Select the fields measured in mm or g into result and print those rows.
path = DATA_DIR / "public" / "penguins.xlsx"
dictionary = pd.read_excel(path, sheet_name="Data_Dictionary", usecols="A:D", engine="openpyxl")
result = dictionary.loc[dictionary["unit"].isin(["mm", "g"])]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(dictionary) == 8
assert result["column"].tolist() == ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
assert result["missing_count"].eq(2).all()
print("Code checks passed. Review the written explanation as well.")

The four measurement fields appear with their units and two missing values each. The dictionary describes the data; it does not fill those missing measurements.

**Quick check:** Which argument selects an Excel worksheet by name?

1. sheet_name
2. orient
3. lines

Answer: 1. sheet_name selects a worksheet. orient and lines are JSON-related arguments.

**Explain what you learned:** Inspect workbook structure and choose a worksheet explicitly.

### M28C02. Read an array of JSON records

Records-oriented JSON represents each row as an object whose keys name fields. The bundled JSON is a documented conversion of the original public CSV. Its null values represent the same missing entries as the source NA tokens.

**Read:** What corresponds to one table row in this JSON structure?

Answer: One object in the outer JSON array.

**Predict:** Will JSON null become the text string "null" in the DataFrame?

Answer: No. It is parsed as a missing value.

**Explain:** Why can equivalent CSV and JSON imports have different inferred pandas dtypes?

Answer: The formats carry values differently and the readers apply their own inference rules; choose a consistent schema when exact dtypes matter.



In [ ]:
# Predict before running: Will JSON null become the text string "null" in the DataFrame?
path = DATA_DIR / "public" / "penguins.records.json"
df = pd.read_json(path, orient="records")
result = df[["species", "island", "body_mass_g"]].head(5)
print(result)
print("Loaded records:", len(df))
print("Missing body masses:", df["body_mass_g"].isna().sum())

**Your turn:** Read the records JSON and summarize the number of observations and known body masses by species. Store the table in result with columns species, Observations, and Known_mass; keep species as a normal column.

For the check, keep these result names: `result`.

**Hint:** Use groupby(..., as_index=False) with size and count.

**Watch for this:** JSON orientation describes structure. Use the orientation that matches the supplied document.

In [ ]:
# Change this code to complete the task.
# Question: Read the records JSON and summarize the number of observations and known body masses by species. Store the table in result with columns species, Observations, and Known_mass; keep species as a normal column.
df = pd.read_json(DATA_DIR / "public" / "penguins.records.json", orient="records")
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_mass=("body_mass_g", "count"))
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Observations"].sum() == 344
assert result["Known_mass"].sum() == 342
assert result.set_index("species").loc["Adelie", "Known_mass"] == 151
print("Code checks passed. Review the written explanation as well.")

Adelie has 152 observations and 151 known body masses; Chinstrap has 68 and 68; Gentoo has 124 and 123.

**Quick check:** In records-oriented JSON, field names are stored as what?

1. Object keys
2. Excel worksheet tabs
3. Pandas row positions

Answer: 1. Each record object maps field names to values.

**Explain what you learned:** Read JSON according to its actual record structure.

### M28C03. Read JSON Lines in manageable batches

JSON Lines stores one complete JSON object on each line, without an outer array. Use lines=True so pandas reads that structure correctly. Batch processing can summarize a larger file without retaining every row at once, provided you combine the statistics correctly.

**Read:** What tells pandas that each line is a separate JSON record?

Answer: lines=True.

**Predict:** How many records should this JSON Lines file contain?

Answer: 344, the same number as the public CSV and records JSON.

**Explain:** Why is the average of chunk means generally not the overall mean?

Answer: Chunks may have different numbers of known observations; their means require weighting by those known counts.



In [ ]:
# Predict before running: How many records should this JSON Lines file contain?
path = DATA_DIR / "public" / "penguins.jsonl"
df = pd.read_json(path, lines=True)
result = df.head(3)
print(result)
print("Rows:", len(df))

**Your turn:** Read the file in chunks of 100 rows. Accumulate total rows, the sum of known body masses, and the count of known body masses. Store a dictionary named result with keys rows, mass_sum_g, known_mass, and mean_mass_g for those totals and the overall mean.

For the check, keep these result names: `result`.

**Hint:** Add each chunk sum and count separately, then divide the combined sum by the combined known count.

**Watch for this:** This file is small; chunking is taught as a technique, not claimed as a speed improvement here.

In [ ]:
# Change this code to complete the task.
# Question: Read the file in chunks of 100 rows. Accumulate total rows, the sum of known body masses, and the count of known body masses. Store a dictionary named result with keys rows, mass_sum_g, known_mass, and mean_mass_g for those totals and the overall mean.
path = DATA_DIR / "public" / "penguins.jsonl"
total_rows = 0
mass_sum = 0.0
known_count = 0
with pd.read_json(path, lines=True, chunksize=100) as reader:
    for chunk in reader:
        total_rows += len(chunk)
        mass_sum += chunk["body_mass_g"].sum()
        known_count += chunk["body_mass_g"].count()
result = {"rows": total_rows, "known_mass": int(known_count), "mass_sum_g": float(mass_sum), "mean_mass_g": mass_sum / known_count}
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["rows"] == 344
assert result["known_mass"] == 342
assert result["mass_sum_g"] == 1437000
assert np.isclose(result["mean_mass_g"], 1437000 / 342)
print("Code checks passed. Review the written explanation as well.")

The four chunks contain 344 rows in total. The 342 known body masses sum to 1,437,000 g, giving a mean of approximately 4,201.75 g. Missing values do not contribute zero measurements to that mean.

**Quick check:** Which calculation correctly combines unequal chunks?

1. The unweighted average of their means
2. Total known-value sum divided by total known-value count
3. The largest chunk mean

Answer: 2. Accumulated sums and matching known counts preserve the proper denominator.

**Explain what you learned:** Combine batch statistics using the matching totals and counts.

### M28C04. Flatten nested JSON and verify the values

Nested JSON can place related fields inside another object. The teaching version groups the four measurements under measurements while preserving all source values. json_normalize flattens those nested fields into columns with dotted names.

**Read:** What does measurements.body_mass_g describe in the flattened table?

Answer: The body_mass_g field that was inside each record's measurements object.

**Predict:** Does flattening change the number of penguin observations?

Answer: No. Each of the 344 record objects still contributes one row.

**Explain:** Why should you check row counts and values after flattening a less familiar JSON document?

Answer: A different nesting structure, especially lists, can change what one output row represents.



In [ ]:
# Predict before running: Does flattening change the number of penguin observations?
path = DATA_DIR / "public" / "penguins.nested.json"
payload = json.loads(path.read_text(encoding="utf-8"))
result = pd.json_normalize(payload, sep=".")
print(result.head(3))
print("Columns:", result.columns.tolist())
print("Known body masses:", result["measurements.body_mass_g"].count())

**Your turn:** Flatten the file, remove the measurements. prefix from column names, and restore the CSV column order. Store the source CSV in reference and the flattened table in result, using the same explicit nullable schema for both. Compare all values and missing entries.

For the check, keep these result names: `reference`, `result`.

**Hint:** Use rename for the four measurement columns, reindex columns from the reference, then astype with the shared schema.

**Watch for this:** The nested structure is a course-created representation, not a claim about an upstream research API.

In [ ]:
# Change this code to complete the task.
# Question: Flatten the file, remove the measurements. prefix from column names, and restore the CSV column order. Store the source CSV in reference and the flattened table in result, using the same explicit nullable schema for both. Compare all values and missing entries.
folder = DATA_DIR / "public"
payload = json.loads((folder / "penguins.nested.json").read_text(encoding="utf-8"))
flat = pd.json_normalize(payload, sep=".")
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
flat = flat.rename(columns={"measurements." + name: name for name in measurements})
schema = {"species": "string", "island": "string", "bill_length_mm": "Float64", "bill_depth_mm": "Float64", "flipper_length_mm": "Int64", "body_mass_g": "Int64", "sex": "string", "year": "Int64"}
reference = pd.read_csv(folder / "penguins.csv", dtype=schema, na_values=["NA"])
result = flat.loc[:, reference.columns].astype(schema)
pd.testing.assert_frame_equal(result, reference)
print(result.head(3))
print("All source values, missing entries, columns, and row order match.")

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (344, 8)
pd.testing.assert_frame_equal(result, reference)
print("Code checks passed. Review the written explanation as well.")

Flattening and renaming change the representation, not the observations. An explicit shared schema makes missing-value and dtype comparisons predictable across file readers.

**Quick check:** Does json_normalize recover missing research measurements?

1. Yes, from the field names
2. No, it changes the structure while preserving missingness
3. Only for nested numbers

Answer: 2. Normalization flattens a representation; it does not infer missing observations.

**Explain what you learned:** Verify semantic equality after changing a data representation.

## Module 29. Analyze a real public dataset

Produce a traceable descriptive analysis with justified exclusions and visible denominators.

Before you start: You can clean, group, plot, and export data; review the public-data README and field dictionary.

Official reference: https://allisonhorst.github.io/palmerpenguins/reference/penguins.html

### M29C01. Check the source and audit missingness

Begin a public-data analysis by identifying the source, version, and field definitions. A checksum can confirm that the local source file matches the recorded bytes, while a missingness audit tells you what is available to analyze. These checks do not prove that every scientific measurement is correct.

**Read:** What does the checksum comparison establish?

Answer: The local file has the exact bytes recorded in the provenance manifest, assuming the manifest itself is trusted.

**Predict:** Which field has more missing values: sex or body_mass_g?

Answer: sex has 11 missing entries, compared with two missing body masses.

**Explain:** Why does file identity not prove scientific validity?

Answer: A file can be an exact copy and still contain measurement limitations, sampling constraints, or source errors.



In [ ]:
# Predict before running: Which field has more missing values: sex or body_mass_g?
import hashlib
folder = DATA_DIR / "public"
metadata = json.loads((folder / "provenance.json").read_text(encoding="utf-8"))
digest = hashlib.sha256((folder / "penguins.csv").read_bytes()).hexdigest()
assert digest == metadata["artifacts"]["penguins.csv"]["sha256"]
df = pd.read_csv(folder / "penguins.csv", na_values=["NA"])
result = pd.DataFrame({"Missing": df.isna().sum(), "Known": df.notna().sum()})
print("Dataset:", metadata["dataset"])
print("Source commit:", metadata["upstream"]["commit"])
print("Shape:", df.shape)
print(result)

**Your turn:** Read the source into df. Store a dictionary named result with keys total_rows, rows_with_any_missing, complete_rows, and rows_with_known_mass. Calculate those four counts across the source fields without deleting rows from df.

For the check, keep these result names: `df`, `result`.

**Hint:** Use isna().any(axis=1), dropna() only for counting, and body_mass_g.notna().sum().

**Watch for this:** A row position is a way to trace the source file, not an animal identifier.

In [ ]:
# Change this code to complete the task.
# Question: Read the source into df. Store a dictionary named result with keys total_rows, rows_with_any_missing, complete_rows, and rows_with_known_mass. Calculate those four counts across the source fields without deleting rows from df.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = {
    "total_rows": len(df),
    "rows_with_any_missing": int(df.isna().any(axis=1).sum()),
    "complete_rows": len(df.dropna()),
    "rows_with_known_mass": int(df["body_mass_g"].notna().sum()),
}
print(result)
print("The original DataFrame still has", len(df), "rows.")

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result == {"total_rows": 344, "rows_with_any_missing": 11, "complete_rows": 333, "rows_with_known_mass": 342}
assert len(df) == 344
print("Code checks passed. Review the written explanation as well.")

There are 344 rows, 11 with at least one missing field, 333 complete rows, and 342 with known body mass. Complete-case filtering would discard nine additional known body masses because another field is missing.

**Quick check:** Which statement is justified by a matching recorded checksum?

1. The research has no limitations
2. The local source bytes match the recorded file
3. Every missing value is random

Answer: 2. A checksum establishes file identity, not representativeness or correctness of the study design.

**Explain what you learned:** Document source identity and data availability before reporting results.

### M29C02. Choose exclusions to match the question

A body-mass summary needs a known body mass and a grouping label. It does not automatically need recorded sex. Keep an exclusion audit and retain original units; this exercise does not replace missing measurements with invented values.

**Read:** Which two fields control inclusion in this cell?

Answer: species and body_mass_g.

**Predict:** Will every retained row also have a recorded sex?

Answer: No. Nine retained rows have a known mass but missing sex.

**Explain:** Why is source_row useful, and what does it not establish?

Answer: It traces a record to its one-based position in the source table. It is not a biological identity or a repeated-measurement ID.



In [ ]:
# Predict before running: Will every retained row also have a recorded sex?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
df.insert(0, "source_row", np.arange(1, len(df) + 1))
required = ["species", "body_mass_g"]
missing_required = df[required].isna().any(axis=1)
audit = df.loc[missing_required, ["source_row"] + required].copy()
result = df.loc[~missing_required].copy()
assert result["body_mass_g"].gt(0).all()
print("Rows retained:", len(result))
print("Excluded rows:")
print(audit)
print("Retained rows with unrecorded sex:", result["sex"].isna().sum())

**Your turn:** Change the question to body-mass summaries by species and sex. Require species, sex, and body_mass_g. Recreate the retained result and exclusion audit from the original file; do not reuse the earlier filtered table.

For the check, keep these result names: `audit`, `result`.

**Hint:** Add sex to required and rebuild the missing-required mask.

**Watch for this:** Report exclusions and consider whether the remaining observations differ systematically; these checks do not establish random missingness.

In [ ]:
# Change this code to complete the task.
# Question: Change the question to body-mass summaries by species and sex. Require species, sex, and body_mass_g. Recreate the retained result and exclusion audit from the original file; do not reuse the earlier filtered table.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
df.insert(0, "source_row", np.arange(1, len(df) + 1))
required = ["species", "sex", "body_mass_g"]
missing_required = df[required].isna().any(axis=1)
audit = df.loc[missing_required, ["source_row"] + required].copy()
result = df.loc[~missing_required].copy()
print("Rows retained:", len(result))
print("Rows excluded:", len(audit))
print(audit)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert len(result) == 333
assert len(audit) == 11
assert result[["species", "sex", "body_mass_g"]].notna().all().all()
assert len(result) + len(audit) == 344
print("Code checks passed. Review the written explanation as well.")

The revised question retains 333 records and excludes 11. The decision changed because the requested grouping now needs sex; it is not a rule that every analysis must use complete cases.

**Quick check:** Why keep some rows with missing sex for the original mass-by-species question?

1. Sex is not required for that stated summary
2. Missing sex is equivalent to male
3. Missing measurements never matter

Answer: 1. A required-field policy should match the analysis question and still report what is missing.

**Explain what you learned:** Define the required fields and preserve an exclusion audit.

### M29C03. Report group means with their denominators

A group mean uses the known measurements in that group. Show total observations, known measurements, and missing measurements beside the mean so readers can see its denominator. The summaries describe these observations; they do not establish causal species or island effects.

**Read:** What is the denominator for the Adelie mean mass?

Answer: 151 known body masses, not all 152 Adelie observations.

**Predict:** Which species has no missing body masses in this table?

Answer: Chinstrap: all 68 observations have known body mass.

**Explain:** Can these group means alone explain why the species differ in body mass?

Answer: No. They are descriptive summaries; causal explanations require an appropriate design and relevant evidence.



In [ ]:
# Predict before running: Which species has no missing body masses in this table?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
print(result.round({"Mean_mass_g": 2}))
print("Total observed records:", result["Observations"].sum())
print("Total known body masses:", result["Known_mass"].sum())

**Your turn:** Group by island instead of species. Store result with island as a normal column, sorted in ascending island order, and report Observations, Known_mass, Missing_mass, and Mean_mass_g. Reconcile the total observations and known measurements with the full table.

For the check, keep these result names: `result`.

**Hint:** Change the grouping key and keep the matching size/count denominators.

**Watch for this:** Sample sizes and species composition differ across islands. A descriptive comparison is not an adjusted causal analysis.

In [ ]:
# Change this code to complete the task.
# Question: Group by island instead of species. Store result with island as a normal column, sorted in ascending island order, and report Observations, Known_mass, Missing_mass, and Mean_mass_g. Reconcile the total observations and known measurements with the full table.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("island", as_index=False).agg(Observations=("island", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
assert result["Observations"].sum() == len(df)
assert result["Known_mass"].sum() == df["body_mass_g"].count()
print(result.round({"Mean_mass_g": 2}))

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Observations"].tolist() == [168, 124, 52]
assert result["Known_mass"].tolist() == [167, 124, 51]
assert result["Missing_mass"].sum() == 2
assert np.isclose(result.set_index("island").loc["Dream", "Mean_mass_g"], 460400 / 124)
print("Code checks passed. Review the written explanation as well.")

Biscoe has 168 observations and 167 known masses; Dream has 124 and 124; Torgersen has 52 and 51. Their mean masses are approximately 4,716.02 g, 3,712.90 g, and 3,706.37 g. Different species composition is one reason to avoid interpreting these as an isolated island effect.

**Quick check:** What should accompany a mean when values are missing?

1. Only the largest value
2. The number of known measurements used
3. A claim that missing values are zero

Answer: 2. The known-value count tells the reader what supports the mean.

**Explain what you learned:** Pair every summary with its relevant count and interpretation.

### M29C04. Create a chart and an auditable export

Combine the source, missingness policy, grouped summary, chart, and export into one repeatable cell. Keep counts in the exported table and label the plotted units. A chart can summarize the sampled measurements without claiming a population estimate or a causal finding.

**Read:** What does each n label above a bar count?

Answer: The known body masses used to calculate that species mean.

**Predict:** Will converting grams to kilograms change which species has the largest mean in this table?

Answer: No. Every mean is divided by the same positive factor of 1,000.

**Explain:** Why export the counts and missing counts as well as the plotted mean?

Answer: They preserve the analysis denominator and make it easier to audit how the result was calculated.



In [ ]:
# Predict before running: Will converting grams to kilograms change which species has the largest mean in this table?
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_mass=("body_mass_g", "count"), Mean_mass_g=("body_mass_g", "mean"))
result["Missing_mass"] = result["Observations"] - result["Known_mass"]
result["Mean_mass_kg"] = result["Mean_mass_g"] / 1000
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result["species"], result["Mean_mass_kg"], color="#326f79")
for i, row in result.iterrows():
    ax.text(i, row["Mean_mass_kg"] + 0.08, f"n={int(row['Known_mass'])}", ha="center")
ax.set(title="Mean observed penguin body mass, 2007–2009", xlabel="Species", ylabel="Mean body mass (kg)", ylim=(0, 6))
fig.tight_layout()
csv_path = OUTPUT_DIR / "public_penguins_mass_summary.csv"
plot_path = OUTPUT_DIR / "public_penguins_mass_summary.png"
result.to_csv(csv_path, index=False)
fig.savefig(plot_path, dpi=160)
print(result.round({"Mean_mass_g": 2, "Mean_mass_kg": 3}))
print("Source: Palmer Penguins; DOI 10.5281/zenodo.3960218. Means use known body masses.")
print("These summaries describe sampled observations, not causal effects or a population census.")
plt.show()

**Your turn:** Build result as a summary by species in ascending species order, with columns species, Observations, Known_length, Median_flipper_mm, and Missing_length. Plot the medians with known-count labels on axes named ax, using the y-axis label "Median flipper length (mm)". Set csv_path and plot_path to public_penguins_flipper_summary.csv and public_penguins_flipper_summary.png in OUTPUT_DIR, then export the table and chart.

For the check, keep these result names: `ax`, `csv_path`, `plot_path`, `result`.

**Hint:** Change the measurement, summary method, labels, and filenames together; do not reuse the mass units.

**Watch for this:** Do not imply uncertainty intervals or representativeness that the chart has not calculated or established.

In [ ]:
# Change this code to complete the task.
# Question: Build result as a summary by species in ascending species order, with columns species, Observations, Known_length, Median_flipper_mm, and Missing_length. Plot the medians with known-count labels on axes named ax, using the y-axis label "Median flipper length (mm)". Set csv_path and plot_path to public_penguins_flipper_summary.csv and public_penguins_flipper_summary.png in OUTPUT_DIR, then export the table and chart.
df = pd.read_csv(DATA_DIR / "public" / "penguins.csv", na_values=["NA"])
result = df.groupby("species", as_index=False).agg(Observations=("species", "size"), Known_length=("flipper_length_mm", "count"), Median_flipper_mm=("flipper_length_mm", "median"))
result["Missing_length"] = result["Observations"] - result["Known_length"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result["species"], result["Median_flipper_mm"], color="#326f79")
for i, row in result.iterrows():
    ax.text(i, row["Median_flipper_mm"] + 3, f"n={int(row['Known_length'])}", ha="center")
ax.set(title="Median observed penguin flipper length, 2007–2009", xlabel="Species", ylabel="Median flipper length (mm)", ylim=(0, 250))
fig.tight_layout()
csv_path = OUTPUT_DIR / "public_penguins_flipper_summary.csv"
plot_path = OUTPUT_DIR / "public_penguins_flipper_summary.png"
result.to_csv(csv_path, index=False)
fig.savefig(plot_path, dpi=160)
print(result)
print("Source: Palmer Penguins; DOI 10.5281/zenodo.3960218. Medians use known lengths.")
print("The chart describes the available study observations; it is not a causal analysis.")
plt.show()

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result["Median_flipper_mm"].tolist() == [190.0, 196.0, 216.0]
assert result["Known_length"].tolist() == [151, 68, 123]
assert result["Missing_length"].sum() == 2
assert csv_path.exists() and plot_path.exists()
assert ax.get_ylabel() == "Median flipper length (mm)"
assert len(pd.read_csv(csv_path)) == 3
print("Code checks passed. Review the written explanation as well.")

The median flipper lengths are 190 mm for Adelie, 196 mm for Chinstrap, and 216 mm for Gentoo, based on 151, 68, and 123 known measurements. The table and chart use the same units and denominators.

**Quick check:** Which label correctly describes the plotted output?

1. The true body mass of every penguin worldwide
2. Mean observed body mass in the supplied study records
3. Proof that island location causes body mass

Answer: 2. The chart summarizes the available observations with their measurement counts.

**Explain what you learned:** Deliver the result with its source, units, counts, and limits.

## Independent practice

Complete the separate Campus Café assessment on unfamiliar data, then review its explanation rubric.

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates